
# PoreVoronoi-FV flow notebook: canonical + ABC synthetic cases

This notebook organises the PB61.8 GPU finite-volume code into the flow-side data products.

It focuses on the **flow-simulation side** for the cases available in this notebook:

- canonical orthogonal duct / fixed-seed wall calibration;
- held-out skewed duct;
- synthetic A/B/C masks: thin wall, narrow throat, and maze/tortuous baffles;
- component study on one representative ABC case;
- CSV/JSON outputs.

Scope note: the label source of these cells is the exact GPU Bellman geodesic relaxation (`geodesic_voronoi_labels_gpu`). The ROI--JFA labeler follows the same contract, `mask + seeds -> labels + distances + timings`. `gpu/code/flow_runner.py` executes code cells 1, 3, 5, 7 and 9 of this notebook (cell 1 is the image-resolved reference solver) and can install the ROI--JFA labeler in place of the exact labels.


In [ ]:
from __future__ import annotations

# PB61.8g preflight: this cell must pass before any later cell is run.
try:
    import cupy as cp
    import cupyx.scipy.sparse as cpx_sp
    import cupyx.scipy.sparse.linalg as cpx_spla
except Exception as exc:
    raise RuntimeError(
        "PB61.8g is GPU-only. CuPy/cupyx could not be imported. "
        "Install the CUDA-matched wheel, e.g. `pip install cupy-cuda12x`, "
        "and then restart the kernel. No CPU fallback is provided."
    ) from exc

ndev = cp.cuda.runtime.getDeviceCount()
if ndev < 1:
    raise RuntimeError("PB61.8g requires at least one visible CUDA GPU; no CPU fallback is available.")
dev = cp.cuda.Device(0)
props = cp.cuda.runtime.getDeviceProperties(0)
name = props.get('name', b'GPU')
if isinstance(name, bytes):
    name = name.decode(errors='ignore')
print(f"PB61.8g preflight OK: {ndev} CUDA device(s), device 0 = {name}, compute capability = {dev.compute_capability}")


#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""PB61.8 — ALL-GPU geodesic-Voronoi velocity-pressure FVM with convection/stability validation.

This file is intentionally GPU-only.  It does not contain a NumPy/SciPy CPU
fallback.  All heavy numerical operations are performed with CuPy arrays,
CuPy RawKernels, or cupyx sparse linear algebra:

  * multi-source geodesic label relaxation on the voxel graph;
  * positive-area facelet extraction and aggregation;
  * wall-moment aggregation;
  * sparse pressure-projection matrix assembly;
  * momentum diffusion/convection kernels;
  * pressure correction solve through cupyx.scipy.sparse.linalg.cg;
  * velocity and face-flux correction.

CPU participation is limited to launching kernels, scalar progress checks, and
optional final output transfer to CSV/JSON.  If CuPy or a CUDA GPU is unavailable,
the code fails immediately rather than falling back to CPU execution.

Interpretation:
  PB61.8 is a prototype collocated velocity-pressure FVM on geodesic Voronoi
  control volumes.  Start with low-Re/Stokes validation.  Convection is included
  as an optional explicit term but should not be used for production claims until
  benchmarked.
"""

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple
import csv
import json
import math
import time

try:
    import cupy as cp
    import cupyx.scipy.sparse as cpx_sp
    import cupyx.scipy.sparse.linalg as cpx_spla
except Exception as exc:  # pragma: no cover
    raise RuntimeError(
        "PB61.8 is GPU-only and requires CuPy + CUDA. "
        "Install a CUDA-matched CuPy wheel, e.g. cupy-cuda12x, and run on a CUDA GPU. "
        "No CPU fallback is provided."
    ) from exc


def require_cuda_gpu() -> None:
    """Fail fast if no CUDA GPU is visible."""
    ndev = cp.cuda.runtime.getDeviceCount()
    if ndev < 1:
        raise RuntimeError("PB61.8 requires at least one CUDA GPU; no CPU fallback is available.")
    _ = cp.cuda.Device(0).compute_capability


@dataclass
class PB615Config:
    # Geometry / graph
    voxel_size: float = 1.0
    distance_connectivity: int = 26               # 6, 18, or 26 for geodesic labels
    periodic_x: bool = True                      # demo mode; real inlet/outlet BC belongs in PB61.8

    # Physics
    rho: float = 1.0
    nu: float = 1.0
    body_force: Tuple[float, float, float] = (2.0e-3, 0.0, 0.0)  # acceleration

    # Time/projection solver.  PB61.8 uses implicit momentum diffusion by default,
    # so dt is a pseudo-time relaxation parameter, not the physical time step used
    # for final permeability reporting.
    dt: float = 2.0
    n_steps: int = 600
    report_every: int = 50
    projection_tol: float = 1.0e-9
    projection_maxiter: int = 3000
    pressure_gauge_eps: float = 1.0e-12

    # N-S/Stokes controls. PB61.8 opens the explicit upwind convective term.
    enable_convection: bool = True               # PB61.8: convection is active by default
    implicit_diffusion: bool = True              # viscous and wall diffusion stay implicit
    momentum_tol: float = 1.0e-8
    momentum_maxiter: int = 5000
    use_jacobi_preconditioner: bool = True
    raise_on_solver_failure: bool = True
    steady_tol: float = 1.0e-7
    steady_min_steps: int = 20
    stop_on_steady: bool = True
    explicit_nonorthogonal_correction: float = 0.0
    velocity_reconstruct_from_flux: bool = False
    velocity_reconstruction_lambda: float = 1.0e-8

    # Initial condition / validation controls
    initial_velocity_mode: str = "zero"          # zero, constant_x, manufactured_advective
    initial_velocity_amplitude: float = 0.0
    diagnostics_every_step: bool = False
    # Low-CFL decay scan retained for regression.  This confirms that the
    # projected explicit-convection update does not introduce spurious mass
    # or energy growth when the manufactured perturbation is weak.
    stability_dt_values: Tuple[float, ...] = (0.25, 0.5, 1.0, 2.0, 4.0)
    stability_n_steps: int = 120
    stability_amplitude: float = 2.0e-2
    stability_energy_growth_fail: float = 5.0

    # PB61.8 adds a high-CFL convection stress scan.  Viscosity is reduced so
    # that implicit diffusion does not hide the explicit-convection stability
    # boundary.  The amplitude/dt combination is designed to push CFL toward
    # O(1) and above on the demo geometry.
    convection_diagnostic_amplitude: float = 1.0
    stress_dt_values: Tuple[float, ...] = (0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0)
    stress_n_steps: int = 80
    stress_amplitude: float = 1.0
    stress_nu: float = 1.0e-3
    stress_energy_growth_fail: float = 10.0

    # Short on/off comparison: identical initial condition, same projection,
    # once with convection disabled and once with convection enabled.
    onoff_dt: float = 1.0
    onoff_n_steps: int = 30

    # Numerical safeguards
    tikhonov: float = 1.0e-10
    transmissibility_floor: float = 1.0e-12
    transmissibility_ratio_clip: Tuple[float, float] = (0.05, 20.0)
    wall_distance_floor: float = 0.35
    geodesic_max_iters: int = 4096
    geodesic_tie_eps: float = 1.0e-12

    # Demo geometry / seeds
    demo_shape_zyx: Tuple[int, int, int] = (16, 16, 32)
    demo_seed_stride_zyx: Tuple[int, int, int] = (4, 4, 4)

    # Output
    out_dir: str = "pb61_7_allgpu_ns_validation_outputs"


@dataclass
class GPUFVMGeometry:
    mask: "cp.ndarray"                 # bool [D,H,W]
    labels: "cp.ndarray"               # int32 [D,H,W]
    dist: "cp.ndarray"                 # float64 [D,H,W]
    n_cells: int
    volume: "cp.ndarray"               # float64 [n]
    centroid: "cp.ndarray"             # float64 [n,3] in x,y,z order
    owner: "cp.ndarray"                # int32 [nf]
    neigh: "cp.ndarray"                # int32 [nf]
    area: "cp.ndarray"                 # float64 [nf]
    avec: "cp.ndarray"                 # float64 [nf,3]
    face_centroid: "cp.ndarray"        # float64 [nf,3]
    dvec: "cp.ndarray"                 # float64 [nf,3], owner -> neigh
    tproj: "cp.ndarray"                # float64 [nf]
    w_owner: "cp.ndarray"              # float64 [nf]
    w_neigh: "cp.ndarray"              # float64 [nf]
    twall: "cp.ndarray"                # float64 [n]
    laplacian: "cpx_sp.csr_matrix"      # GPU sparse [n,n]


# -----------------------------------------------------------------------------
# CUDA kernels
# -----------------------------------------------------------------------------

_GEODESIC_RELAX_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void geodesic_relax_kernel(
    const unsigned char* mask,
    const double* dist_in,
    const int* label_in,
    double* dist_out,
    int* label_out,
    int* changed,
    const int* dzs,
    const int* dys,
    const int* dxs,
    const double* costs,
    const int n_offsets,
    const int D,
    const int H,
    const int W,
    const int periodic_x,
    const double tie_eps,
    const double INF)
{
    long idx = (long)blockDim.x * blockIdx.x + threadIdx.x;
    long N = (long)D * H * W;
    if (idx >= N) return;

    if (!mask[idx]) {
        dist_out[idx] = INF;
        label_out[idx] = -1;
        return;
    }

    int x = idx % W;
    int y = (idx / W) % H;
    int z = idx / (W * H);

    double best = dist_in[idx];
    int best_lab = label_in[idx];

    for (int m = 0; m < n_offsets; ++m) {
        int nz = z + dzs[m];
        int ny = y + dys[m];
        int nx = x + dxs[m];
        if (nz < 0 || nz >= D || ny < 0 || ny >= H) continue;
        if (nx < 0 || nx >= W) {
            if (periodic_x) {
                if (nx < 0) nx += W;
                if (nx >= W) nx -= W;
            } else {
                continue;
            }
        }
        long nidx = ((long)nz * H + ny) * W + nx;
        if (!mask[nidx]) continue;
        int nlab = label_in[nidx];
        if (nlab < 0) continue;
        double cand = dist_in[nidx] + costs[m];
        if ((cand < best - tie_eps) || (fabs(cand - best) <= tie_eps && (best_lab < 0 || nlab < best_lab))) {
            best = cand;
            best_lab = nlab;
        }
    }

    dist_out[idx] = best;
    label_out[idx] = best_lab;
    if (best_lab != label_in[idx] || fabs(best - dist_in[idx]) > tie_eps) {
        atomicExch(changed, 1);
    }
}
''', 'geodesic_relax_kernel')

_DIFFUSION_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void diffusion_kernel(
    const int nf,
    const int* owner,
    const int* neigh,
    const double* tproj,
    const double* volume,
    const double* U,
    double* Dterm,
    const double nu)
{
    int f = blockDim.x * blockIdx.x + threadIdx.x;
    if (f >= nf) return;
    int i = owner[f];
    int j = neigh[f];
    double T = tproj[f];
    double Vi = volume[i];
    double Vj = volume[j];
    for (int c = 0; c < 3; ++c) {
        double diff = nu * T * (U[3*j + c] - U[3*i + c]);
        atomicAdd(&Dterm[3*i + c], diff / Vi);
        atomicAdd(&Dterm[3*j + c], -diff / Vj);
    }
}
''', 'diffusion_kernel')

_WALL_DIFFUSION_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void wall_diffusion_kernel(
    const int n,
    const double* volume,
    const double* twall,
    const double* U,
    double* Dterm,
    const double nu)
{
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i >= n) return;
    double coef = -nu * twall[i] / volume[i];
    Dterm[3*i + 0] += coef * U[3*i + 0];
    Dterm[3*i + 1] += coef * U[3*i + 1];
    Dterm[3*i + 2] += coef * U[3*i + 2];
}
''', 'wall_diffusion_kernel')

_FACE_FLUX_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void face_flux_kernel(
    const int nf,
    const int* owner,
    const int* neigh,
    const double* w_owner,
    const double* w_neigh,
    const double* avec,
    const double* U,
    double* phi)
{
    int f = blockDim.x * blockIdx.x + threadIdx.x;
    if (f >= nf) return;
    int i = owner[f];
    int j = neigh[f];
    double wi = w_owner[f];
    double wj = w_neigh[f];
    double ux = wi * U[3*i + 0] + wj * U[3*j + 0];
    double uy = wi * U[3*i + 1] + wj * U[3*j + 1];
    double uz = wi * U[3*i + 2] + wj * U[3*j + 2];
    phi[f] = ux * avec[3*f + 0] + uy * avec[3*f + 1] + uz * avec[3*f + 2];
}
''', 'face_flux_kernel')

_DIVERGENCE_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void divergence_kernel(
    const int nf,
    const int* owner,
    const int* neigh,
    const double* phi,
    double* div)
{
    int f = blockDim.x * blockIdx.x + threadIdx.x;
    if (f >= nf) return;
    int i = owner[f];
    int j = neigh[f];
    double q = phi[f];
    atomicAdd(&div[i], q);
    atomicAdd(&div[j], -q);
}
''', 'divergence_kernel')

_CORRECT_FLUX_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void correct_flux_kernel(
    const int nf,
    const int* owner,
    const int* neigh,
    const double* tproj,
    const double* pcorr,
    const double* phi_star,
    double* phi_out,
    const double dt_over_rho)
{
    int f = blockDim.x * blockIdx.x + threadIdx.x;
    if (f >= nf) return;
    int i = owner[f];
    int j = neigh[f];
    phi_out[f] = phi_star[f] + dt_over_rho * tproj[f] * (pcorr[i] - pcorr[j]);
}
''', 'correct_flux_kernel')

_CONVECTION_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void convection_kernel(
    const int nf,
    const int* owner,
    const int* neigh,
    const double* volume,
    const double* phi,
    const double* U,
    double* Cterm)
{
    int f = blockDim.x * blockIdx.x + threadIdx.x;
    if (f >= nf) return;
    int i = owner[f];
    int j = neigh[f];
    double q = phi[f];
    int up = (q >= 0.0) ? i : j;
    for (int c = 0; c < 3; ++c) {
        double val = q * U[3*up + c];
        atomicAdd(&Cterm[3*i + c], val / volume[i]);
        atomicAdd(&Cterm[3*j + c], -val / volume[j]);
    }
}
''', 'convection_kernel')

_LSQ_ASSEMBLE_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void lsq_assemble_kernel(
    const int nf,
    const int* owner,
    const int* neigh,
    const double* dvec,
    const double* weights,
    const double* scalar,
    double* M,
    double* b)
{
    int f = blockDim.x * blockIdx.x + threadIdx.x;
    if (f >= nf) return;
    int i = owner[f];
    int j = neigh[f];
    double w = weights[f];
    double dx = dvec[3*f + 0];
    double dy = dvec[3*f + 1];
    double dz = dvec[3*f + 2];
    double dp = scalar[j] - scalar[i];

    double vv[3] = {dx, dy, dz};
    for (int a = 0; a < 3; ++a) {
        for (int c = 0; c < 3; ++c) {
            double mval = w * vv[a] * vv[c];
            atomicAdd(&M[9*i + 3*a + c], mval);
            atomicAdd(&M[9*j + 3*a + c], mval);
        }
        double bval = w * vv[a] * dp;
        atomicAdd(&b[3*i + a], bval);
        atomicAdd(&b[3*j + a], bval);
    }
}
''', 'lsq_assemble_kernel')

_FLUX_RECON_ASSEMBLE_KERNEL = cp.RawKernel(r'''
extern "C" __global__
void flux_recon_assemble_kernel(
    const int nf,
    const int* owner,
    const int* neigh,
    const double* avec,
    const double* area,
    const double* phi,
    double* M,
    double* b)
{
    int f = blockDim.x * blockIdx.x + threadIdx.x;
    if (f >= nf) return;
    int i = owner[f];
    int j = neigh[f];
    double A = fmax(area[f], 1.0e-30);
    double nx = avec[3*f + 0] / A;
    double ny = avec[3*f + 1] / A;
    double nz = avec[3*f + 2] / A;
    double un = phi[f] / A;
    double v[3] = {nx, ny, nz};
    for (int a = 0; a < 3; ++a) {
        for (int c = 0; c < 3; ++c) {
            double mval = A * v[a] * v[c];
            atomicAdd(&M[9*i + 3*a + c], mval);
            atomicAdd(&M[9*j + 3*a + c], mval);
        }
        double b_i = A * v[a] * un;
        atomicAdd(&b[3*i + a], b_i);
        atomicAdd(&b[3*j + a], b_i);
    }
}
''', 'flux_recon_assemble_kernel')


# -----------------------------------------------------------------------------
# GPU utility functions
# -----------------------------------------------------------------------------

def _threads_blocks(n: int, threads: int = 256) -> Tuple[Tuple[int], Tuple[int]]:
    return ((int((n + threads - 1) // threads),), (threads,))


def neighbor_offsets_gpu(connectivity: int, voxel_size: float) -> Tuple[cp.ndarray, cp.ndarray, cp.ndarray, cp.ndarray]:
    if connectivity not in (6, 18, 26):
        raise ValueError("distance_connectivity must be 6, 18, or 26")
    dzs = []
    dys = []
    dxs = []
    costs = []
    for dz in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                if dz == 0 and dy == 0 and dx == 0:
                    continue
                man = abs(dz) + abs(dy) + abs(dx)
                if connectivity == 6 and man != 1:
                    continue
                if connectivity == 18 and man > 2:
                    continue
                dzs.append(dz); dys.append(dy); dxs.append(dx)
                costs.append(math.sqrt(dx*dx + dy*dy + dz*dz) * voxel_size)
    return (cp.asarray(dzs, dtype=cp.int32),
            cp.asarray(dys, dtype=cp.int32),
            cp.asarray(dxs, dtype=cp.int32),
            cp.asarray(costs, dtype=cp.float64))


def make_periodic_channel_mask_gpu(shape_zyx: Tuple[int, int, int]) -> cp.ndarray:
    """Full traversable box. y/z exterior boundaries are treated as walls by wall moments."""
    D, H, W = shape_zyx
    return cp.ones((D, H, W), dtype=cp.bool_)


def make_regular_seeds_gpu(mask: cp.ndarray, stride_zyx: Tuple[int, int, int]) -> cp.ndarray:
    """Return seed flat indices as a GPU int64 array."""
    D, H, W = mask.shape
    sz, sy, sx = stride_zyx
    zz = cp.arange(sz // 2, D, sz, dtype=cp.int64)
    yy = cp.arange(sy // 2, H, sy, dtype=cp.int64)
    xx = cp.arange(sx // 2, W, sx, dtype=cp.int64)
    Z, Y, X = cp.meshgrid(zz, yy, xx, indexing="ij")
    flat = (Z * H + Y) * W + X
    flat = flat.ravel()
    valid = mask.ravel()[flat]
    flat = flat[valid]
    if int(flat.size) < 1:
        raise RuntimeError("No valid seeds generated on GPU.")
    return flat.astype(cp.int64)


def geodesic_voronoi_labels_gpu(mask: cp.ndarray, seed_flat: cp.ndarray, cfg: PB615Config) -> Tuple[cp.ndarray, cp.ndarray]:
    """GPU-only parallel Bellman relaxation for multi-source graph distance labels.

    This is an all-GPU exact relaxation in the finite-graph sense after enough
    iterations. It is intended as a strict GPU reference/prototype. Production
    scaling should replace it with the ROI-JFA GPU kernels while keeping the same
    labels/dist output contract.
    """
    D, H, W = [int(v) for v in mask.shape]
    N = D * H * W
    INF = 1.0e100
    dist_a = cp.full(N, INF, dtype=cp.float64)
    label_a = cp.full(N, -1, dtype=cp.int32)
    seed_flat = seed_flat.astype(cp.int64, copy=False)
    nseed = int(seed_flat.size)
    dist_a[seed_flat] = 0.0
    label_a[seed_flat] = cp.arange(nseed, dtype=cp.int32)

    dist_b = cp.empty_like(dist_a)
    label_b = cp.empty_like(label_a)
    changed = cp.zeros(1, dtype=cp.int32)
    dzs, dys, dxs, costs = neighbor_offsets_gpu(cfg.distance_connectivity, cfg.voxel_size)
    grid, block = _threads_blocks(N)

    mask_u8 = mask.ravel().astype(cp.uint8, copy=False)
    for it in range(int(cfg.geodesic_max_iters)):
        changed.fill(0)
        _GEODESIC_RELAX_KERNEL(
            grid, block,
            (mask_u8, dist_a, label_a, dist_b, label_b, changed,
             dzs, dys, dxs, costs, int(dzs.size), D, H, W, int(cfg.periodic_x),
             float(cfg.geodesic_tie_eps), float(INF))
        )
        dist_a, dist_b = dist_b, dist_a
        label_a, label_b = label_b, label_a
        # One scalar device->host check. Computation remains GPU-only.
        if int(changed.get()[0]) == 0:
            break
    else:
        raise RuntimeError(
            f"Geodesic GPU relaxation did not converge within {cfg.geodesic_max_iters} iterations. "
            "Increase geodesic_max_iters or replace with GPU ROI-JFA for large domains."
        )

    return dist_a.reshape(mask.shape), label_a.reshape(mask.shape)


def _aggregate_faces_by_key(keys: cp.ndarray,
                            ax: cp.ndarray,
                            ay: cp.ndarray,
                            az: cp.ndarray,
                            xc: cp.ndarray,
                            yc: cp.ndarray,
                            zc: cp.ndarray,
                            area_w: cp.ndarray,
                            n_cells: int) -> Dict[str, cp.ndarray]:
    if int(keys.size) == 0:
        empty_i = cp.empty(0, dtype=cp.int32)
        empty_f = cp.empty(0, dtype=cp.float64)
        return dict(keys=cp.empty(0, dtype=cp.int64), owner=empty_i, neigh=empty_i,
                    area=empty_f, ax=empty_f, ay=empty_f, az=empty_f,
                    x0=empty_f, y0=empty_f, z0=empty_f)
    uniq, inv = cp.unique(keys, return_inverse=True)
    nf = int(uniq.size)
    area = cp.bincount(inv, weights=area_w, minlength=nf).astype(cp.float64)
    sum_ax = cp.bincount(inv, weights=ax, minlength=nf).astype(cp.float64)
    sum_ay = cp.bincount(inv, weights=ay, minlength=nf).astype(cp.float64)
    sum_az = cp.bincount(inv, weights=az, minlength=nf).astype(cp.float64)
    sum_x = cp.bincount(inv, weights=xc * area_w, minlength=nf).astype(cp.float64)
    sum_y = cp.bincount(inv, weights=yc * area_w, minlength=nf).astype(cp.float64)
    sum_z = cp.bincount(inv, weights=zc * area_w, minlength=nf).astype(cp.float64)
    owner = (uniq // n_cells).astype(cp.int32)
    neigh = (uniq % n_cells).astype(cp.int32)
    return dict(keys=uniq, owner=owner, neigh=neigh, area=area,
                ax=sum_ax, ay=sum_ay, az=sum_az,
                x0=sum_x / cp.maximum(area, 1.0e-300),
                y0=sum_y / cp.maximum(area, 1.0e-300),
                z0=sum_z / cp.maximum(area, 1.0e-300))


def build_positive_area_faces_gpu(mask: cp.ndarray, labels: cp.ndarray, n_cells: int, cfg: PB615Config) -> Dict[str, cp.ndarray]:
    """Extract and aggregate positive-area intercell facelets on GPU."""
    h = float(cfg.voxel_size)
    area0 = h * h
    D, H, W = [int(v) for v in labels.shape]
    all_keys = []
    all_ax = []
    all_ay = []
    all_az = []
    all_xc = []
    all_yc = []
    all_zc = []
    all_area = []

    def append_faces(li, lj, valid, base_vec, coord_builder):
        if int(valid.sum().get()) == 0:
            return
        idx = cp.nonzero(valid)
        lab_i = li[idx].astype(cp.int64)
        lab_j = lj[idx].astype(cp.int64)
        lo = cp.minimum(lab_i, lab_j)
        hi = cp.maximum(lab_i, lab_j)
        keys = lo * int(n_cells) + hi
        sign = cp.where(lab_i == lo, 1.0, -1.0).astype(cp.float64)
        x, y, z = coord_builder(idx)
        area = cp.full(keys.shape, area0, dtype=cp.float64)
        all_keys.append(keys.astype(cp.int64))
        all_ax.append(sign * float(base_vec[0]) * area0)
        all_ay.append(sign * float(base_vec[1]) * area0)
        all_az.append(sign * float(base_vec[2]) * area0)
        all_xc.append(x.astype(cp.float64))
        all_yc.append(y.astype(cp.float64))
        all_zc.append(z.astype(cp.float64))
        all_area.append(area)

    # +x faces
    if cfg.periodic_x:
        li = labels
        lj = cp.roll(labels, -1, axis=2)
        valid = mask & cp.roll(mask, -1, axis=2) & (li >= 0) & (lj >= 0) & (li != lj)
        def coords_x(idx):
            z, y, x = idx
            return (x.astype(cp.float64) + 1.0) * h, (y.astype(cp.float64) + 0.5) * h, (z.astype(cp.float64) + 0.5) * h
        append_faces(li, lj, valid, (1.0, 0.0, 0.0), coords_x)
    else:
        li = labels[:, :, :-1]
        lj = labels[:, :, 1:]
        valid = mask[:, :, :-1] & mask[:, :, 1:] & (li >= 0) & (lj >= 0) & (li != lj)
        def coords_x_np(idx):
            z, y, x = idx
            return (x.astype(cp.float64) + 1.0) * h, (y.astype(cp.float64) + 0.5) * h, (z.astype(cp.float64) + 0.5) * h
        append_faces(li, lj, valid, (1.0, 0.0, 0.0), coords_x_np)

    # +y faces
    li = labels[:, :-1, :]
    lj = labels[:, 1:, :]
    valid = mask[:, :-1, :] & mask[:, 1:, :] & (li >= 0) & (lj >= 0) & (li != lj)
    def coords_y(idx):
        z, y, x = idx
        return (x.astype(cp.float64) + 0.5) * h, (y.astype(cp.float64) + 1.0) * h, (z.astype(cp.float64) + 0.5) * h
    append_faces(li, lj, valid, (0.0, 1.0, 0.0), coords_y)

    # +z faces
    li = labels[:-1, :, :]
    lj = labels[1:, :, :]
    valid = mask[:-1, :, :] & mask[1:, :, :] & (li >= 0) & (lj >= 0) & (li != lj)
    def coords_z(idx):
        z, y, x = idx
        return (x.astype(cp.float64) + 0.5) * h, (y.astype(cp.float64) + 0.5) * h, (z.astype(cp.float64) + 1.0) * h
    append_faces(li, lj, valid, (0.0, 0.0, 1.0), coords_z)

    if not all_keys:
        raise RuntimeError("No positive-area intercell faces were found; increase seed count or check mask.")

    keys = cp.concatenate(all_keys)
    ax = cp.concatenate(all_ax)
    ay = cp.concatenate(all_ay)
    az = cp.concatenate(all_az)
    xc = cp.concatenate(all_xc)
    yc = cp.concatenate(all_yc)
    zc = cp.concatenate(all_zc)
    aw = cp.concatenate(all_area)
    return _aggregate_faces_by_key(keys, ax, ay, az, xc, yc, zc, aw, n_cells)


def build_cells_gpu(mask: cp.ndarray, labels: cp.ndarray, n_cells: int, cfg: PB615Config) -> Tuple[cp.ndarray, cp.ndarray]:
    h = float(cfg.voxel_size)
    D, H, W = [int(v) for v in labels.shape]
    flat_lab = labels.ravel()
    valid = mask.ravel() & (flat_lab >= 0)
    lab = flat_lab[valid].astype(cp.int32)
    count = cp.bincount(lab, minlength=n_cells).astype(cp.float64)
    if bool(cp.any(count <= 0).get()):
        raise RuntimeError("At least one seed has no assigned voxel; remove empty seeds or rebuild labels.")
    zz, yy, xx = cp.indices(labels.shape, dtype=cp.float64)
    xc = ((xx.ravel()[valid] + 0.5) * h).astype(cp.float64)
    yc = ((yy.ravel()[valid] + 0.5) * h).astype(cp.float64)
    zc = ((zz.ravel()[valid] + 0.5) * h).astype(cp.float64)
    sx = cp.bincount(lab, weights=xc, minlength=n_cells).astype(cp.float64)
    sy = cp.bincount(lab, weights=yc, minlength=n_cells).astype(cp.float64)
    sz = cp.bincount(lab, weights=zc, minlength=n_cells).astype(cp.float64)
    volume = count * (h ** 3)
    centroid = cp.stack([sx / count, sy / count, sz / count], axis=1)
    return volume, centroid


def build_wall_moments_gpu(mask: cp.ndarray, labels: cp.ndarray, centroid: cp.ndarray, n_cells: int, cfg: PB615Config) -> cp.ndarray:
    h = float(cfg.voxel_size)
    area0 = h * h
    D, H, W = [int(v) for v in labels.shape]
    accum_lab = []
    accum_w = []

    def add_wall(valid, lab_arr, coord_builder, normal):
        if int(valid.sum().get()) == 0:
            return
        idx = cp.nonzero(valid)
        lab = lab_arr[idx].astype(cp.int32)
        x, y, z = coord_builder(idx)
        c = centroid[lab]
        nx, ny, nz = [float(v) for v in normal]
        delta = (x - c[:, 0]) * nx + (y - c[:, 1]) * ny + (z - c[:, 2]) * nz
        delta = cp.maximum(delta, float(cfg.wall_distance_floor) * h)
        accum_lab.append(lab)
        accum_w.append(cp.full(lab.shape, area0, dtype=cp.float64) / delta)

    # x- wall/outside, skip periodic x outside. Interior solid still wall.
    if not cfg.periodic_x:
        valid = mask[:, :, 0] & (labels[:, :, 0] >= 0)
        def c_xm(idx):
            z, y = idx
            return cp.zeros_like(y, dtype=cp.float64) * h, (y.astype(cp.float64) + 0.5) * h, (z.astype(cp.float64) + 0.5) * h
        add_wall(valid, labels[:, :, 0], c_xm, (-1.0, 0.0, 0.0))
        valid = mask[:, :, -1] & (labels[:, :, -1] >= 0)
        def c_xp(idx):
            z, y = idx
            return cp.full_like(y, W, dtype=cp.float64) * h, (y.astype(cp.float64) + 0.5) * h, (z.astype(cp.float64) + 0.5) * h
        add_wall(valid, labels[:, :, -1], c_xp, (1.0, 0.0, 0.0))

    # Interior solid faces in x even if x periodic.
    li = labels[:, :, :-1]; lj = labels[:, :, 1:]
    mi = mask[:, :, :-1]; mj = mask[:, :, 1:]
    valid = mi & (~mj) & (li >= 0)
    def c_xp_int(idx):
        z, y, x = idx
        return (x.astype(cp.float64) + 1.0) * h, (y.astype(cp.float64) + 0.5) * h, (z.astype(cp.float64) + 0.5) * h
    add_wall(valid, li, c_xp_int, (1.0, 0.0, 0.0))
    valid = (~mi) & mj & (lj >= 0)
    def c_xm_int(idx):
        z, y, x = idx
        return (x.astype(cp.float64) + 1.0) * h, (y.astype(cp.float64) + 0.5) * h, (z.astype(cp.float64) + 0.5) * h
    add_wall(valid, lj, c_xm_int, (-1.0, 0.0, 0.0))

    # y outside boundaries and interior solid faces.
    valid = mask[:, 0, :] & (labels[:, 0, :] >= 0)
    def c_ym(idx):
        z, x = idx
        return (x.astype(cp.float64) + 0.5) * h, cp.zeros_like(x, dtype=cp.float64) * h, (z.astype(cp.float64) + 0.5) * h
    add_wall(valid, labels[:, 0, :], c_ym, (0.0, -1.0, 0.0))
    valid = mask[:, -1, :] & (labels[:, -1, :] >= 0)
    def c_yp(idx):
        z, x = idx
        return (x.astype(cp.float64) + 0.5) * h, cp.full_like(x, H, dtype=cp.float64) * h, (z.astype(cp.float64) + 0.5) * h
    add_wall(valid, labels[:, -1, :], c_yp, (0.0, 1.0, 0.0))
    li = labels[:, :-1, :]; lj = labels[:, 1:, :]
    mi = mask[:, :-1, :]; mj = mask[:, 1:, :]
    valid = mi & (~mj) & (li >= 0)
    def c_yp_int(idx):
        z, y, x = idx
        return (x.astype(cp.float64) + 0.5) * h, (y.astype(cp.float64) + 1.0) * h, (z.astype(cp.float64) + 0.5) * h
    add_wall(valid, li, c_yp_int, (0.0, 1.0, 0.0))
    valid = (~mi) & mj & (lj >= 0)
    def c_ym_int(idx):
        z, y, x = idx
        return (x.astype(cp.float64) + 0.5) * h, (y.astype(cp.float64) + 1.0) * h, (z.astype(cp.float64) + 0.5) * h
    add_wall(valid, lj, c_ym_int, (0.0, -1.0, 0.0))

    # z outside boundaries and interior solid faces.
    valid = mask[0, :, :] & (labels[0, :, :] >= 0)
    def c_zm(idx):
        y, x = idx
        return (x.astype(cp.float64) + 0.5) * h, (y.astype(cp.float64) + 0.5) * h, cp.zeros_like(x, dtype=cp.float64) * h
    add_wall(valid, labels[0, :, :], c_zm, (0.0, 0.0, -1.0))
    valid = mask[-1, :, :] & (labels[-1, :, :] >= 0)
    def c_zp(idx):
        y, x = idx
        return (x.astype(cp.float64) + 0.5) * h, (y.astype(cp.float64) + 0.5) * h, cp.full_like(x, D, dtype=cp.float64) * h
    add_wall(valid, labels[-1, :, :], c_zp, (0.0, 0.0, 1.0))
    li = labels[:-1, :, :]; lj = labels[1:, :, :]
    mi = mask[:-1, :, :]; mj = mask[1:, :, :]
    valid = mi & (~mj) & (li >= 0)
    def c_zp_int(idx):
        z, y, x = idx
        return (x.astype(cp.float64) + 0.5) * h, (y.astype(cp.float64) + 0.5) * h, (z.astype(cp.float64) + 1.0) * h
    add_wall(valid, li, c_zp_int, (0.0, 0.0, 1.0))
    valid = (~mi) & mj & (lj >= 0)
    def c_zm_int(idx):
        z, y, x = idx
        return (x.astype(cp.float64) + 0.5) * h, (y.astype(cp.float64) + 0.5) * h, (z.astype(cp.float64) + 1.0) * h
    add_wall(valid, lj, c_zm_int, (0.0, 0.0, -1.0))

    if not accum_lab:
        return cp.zeros(n_cells, dtype=cp.float64)
    labs = cp.concatenate(accum_lab)
    weights = cp.concatenate(accum_w)
    return cp.bincount(labs, weights=weights, minlength=n_cells).astype(cp.float64)


def build_geometry_gpu(mask: cp.ndarray, labels: cp.ndarray, dist: cp.ndarray, seed_flat: cp.ndarray, cfg: PB615Config) -> GPUFVMGeometry:
    n_cells = int(seed_flat.size)
    volume, centroid = build_cells_gpu(mask, labels, n_cells, cfg)
    faces = build_positive_area_faces_gpu(mask, labels, n_cells, cfg)
    owner = faces["owner"]
    neigh = faces["neigh"]
    area = faces["area"].astype(cp.float64)
    avec = cp.stack([faces["ax"], faces["ay"], faces["az"]], axis=1).astype(cp.float64)
    face_centroid = cp.stack([faces["x0"], faces["y0"], faces["z0"]], axis=1).astype(cp.float64)

    ci = centroid[owner]
    cj = centroid[neigh]
    dvec = cj - ci
    if cfg.periodic_x:
        Lx = int(mask.shape[2]) * float(cfg.voxel_size)
        dx = dvec[:, 0]
        dvec[:, 0] = cp.where(dx > 0.5 * Lx, dx - Lx, cp.where(dx < -0.5 * Lx, dx + Lx, dx))

    # Euclidean interpolation weights from face centroid to cell centroids.
    # For periodic x faces, unwrap face-centroid x relative to each centroid minimally.
    xi = ci.copy(); xj = cj.copy(); xf = face_centroid.copy()
    if cfg.periodic_x:
        Lx = int(mask.shape[2]) * float(cfg.voxel_size)
        dfi = xf[:, 0] - xi[:, 0]
        dfj = xf[:, 0] - xj[:, 0]
        dfi = cp.where(dfi > 0.5 * Lx, dfi - Lx, cp.where(dfi < -0.5 * Lx, dfi + Lx, dfi))
        dfj = cp.where(dfj > 0.5 * Lx, dfj - Lx, cp.where(dfj < -0.5 * Lx, dfj + Lx, dfj))
        ri = cp.sqrt(dfi*dfi + (xf[:,1]-xi[:,1])**2 + (xf[:,2]-xi[:,2])**2)
        rj = cp.sqrt(dfj*dfj + (xf[:,1]-xj[:,1])**2 + (xf[:,2]-xj[:,2])**2)
    else:
        ri = cp.linalg.norm(face_centroid - ci, axis=1)
        rj = cp.linalg.norm(face_centroid - cj, axis=1)
    denom = cp.maximum(ri + rj, 1.0e-300)
    w_owner = rj / denom
    w_neigh = 1.0 - w_owner

    adotd = cp.sum(avec * dvec, axis=1)
    anorm2 = cp.sum(avec * avec, axis=1)
    dnorm = cp.maximum(cp.linalg.norm(dvec, axis=1), 1.0e-300)
    fallback = area / dnorm
    raw = anorm2 / cp.maximum(adotd, 1.0e-300)
    lo, hi = cfg.transmissibility_ratio_clip
    ratio = raw / cp.maximum(fallback, 1.0e-300)
    tproj = cp.where((adotd > 0.0) & (ratio >= lo) & (ratio <= hi), raw, fallback)
    tproj = cp.maximum(tproj, float(cfg.transmissibility_floor))

    twall = build_wall_moments_gpu(mask, labels, centroid, n_cells, cfg)

    # GPU sparse graph Laplacian with tiny gauge regularization.
    nf = int(owner.size)
    rows = cp.concatenate([owner, owner, neigh, neigh]).astype(cp.int32)
    cols = cp.concatenate([owner, neigh, neigh, owner]).astype(cp.int32)
    vals = cp.concatenate([tproj, -tproj, tproj, -tproj]).astype(cp.float64)
    L = cpx_sp.coo_matrix((vals, (rows, cols)), shape=(n_cells, n_cells)).tocsr()
    if cfg.pressure_gauge_eps > 0:
        L = L + float(cfg.pressure_gauge_eps) * cpx_sp.eye(n_cells, dtype=cp.float64, format="csr")

    return GPUFVMGeometry(mask=mask, labels=labels, dist=dist, n_cells=n_cells,
                          volume=volume, centroid=centroid, owner=owner, neigh=neigh,
                          area=area, avec=avec, face_centroid=face_centroid, dvec=dvec,
                          tproj=tproj, w_owner=w_owner, w_neigh=w_neigh, twall=twall,
                          laplacian=L)


def face_divergence_gpu(phi: cp.ndarray, geom: GPUFVMGeometry) -> cp.ndarray:
    n = geom.n_cells
    nf = int(geom.owner.size)
    div = cp.zeros(n, dtype=cp.float64)
    grid, block = _threads_blocks(nf)
    _DIVERGENCE_KERNEL(grid, block, (nf, geom.owner, geom.neigh, phi, div))
    return div


def face_flux_from_velocity_gpu(U: cp.ndarray, geom: GPUFVMGeometry) -> cp.ndarray:
    nf = int(geom.owner.size)
    phi = cp.empty(nf, dtype=cp.float64)
    grid, block = _threads_blocks(nf)
    _FACE_FLUX_KERNEL(grid, block, (nf, geom.owner, geom.neigh, geom.w_owner, geom.w_neigh, geom.avec.reshape(-1), U.reshape(-1), phi))
    return phi


def lsq_gradient_gpu(scalar: cp.ndarray, geom: GPUFVMGeometry, cfg: PB615Config) -> cp.ndarray:
    n = geom.n_cells
    nf = int(geom.owner.size)
    M = cp.zeros((n, 3, 3), dtype=cp.float64)
    b = cp.zeros((n, 3), dtype=cp.float64)
    grid, block = _threads_blocks(nf)
    _LSQ_ASSEMBLE_KERNEL(grid, block, (nf, geom.owner, geom.neigh, geom.dvec.reshape(-1), geom.tproj, scalar, M.reshape(-1), b.reshape(-1)))
    diag = cp.arange(3)
    M[:, diag, diag] += float(cfg.tikhonov)
    return cp.linalg.solve(M, b)


def diffusion_term_gpu(U: cp.ndarray, geom: GPUFVMGeometry, cfg: PB615Config) -> cp.ndarray:
    n = geom.n_cells
    nf = int(geom.owner.size)
    Dterm = cp.zeros_like(U)
    grid_f, block_f = _threads_blocks(nf)
    _DIFFUSION_KERNEL(grid_f, block_f, (nf, geom.owner, geom.neigh, geom.tproj, geom.volume, U.reshape(-1), Dterm.reshape(-1), float(cfg.nu)))
    grid_c, block_c = _threads_blocks(n)
    _WALL_DIFFUSION_KERNEL(grid_c, block_c, (n, geom.volume, geom.twall, U.reshape(-1), Dterm.reshape(-1), float(cfg.nu)))
    return Dterm


def convection_term_gpu(U: cp.ndarray, phi: cp.ndarray, geom: GPUFVMGeometry) -> cp.ndarray:
    nf = int(geom.owner.size)
    Cterm = cp.zeros_like(U)
    grid, block = _threads_blocks(nf)
    _CONVECTION_KERNEL(grid, block, (nf, geom.owner, geom.neigh, geom.volume, phi, U.reshape(-1), Cterm.reshape(-1)))
    return Cterm



def _make_jacobi_preconditioner_gpu(A):
    """Return a GPU Jacobi preconditioner as a cupyx LinearOperator.

    This is intentionally lightweight and all-GPU.  It improves the robustness of
    the implicit momentum and pressure-correction CG solves in high-CFL stress
    scans.  If a particular cupyx build does not expose LinearOperator, the
    caller can pass M=None and the solve remains GPU-only.
    """
    try:
        diag = A.diagonal().astype(cp.float64)
        safe = cp.where(cp.abs(diag) > 1.0e-300, diag, 1.0)
        def _mv(x):
            return x / safe
        return cpx_spla.LinearOperator(A.shape, matvec=_mv, dtype=cp.float64)
    except Exception:
        return None


def _cg_gpu_compat(A, b, cfg: PB615Config, M=None):
    """CuPy CG compatibility wrapper with optional Jacobi preconditioning."""
    attempts = []
    if M is not None:
        attempts.append(dict(tol=float(cfg.projection_tol), maxiter=int(cfg.projection_maxiter), M=M))
        attempts.append(dict(rtol=float(cfg.projection_tol), atol=0.0, maxiter=int(cfg.projection_maxiter), M=M))
    attempts.append(dict(tol=float(cfg.projection_tol), maxiter=int(cfg.projection_maxiter)))
    attempts.append(dict(rtol=float(cfg.projection_tol), atol=0.0, maxiter=int(cfg.projection_maxiter)))
    last = None
    for kw in attempts:
        try:
            return cpx_spla.cg(A, b, **kw)
        except TypeError as exc:
            last = exc
            continue
    raise last if last is not None else RuntimeError("cupyx CG call failed")


def solve_pressure_correction_gpu(rhs: cp.ndarray, geom: GPUFVMGeometry, cfg: PB615Config, M_p=None) -> Tuple[cp.ndarray, int]:
    # Remove nullspace component on device for Neumann-like periodic cases.
    rhs = rhs - cp.mean(rhs)
    pcorr, info = _cg_gpu_compat(geom.laplacian, rhs, cfg, M=M_p)
    info_i = int(info)
    if info_i != 0:
        # info is a small host scalar from cupyx; keep failure explicit.
        raise RuntimeError(f"GPU CG did not converge; info={info_i}")
    pcorr = pcorr - cp.mean(pcorr)
    return pcorr, info_i


def correct_flux_gpu(phi_star: cp.ndarray, pcorr: cp.ndarray, geom: GPUFVMGeometry, cfg: PB615Config) -> cp.ndarray:
    nf = int(geom.owner.size)
    phi = cp.empty_like(phi_star)
    grid, block = _threads_blocks(nf)
    _CORRECT_FLUX_KERNEL(grid, block, (nf, geom.owner, geom.neigh, geom.tproj, pcorr, phi_star, phi, float(cfg.dt / cfg.rho)))
    return phi


def reconstruct_velocity_from_flux_gpu(phi: cp.ndarray, U_prior: cp.ndarray, geom: GPUFVMGeometry, cfg: PB615Config) -> cp.ndarray:
    n = geom.n_cells
    nf = int(geom.owner.size)
    M = cp.zeros((n, 3, 3), dtype=cp.float64)
    b = cp.zeros((n, 3), dtype=cp.float64)
    grid, block = _threads_blocks(nf)
    _FLUX_RECON_ASSEMBLE_KERNEL(grid, block, (nf, geom.owner, geom.neigh, geom.avec.reshape(-1), geom.area, phi, M.reshape(-1), b.reshape(-1)))
    lam = float(cfg.velocity_reconstruction_lambda)
    diag = cp.arange(3)
    M[:, diag, diag] += lam + float(cfg.tikhonov)
    b += lam * U_prior
    return cp.linalg.solve(M, b)




def _cg_gpu_tol(A, b, tol: float, maxiter: int, M=None):
    """CuPy CG wrapper with optional Jacobi preconditioning and API compatibility."""
    attempts = []
    if M is not None:
        attempts.append(dict(tol=float(tol), maxiter=int(maxiter), M=M))
        attempts.append(dict(rtol=float(tol), atol=0.0, maxiter=int(maxiter), M=M))
    attempts.append(dict(tol=float(tol), maxiter=int(maxiter)))
    attempts.append(dict(rtol=float(tol), atol=0.0, maxiter=int(maxiter)))
    last = None
    for kw in attempts:
        try:
            return cpx_spla.cg(A, b, **kw)
        except TypeError as exc:
            last = exc
            continue
    raise last if last is not None else RuntimeError("cupyx CG call failed")


def build_implicit_velocity_matrix_gpu(geom: GPUFVMGeometry, cfg: PB615Config):
    """Build the all-GPU SPD matrix for implicit viscous Stokes momentum prediction.

    The solved equation for each velocity component is

        (V_i/dt) U_i^* - nu * [sum_j T_ij (U_j^* - U_i^*) - T_wall_i U_i^*] = RHS_i.

    This keeps the viscous and wall no-slip resistance implicit, so a body-force
    channel test converges to a steady field instead of just accumulating velocity
    for a short explicit pseudo-time window.
    """
    n = geom.n_cells
    nf = int(geom.owner.size)
    diag_index = cp.arange(n, dtype=cp.int32)
    rows = cp.concatenate([geom.owner, geom.owner, geom.neigh, geom.neigh, diag_index, diag_index]).astype(cp.int32)
    cols = cp.concatenate([geom.owner, geom.neigh, geom.neigh, geom.owner, diag_index, diag_index]).astype(cp.int32)
    vals = cp.concatenate([
        float(cfg.nu) * geom.tproj,
        -float(cfg.nu) * geom.tproj,
        float(cfg.nu) * geom.tproj,
        -float(cfg.nu) * geom.tproj,
        geom.volume / float(cfg.dt),
        float(cfg.nu) * geom.twall,
    ]).astype(cp.float64)
    A = cpx_sp.coo_matrix((vals, (rows, cols)), shape=(n, n)).tocsr()
    return A


def implicit_momentum_predictor_gpu(
    U: cp.ndarray,
    p: cp.ndarray,
    phi: cp.ndarray,
    geom: GPUFVMGeometry,
    cfg: PB615Config,
    A_vel,
    M_vel=None,
) -> cp.ndarray:
    """Implicit all-GPU Stokes/N-S predictor for cell-centered velocity."""
    gradp = lsq_gradient_gpu(p, geom, cfg)
    if cfg.enable_convection:
        Cterm = convection_term_gpu(U, phi, geom)
    else:
        Cterm = cp.zeros_like(U)
    body = cp.asarray(cfg.body_force, dtype=cp.float64)[None, :]
    rhs = (geom.volume[:, None] / float(cfg.dt)) * U + geom.volume[:, None] * (body - gradp / float(cfg.rho) - Cterm)
    Ustar = cp.empty_like(U)
    for c in range(3):
        sol, info = _cg_gpu_tol(A_vel, rhs[:, c], float(cfg.momentum_tol), int(cfg.momentum_maxiter), M=M_vel)
        info_i = int(info)
        if info_i != 0:
            raise RuntimeError(f"GPU momentum CG did not converge for component {c}; info={info_i}")
        Ustar[:, c] = sol
    return Ustar


def steady_momentum_residual_gpu(U: cp.ndarray, p: cp.ndarray, phi: cp.ndarray, geom: GPUFVMGeometry, cfg: PB615Config) -> cp.ndarray:
    """Return steady residual -C + D + body - grad(p)/rho on the GPU."""
    gradp = lsq_gradient_gpu(p, geom, cfg)
    Dterm = diffusion_term_gpu(U, geom, cfg)
    if cfg.enable_convection:
        Cterm = convection_term_gpu(U, phi, geom)
    else:
        Cterm = cp.zeros_like(U)
    body = cp.asarray(cfg.body_force, dtype=cp.float64)[None, :]
    return -Cterm + Dterm + body - gradp / float(cfg.rho)



def make_initial_velocity_gpu(geom: GPUFVMGeometry, cfg: PB615Config) -> cp.ndarray:
    """GPU-only initial velocity builder for N-S/stability diagnostics."""
    n = geom.n_cells
    U = cp.zeros((n, 3), dtype=cp.float64)
    mode = str(getattr(cfg, "initial_velocity_mode", "zero")).lower()
    amp = float(getattr(cfg, "initial_velocity_amplitude", 0.0))
    if mode in ("zero", "none") or amp == 0.0:
        return U
    x = geom.centroid[:, 0]
    y = geom.centroid[:, 1]
    z = geom.centroid[:, 2]
    D, H, W = [float(v) for v in cfg.demo_shape_zyx]
    h = float(cfg.voxel_size)
    Lx = max(W * h, h)
    Ly = max(H * h, h)
    Lz = max(D * h, h)
    twopi = 2.0 * math.pi
    if mode == "constant_x":
        U[:, 0] = amp
    elif mode == "manufactured_advective":
        U[:, 0] = amp * (cp.sin(twopi * y / Ly) + 0.25 * cp.cos(twopi * z / Lz))
        U[:, 1] = 0.50 * amp * cp.sin(twopi * x / Lx + 0.25 * cp.cos(twopi * z / Lz))
        U[:, 2] = 0.25 * amp * cp.cos(twopi * x / Lx + twopi * y / Ly)
    else:
        raise ValueError(f"Unknown initial_velocity_mode={cfg.initial_velocity_mode!r}")
    return U


def kinetic_energy_gpu(U: cp.ndarray, geom: GPUFVMGeometry) -> cp.ndarray:
    return 0.5 * cp.sum(geom.volume * cp.sum(U * U, axis=1)) / cp.maximum(cp.sum(geom.volume), 1.0e-300)


def convective_cfl_gpu(phi: cp.ndarray, geom: GPUFVMGeometry, cfg: PB615Config) -> cp.ndarray:
    n = geom.n_cells
    out_owner = cp.maximum(phi, 0.0)
    out_neigh = cp.maximum(-phi, 0.0)
    out = cp.bincount(geom.owner, weights=out_owner, minlength=n).astype(cp.float64)
    out += cp.bincount(geom.neigh, weights=out_neigh, minlength=n).astype(cp.float64)
    local = float(cfg.dt) * out / cp.maximum(geom.volume, 1.0e-300)
    return cp.max(local)


def diffusion_stiffness_number_gpu(geom: GPUFVMGeometry, cfg: PB615Config) -> cp.ndarray:
    n = geom.n_cells
    diag = cp.bincount(geom.owner, weights=geom.tproj, minlength=n).astype(cp.float64)
    diag += cp.bincount(geom.neigh, weights=geom.tproj, minlength=n).astype(cp.float64)
    diag += geom.twall
    return cp.max(float(cfg.dt) * float(cfg.nu) * diag / cp.maximum(geom.volume, 1.0e-300))


def _finite_gpu_scalar(x: float) -> bool:
    return math.isfinite(float(x))


def run_velocity_pressure_projection_gpu(geom: GPUFVMGeometry, cfg: PB615Config, initial_U: Optional[cp.ndarray] = None, run_label: str = "main") -> Dict[str, cp.ndarray | float | int]:
    n = geom.n_cells
    U = initial_U.copy() if initial_U is not None else make_initial_velocity_gpu(geom, cfg)
    p = cp.zeros(n, dtype=cp.float64)
    phi = face_flux_from_velocity_gpu(U, geom)
    A_vel = build_implicit_velocity_matrix_gpu(geom, cfg) if bool(cfg.implicit_diffusion) else None
    M_vel = _make_jacobi_preconditioner_gpu(A_vel) if (A_vel is not None and bool(getattr(cfg, "use_jacobi_preconditioner", True))) else None
    M_p = _make_jacobi_preconditioner_gpu(geom.laplacian) if bool(getattr(cfg, "use_jacobi_preconditioner", True)) else None

    t0 = time.perf_counter()
    last_mass_inf = None
    steady_converged = False
    rel_change = float('inf')
    mom_inf = float('inf')
    U_prev = U.copy()
    U0_norm_inf = float(cp.max(cp.linalg.norm(U, axis=1)).get())
    energy_initial = float(kinetic_energy_gpu(U, geom).get())
    energy_max = energy_initial
    max_cfl = 0.0
    max_convection_inf = 0.0
    rel_change_ref = float('inf')
    max_diffusion_number = float(diffusion_stiffness_number_gpu(geom, cfg).get())
    nan_or_inf_detected = False
    history = []
    solver_failure = ""
    failure_step = -1
    steps_completed = 0
    energy_now = energy_initial
    cfl_now = float(convective_cfl_gpu(phi, geom, cfg).get())

    for step in range(1, int(cfg.n_steps) + 1):
        try:
            if bool(cfg.implicit_diffusion):
                Ustar = implicit_momentum_predictor_gpu(U, p, phi, geom, cfg, A_vel, M_vel=M_vel)
            else:
                gradp = lsq_gradient_gpu(p, geom, cfg)
                Dterm = diffusion_term_gpu(U, geom, cfg)
                if cfg.enable_convection:
                    Cterm = convection_term_gpu(U, phi, geom)
                else:
                    Cterm = cp.zeros_like(U)
                body = cp.asarray(cfg.body_force, dtype=cp.float64)[None, :]
                Ustar = U + float(cfg.dt) * (-Cterm + Dterm + body - gradp / float(cfg.rho))

            phi_star = face_flux_from_velocity_gpu(Ustar, geom)
            div_star = face_divergence_gpu(phi_star, geom)
            rhs = -(float(cfg.rho) / float(cfg.dt)) * div_star
            pcorr, _ = solve_pressure_correction_gpu(rhs, geom, cfg, M_p=M_p)
            phi = correct_flux_gpu(phi_star, pcorr, geom, cfg)
            grad_pcorr = lsq_gradient_gpu(pcorr, geom, cfg)
            U = Ustar - float(cfg.dt / cfg.rho) * grad_pcorr
            if cfg.velocity_reconstruct_from_flux:
                U = reconstruct_velocity_from_flux_gpu(phi, U, geom, cfg)
            p = p + pcorr
            p = p - cp.mean(p)
            steps_completed = int(step)
        except RuntimeError as exc:
            solver_failure = str(exc)
            failure_step = int(step)
            nan_or_inf_detected = True
            if bool(getattr(cfg, "raise_on_solver_failure", True)):
                raise
            print(f"[PB61.8-GPU] {run_label} solver failure at step {step}: {solver_failure}")
            break

        do_diag = bool(getattr(cfg, "diagnostics_every_step", False)) or (cfg.report_every and (step % int(cfg.report_every) == 0 or step == 1 or step == cfg.n_steps))
        if do_diag:
            energy_now = float(kinetic_energy_gpu(U, geom).get())
            cfl_now = float(convective_cfl_gpu(phi, geom, cfg).get())
            energy_max = max(float(energy_max), energy_now)
            max_cfl = max(float(max_cfl), cfl_now)
            if not (_finite_gpu_scalar(energy_now) and _finite_gpu_scalar(cfl_now)):
                nan_or_inf_detected = True
            if bool(cfg.enable_convection):
                Cdiag = convection_term_gpu(U, phi, geom)
                conv_inf_now = float(cp.max(cp.linalg.norm(Cdiag, axis=1)).get())
            else:
                conv_inf_now = 0.0
            max_convection_inf = max(float(max_convection_inf), float(conv_inf_now))
            if bool(getattr(cfg, "diagnostics_every_step", False)):
                history.append({"step": int(step), "energy": energy_now, "cfl": cfl_now, "convection_inf": conv_inf_now})

        if cfg.report_every and (step % int(cfg.report_every) == 0 or step == 1 or step == cfg.n_steps):
            div = face_divergence_gpu(phi, geom)
            mass_inf = cp.max(cp.abs(div / geom.volume))
            last_mass_inf = float(mass_inf.get())
            umax = float(cp.max(cp.linalg.norm(U, axis=1)).get())
            dU = cp.max(cp.linalg.norm(U - U_prev, axis=1))
            Un = cp.maximum(cp.max(cp.linalg.norm(U, axis=1)), 1.0e-300)
            rel_change = float((dU / Un).get())
            rel_denom_ref = max(float(U0_norm_inf), float(Un.get()), 1.0e-300)
            rel_change_ref = float(dU.get()) / rel_denom_ref
            mom = steady_momentum_residual_gpu(U, p, phi, geom, cfg)
            mom_inf = float(cp.max(cp.linalg.norm(mom, axis=1)).get())
            conv_inf_print = float(max_convection_inf)
            print(f"[PB61.8-GPU] {run_label} step {step:5d}/{cfg.n_steps}: mass_inf/V={last_mass_inf:.3e}, umax={umax:.6e}, rel_dU={rel_change:.3e}, rel_dU_ref={rel_change_ref:.3e}, mom_inf={mom_inf:.3e}, CFL={cfl_now:.3e}, Cinf={conv_inf_print:.3e}, E={energy_now:.3e}")
            U_prev = U.copy()
            if bool(getattr(cfg, "stop_on_steady", True)) and step >= int(cfg.steady_min_steps) and rel_change < float(cfg.steady_tol) and mom_inf < max(10.0 * float(cfg.steady_tol), 1.0e-10):
                steady_converged = True
                print(f"[PB61.8-GPU] steady convergence reached at step {step}: rel_dU={rel_change:.3e}, mom_inf={mom_inf:.3e}")
                break

    elapsed = time.perf_counter() - t0
    div = face_divergence_gpu(phi, geom)
    mass_inf = float(cp.max(cp.abs(div / geom.volume)).get())
    mass_l2 = float(cp.sqrt(cp.mean((div / geom.volume) ** 2)).get())
    mean_U = cp.sum(U * geom.volume[:, None], axis=0) / cp.sum(geom.volume)
    mean_U_host = [float(v) for v in mean_U.get()]
    fx = float(cfg.body_force[0])
    k_eff_x = float((cfg.nu * mean_U[0] / fx).get()) if abs(fx) > 0 else float('nan')
    mom = steady_momentum_residual_gpu(U, p, phi, geom, cfg)
    mom_inf = float(cp.max(cp.linalg.norm(mom, axis=1)).get())
    energy_final = float(kinetic_energy_gpu(U, geom).get())
    return dict(U=U, p=p, phi=phi, div=div,
                elapsed_s=elapsed,
                mass_inf_per_volume=mass_inf,
                mass_l2_per_volume=mass_l2,
                mean_U_x=mean_U_host[0], mean_U_y=mean_U_host[1], mean_U_z=mean_U_host[2],
                K_eff_x=k_eff_x,
                n_cells=n, n_faces=int(geom.owner.size),
                last_mass_inf=last_mass_inf if last_mass_inf is not None else mass_inf,
                steady_converged=bool(steady_converged),
                final_rel_dU=float(rel_change),
                final_rel_dU_ref=float(rel_change_ref),
                steady_momentum_inf=float(mom_inf),
                steps_completed=int(steps_completed),
                failure_step=int(failure_step),
                solver_failure=str(solver_failure),
                implicit_diffusion=bool(cfg.implicit_diffusion),
                enable_convection=bool(cfg.enable_convection),
                energy_initial=float(energy_initial),
                energy_final=float(energy_final),
                energy_max=float(energy_max),
                max_cfl=float(max_cfl),
                max_convection_inf=float(max_convection_inf),
                max_diffusion_number=float(max_diffusion_number),
                nan_or_inf_detected=bool(nan_or_inf_detected),
                run_label=str(run_label),
                history=history)


# -----------------------------------------------------------------------------
# Output helpers. These transfer final arrays to host only after GPU computation.
# -----------------------------------------------------------------------------

def write_outputs(result: Dict[str, cp.ndarray | float | int], geom: GPUFVMGeometry, cfg: PB615Config) -> None:
    out = Path(cfg.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    summary = {
        "mode": "PB61.8_ALL_GPU_NS_CONVECTION_STABILITY_CUPY_RAWKERNEL_CUPYX",
        "n_cells": int(result["n_cells"]),
        "n_faces": int(result["n_faces"]),
        "elapsed_s": float(result["elapsed_s"]),
        "mass_inf_per_volume": float(result["mass_inf_per_volume"]),
        "mass_l2_per_volume": float(result["mass_l2_per_volume"]),
        "mean_U_x": float(result["mean_U_x"]),
        "mean_U_y": float(result["mean_U_y"]),
        "mean_U_z": float(result["mean_U_z"]),
        "K_eff_x": float(result["K_eff_x"]),
        "steady_converged": bool(result.get("steady_converged", False)),
        "steps_completed": int(result.get("steps_completed", -1)),
        "final_rel_dU": float(result.get("final_rel_dU", float("nan"))),
        "final_rel_dU_ref": float(result.get("final_rel_dU_ref", float("nan"))),
        "steady_momentum_inf": float(result.get("steady_momentum_inf", float("nan"))),
        "implicit_diffusion": bool(result.get("implicit_diffusion", False)),
        "enable_convection": bool(result.get("enable_convection", False)),
        "energy_initial": float(result.get("energy_initial", float("nan"))),
        "energy_final": float(result.get("energy_final", float("nan"))),
        "energy_max": float(result.get("energy_max", float("nan"))),
        "max_cfl": float(result.get("max_cfl", float("nan"))),
        "max_convection_inf": float(result.get("max_convection_inf", float("nan"))),
        "max_diffusion_number": float(result.get("max_diffusion_number", float("nan"))),
        "nan_or_inf_detected": bool(result.get("nan_or_inf_detected", False)),
        "run_label": str(result.get("run_label", "main")),
        "gpu_device": cp.cuda.runtime.getDeviceProperties(0)["name"].decode("utf-8") if isinstance(cp.cuda.runtime.getDeviceProperties(0)["name"], bytes) else str(cp.cuda.runtime.getDeviceProperties(0)["name"]),
        "note": "Computation GPU-only; CSV/JSON writing transfers final diagnostics to host. PB61.8 opens explicit upwind convection and reports CFL/energy/time-stability diagnostics. K_eff_x is valid as a steady permeability only when steady_converged is true or final_rel_dU/momentum residual are small."
    }
    (out / "pb61_7_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    with (out / "pb61_7_summary.csv").open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(summary.keys())
        w.writerow(summary.values())

    U = cp.asnumpy(result["U"])
    p = cp.asnumpy(result["p"])
    cent = cp.asnumpy(geom.centroid)
    vol = cp.asnumpy(geom.volume)
    with (out / "pb61_7_cell_state.csv").open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["cell", "volume", "x", "y", "z", "Ux", "Uy", "Uz", "p"])
        for i in range(U.shape[0]):
            w.writerow([i, vol[i], cent[i,0], cent[i,1], cent[i,2], U[i,0], U[i,1], U[i,2], p[i]])

    phi = cp.asnumpy(result["phi"])
    owner = cp.asnumpy(geom.owner)
    neigh = cp.asnumpy(geom.neigh)
    area = cp.asnumpy(geom.area)
    avec = cp.asnumpy(geom.avec)
    with (out / "pb61_7_face_flux.csv").open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["face", "owner", "neigh", "area", "ax", "ay", "az", "phi", "un"])
        for k in range(phi.shape[0]):
            un = phi[k] / max(area[k], 1.0e-300)
            w.writerow([k, owner[k], neigh[k], area[k], avec[k,0], avec[k,1], avec[k,2], phi[k], un])



    hist = result.get("history", [])
    if isinstance(hist, list) and len(hist) > 0:
        with (out / "pb61_7_history.csv").open("w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["step", "energy", "cfl", "convection_inf"])
            for row in hist:
                w.writerow([row.get("step"), row.get("energy"), row.get("cfl"), row.get("convection_inf")])


# -----------------------------------------------------------------------------
# PB61.8 validation helpers: convection activation and explicit time-stability
# -----------------------------------------------------------------------------

def build_demo_geometry_gpu(cfg: Optional[PB615Config] = None) -> GPUFVMGeometry:
    """Build the demo mask/labels/geometry once on GPU."""
    require_cuda_gpu()
    cfg = cfg or PB615Config()
    mask = make_periodic_channel_mask_gpu(cfg.demo_shape_zyx)
    seeds = make_regular_seeds_gpu(mask, cfg.demo_seed_stride_zyx)
    t0 = time.perf_counter()
    dist, labels = geodesic_voronoi_labels_gpu(mask, seeds, cfg)
    cp.cuda.Stream.null.synchronize()
    print(f"[PB61.8-GPU] demo geometry labels built in {time.perf_counter()-t0:.3f}s; seeds={int(seeds.size)}")
    geom = build_geometry_gpu(mask, labels, dist, seeds, cfg)
    cp.cuda.Stream.null.synchronize()
    print(f"[PB61.8-GPU] demo geometry: n_cells={geom.n_cells}, n_faces={int(geom.owner.size)}, wall_nonzero={int(cp.count_nonzero(geom.twall).get())}")
    return geom


def run_convective_term_diagnostic_gpu(cfg: Optional[PB615Config] = None) -> Dict[str, float]:
    """Verify that convection is zero/near-zero for a constant periodic streamwise field and nonzero for a manufactured advective field."""
    cfg = cfg or PB615Config()
    cfg.enable_convection = True
    geom = build_demo_geometry_gpu(cfg)
    U_const = cp.zeros((geom.n_cells, 3), dtype=cp.float64)
    U_const[:, 0] = float(max(cfg.convection_diagnostic_amplitude, 1.0e-3))
    phi_const = face_flux_from_velocity_gpu(U_const, geom)
    C_const = convection_term_gpu(U_const, phi_const, geom)
    cfg_adv = PB615Config(**{**cfg.__dict__})
    cfg_adv.initial_velocity_mode = "manufactured_advective"
    cfg_adv.initial_velocity_amplitude = float(cfg.convection_diagnostic_amplitude)
    U_adv = make_initial_velocity_gpu(geom, cfg_adv)
    phi_adv = face_flux_from_velocity_gpu(U_adv, geom)
    C_adv = convection_term_gpu(U_adv, phi_adv, geom)
    const_inf = float(cp.max(cp.linalg.norm(C_const, axis=1)).get())
    adv_inf = float(cp.max(cp.linalg.norm(C_adv, axis=1)).get())
    adv_l2 = float(cp.sqrt(cp.mean(cp.sum(C_adv * C_adv, axis=1))).get())
    adv_cfl = float(convective_cfl_gpu(phi_adv, geom, cfg_adv).get())
    activation_ratio = adv_inf / max(const_inf, 1.0e-300)
    diag = dict(constant_convection_inf=const_inf, manufactured_convection_inf=adv_inf,
                manufactured_convection_l2=adv_l2, manufactured_initial_cfl=adv_cfl,
                activation_ratio=activation_ratio, n_cells=int(geom.n_cells), n_faces=int(geom.owner.size))
    out = Path(cfg.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    with (out / "pb61_7_convective_term_diagnostic.csv").open("w", newline="") as f:
        w = csv.writer(f); w.writerow(diag.keys()); w.writerow(diag.values())
    print("[PB61.8-GPU] convective diagnostic:", diag)
    return diag


def _run_one_stability_family_gpu(base: PB615Config, geom: GPUFVMGeometry, *, family: str, dt_values: Tuple[float, ...], amplitude: float, n_steps: int, nu: float, energy_growth_fail: float) -> list:
    """Internal all-GPU stability-family runner."""
    rows = []
    for dt in tuple(dt_values):
        local = PB615Config(**{**base.__dict__})
        local.dt = float(dt)
        local.nu = float(nu)
        local.n_steps = int(n_steps)
        local.report_every = max(1, int(n_steps) // 4)
        local.enable_convection = True
        local.implicit_diffusion = True
        local.body_force = (0.0, 0.0, 0.0)
        local.initial_velocity_mode = "manufactured_advective"
        local.initial_velocity_amplitude = float(amplitude)
        local.stop_on_steady = False
        local.diagnostics_every_step = True
        # In a stability scan, solver non-convergence is a result, not a notebook-killing exception.
        local.raise_on_solver_failure = False
        print(f"[PB61.8-GPU] {family} stability scan dt={dt}, amp={amplitude}, nu={nu}")
        try:
            result = run_velocity_pressure_projection_gpu(geom, local, run_label=f"{family}_dt={dt:g}")
            growth = float(result["energy_max"]) / max(float(result["energy_initial"]), 1.0e-300)
            solver_failure = str(result.get("solver_failure", ""))
            stable = (solver_failure == "") and (not bool(result["nan_or_inf_detected"])) and math.isfinite(growth) and growth <= float(energy_growth_fail)
            hist = result.get("history", []) or []
            finite_hist_cfl = [float(h.get("cfl", float("nan"))) for h in hist if math.isfinite(float(h.get("cfl", float("nan"))))]
            initial_cfl = finite_hist_cfl[0] if finite_hist_cfl else float(result.get("max_cfl", float("nan")))
            last_finite_cfl = finite_hist_cfl[-1] if finite_hist_cfl else float(result.get("max_cfl", float("nan")))
            row = dict(family=str(family), dt=float(dt), amplitude=float(amplitude), nu=float(nu), stable=bool(stable),
                       steps_completed=int(result["steps_completed"]),
                       failure_step=int(result.get("failure_step", -1)),
                       solver_failure=solver_failure,
                       initial_cfl=float(initial_cfl), last_finite_cfl=float(last_finite_cfl),
                       max_cfl=float(result["max_cfl"]), max_diffusion_number=float(result["max_diffusion_number"]),
                       max_convection_inf=float(result.get("max_convection_inf", float("nan"))),
                       energy_initial=float(result["energy_initial"]), energy_final=float(result["energy_final"]),
                       energy_max=float(result["energy_max"]), energy_growth_max=growth,
                       final_rel_dU=float(result.get("final_rel_DU", result.get("final_rel_dU", float("nan")))),
                       final_rel_dU_ref=float(result.get("final_rel_dU_ref", float("nan"))),
                       mass_inf_per_volume=float(result["mass_inf_per_volume"]), mass_l2_per_volume=float(result["mass_l2_per_volume"]),
                       steady_momentum_inf=float(result["steady_momentum_inf"]), nan_or_inf_detected=bool(result["nan_or_inf_detected"]))
        except Exception as exc:
            row = dict(family=str(family), dt=float(dt), amplitude=float(amplitude), nu=float(nu), stable=False,
                       steps_completed=0, failure_step=-1, solver_failure=str(exc),
                       initial_cfl=float("nan"), last_finite_cfl=float("nan"),
                       max_cfl=float("nan"), max_diffusion_number=float("nan"), max_convection_inf=float("nan"),
                       energy_initial=float("nan"), energy_final=float("nan"), energy_max=float("nan"), energy_growth_max=float("nan"),
                       final_rel_dU=float("nan"), final_rel_dU_ref=float("nan"), mass_inf_per_volume=float("nan"),
                       mass_l2_per_volume=float("nan"), steady_momentum_inf=float("nan"), nan_or_inf_detected=True)
        rows.append(row)
    return rows


def run_time_stability_scan_gpu(cfg: Optional[PB615Config] = None) -> Dict[str, object]:
    """Run two explicit-convection stability scans on the GPU.

    The low-CFL family is the PB61.3 regression test.  The high-CFL stress
    family is new in PB61.8 and intentionally uses a larger manufactured
    velocity amplitude and much smaller viscosity, so that the explicit
    convection term is not hidden by implicit viscous/wall damping.
    """
    base = cfg or PB615Config()
    geom = build_demo_geometry_gpu(base)
    rows = []
    rows += _run_one_stability_family_gpu(base, geom, family="low_cfl_decay",
                                          dt_values=tuple(base.stability_dt_values),
                                          amplitude=float(base.stability_amplitude),
                                          n_steps=int(base.stability_n_steps),
                                          nu=float(base.nu),
                                          energy_growth_fail=float(base.stability_energy_growth_fail))
    rows += _run_one_stability_family_gpu(base, geom, family="high_cfl_convection_stress",
                                          dt_values=tuple(base.stress_dt_values),
                                          amplitude=float(base.stress_amplitude),
                                          n_steps=int(base.stress_n_steps),
                                          nu=float(base.stress_nu),
                                          energy_growth_fail=float(base.stress_energy_growth_fail))
    out = Path(base.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    with (out / "pb61_7_time_stability_scan.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); [w.writerow(row) for row in rows]
    print("[PB61.8-GPU] stability scan rows:")
    for row in rows: print(row)
    return {"rows": rows, "out_dir": str(out)}


def run_convection_on_off_comparison_gpu(cfg: Optional[PB615Config] = None) -> Dict[str, float]:
    """Run an all-GPU on/off comparison to prove the convection term changes the transient evolution.

    Same geometry, same initial manufactured velocity, same pressure projection,
    same implicit diffusion, same dt.  The only changed flag is enable_convection.
    This is a stronger diagnostic than a decay-only stability scan because it
    directly measures the solution difference caused by the convective operator.
    """
    base = cfg or PB615Config()
    geom = build_demo_geometry_gpu(base)
    local = PB615Config(**{**base.__dict__})
    local.dt = float(base.onoff_dt)
    local.n_steps = int(base.onoff_n_steps)
    local.report_every = max(1, int(base.onoff_n_steps) // 3)
    local.nu = float(base.stress_nu)
    local.body_force = (0.0, 0.0, 0.0)
    local.initial_velocity_mode = "manufactured_advective"
    local.initial_velocity_amplitude = float(base.stress_amplitude)
    local.stop_on_steady = False
    local.diagnostics_every_step = True
    U0 = make_initial_velocity_gpu(geom, local)
    phi0 = face_flux_from_velocity_gpu(U0, geom)
    C0 = convection_term_gpu(U0, phi0, geom)
    C0_inf = float(cp.max(cp.linalg.norm(C0, axis=1)).get())
    C0_l2 = float(cp.sqrt(cp.mean(cp.sum(C0*C0, axis=1))).get())
    cfg_off = PB615Config(**{**local.__dict__}); cfg_off.enable_convection = False
    cfg_on = PB615Config(**{**local.__dict__}); cfg_on.enable_convection = True
    print("[PB61.8-GPU] running convection OFF branch")
    off = run_velocity_pressure_projection_gpu(geom, cfg_off, initial_U=U0, run_label="convection_OFF")
    print("[PB61.8-GPU] running convection ON branch")
    on = run_velocity_pressure_projection_gpu(geom, cfg_on, initial_U=U0, run_label="convection_ON")
    dU = on["U"] - off["U"]
    diff_inf = float(cp.max(cp.linalg.norm(dU, axis=1)).get())
    diff_l2 = float(cp.sqrt(cp.mean(cp.sum(dU*dU, axis=1))).get())
    ref_l2 = float(cp.sqrt(cp.mean(cp.sum(off["U"]*off["U"], axis=1))).get())
    rel_l2 = diff_l2 / max(ref_l2, 1.0e-300)
    diag = dict(initial_convection_inf=C0_inf, initial_convection_l2=C0_l2,
                dt=float(local.dt), n_steps=int(local.n_steps), amplitude=float(local.initial_velocity_amplitude), nu=float(local.nu),
                off_energy_final=float(off["energy_final"]), on_energy_final=float(on["energy_final"]),
                off_max_cfl=float(off["max_cfl"]), on_max_cfl=float(on["max_cfl"]),
                U_difference_inf=diff_inf, U_difference_l2=diff_l2, U_difference_rel_l2=rel_l2,
                off_mass_inf=float(off["mass_inf_per_volume"]), on_mass_inf=float(on["mass_inf_per_volume"]),
                off_nan=bool(off["nan_or_inf_detected"]), on_nan=bool(on["nan_or_inf_detected"]))
    out = Path(base.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    with (out / "pb61_7_convection_on_off_comparison.csv").open("w", newline="") as f:
        w = csv.writer(f); w.writerow(diag.keys()); w.writerow(diag.values())
    print("[PB61.8-GPU] convection on/off diagnostic:", diag)
    return diag

# -----------------------------------------------------------------------------
# Demo / main entry point
# -----------------------------------------------------------------------------

def run_periodic_channel_demo_gpu(cfg: Optional[PB615Config] = None) -> Tuple[GPUFVMGeometry, Dict[str, cp.ndarray | float | int]]:
    require_cuda_gpu()
    cfg = cfg or PB615Config()
    print("[PB61.8-GPU] CUDA device:", cp.cuda.runtime.getDeviceProperties(0)["name"])
    mask = make_periodic_channel_mask_gpu(cfg.demo_shape_zyx)
    seeds = make_regular_seeds_gpu(mask, cfg.demo_seed_stride_zyx)
    print(f"[PB61.8-GPU] mask={tuple(mask.shape)}, seeds={int(seeds.size)}, connectivity={cfg.distance_connectivity}")
    t0 = time.perf_counter()
    dist, labels = geodesic_voronoi_labels_gpu(mask, seeds, cfg)
    cp.cuda.Stream.null.synchronize()
    print(f"[PB61.8-GPU] geodesic labels built on GPU in {time.perf_counter()-t0:.3f}s")
    geom = build_geometry_gpu(mask, labels, dist, seeds, cfg)
    cp.cuda.Stream.null.synchronize()
    print(f"[PB61.8-GPU] geometry: n_cells={geom.n_cells}, n_faces={int(geom.owner.size)}, wall_nonzero={int(cp.count_nonzero(geom.twall).get())}")
    result = run_velocity_pressure_projection_gpu(geom, cfg)
    write_outputs(result, geom, cfg)
    print(f"[PB61.8-GPU] wrote outputs to {cfg.out_dir}")
    return geom, result


def run_pb615_validation_suite_gpu(cfg: Optional[PB615Config] = None) -> Dict[str, object]:
    """Run the PB61.8 all-GPU validation suite.

    The suite contains: a convection-enabled steady channel smoke test, an
    operator activation diagnostic, a direct convection on/off transient
    comparison, and a two-family CFL/time-stability scan.
    """
    cfg = cfg or PB615Config()
    geom, result = run_periodic_channel_demo_gpu(cfg)
    conv_diag = run_convective_term_diagnostic_gpu(cfg)
    onoff = run_convection_on_off_comparison_gpu(cfg)
    stability = run_time_stability_scan_gpu(cfg)
    return {"geom": geom, "channel_result": result, "convective_diagnostic": conv_diag, "convection_on_off": onoff, "stability": stability}


def main() -> None:
    cfg = PB615Config()
    run_pb615_validation_suite_gpu(cfg)


# Notebook self-contained mode: main() is not called automatically.



PB615_DEFINITIONS_LOADED = True
print("PB61.8g definitions loaded. All numerical kernels/solvers remain GPU-only.")



# =============================================================================
# PB61.8g — full validation suite add-on
# =============================================================================
# These routines extend PB61.8g.  They keep the numerical work on GPU and only
# transfer scalar diagnostics / final CSV outputs to host.
# Important: the explicit upwind convection scheme is not mathematically stable
# for all possible CFL numbers.  PB61.8 therefore provides a documented CFL
# stability envelope and an explicit audit that refuses the unconditional
# statement "stable for all CFL" unless a different unconditionally stable time
# discretization is implemented and proven.

PB617_OUT_DIR = "pb61_7_allgpu_full_validation_outputs"


def _pb616_clean_cfg(base: Optional[PB615Config] = None, **updates) -> PB615Config:
    """Create a PB615Config while filtering out non-config attributes."""
    src = base or PB615Config()
    allowed = set(PB615Config.__dataclass_fields__.keys())
    data = {k: getattr(src, k) for k in allowed if hasattr(src, k)}
    data.update({k: v for k, v in updates.items() if k in allowed})
    return PB615Config(**data)


def _pb616_out(cfg: Optional[PB615Config] = None) -> Path:
    out = Path((cfg.out_dir if cfg is not None else PB617_OUT_DIR) or PB617_OUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    return out


def build_geometry_from_mask_stride_gpu(mask: cp.ndarray, stride_zyx: Tuple[int, int, int], cfg: PB615Config) -> GPUFVMGeometry:
    seeds = make_regular_seeds_gpu(mask, stride_zyx)
    t0 = time.perf_counter()
    dist, labels = geodesic_voronoi_labels_gpu(mask, seeds, cfg)
    cp.cuda.Stream.null.synchronize()
    print(f"[PB61.8-GPU] labels built: shape={tuple(mask.shape)}, stride={stride_zyx}, seeds={int(seeds.size)}, elapsed={time.perf_counter()-t0:.3f}s")
    geom = build_geometry_gpu(mask, labels, dist, seeds, cfg)
    cp.cuda.Stream.null.synchronize()
    print(f"[PB61.8-GPU] geometry: n_cells={geom.n_cells}, n_faces={int(geom.owner.size)}, wall_nonzero={int(cp.count_nonzero(geom.twall).get())}")
    return geom


def make_baffled_periodic_channel_mask_gpu(shape_zyx: Tuple[int, int, int]) -> cp.ndarray:
    """A small tortuous periodic-x channel with two offset baffles.

    This is used only for nonlinear/convection stress tests.  It creates a
    geometry where the advective term is not trivially zero, unlike a perfectly
    developed straight channel.
    """
    D, H, W = [int(v) for v in shape_zyx]
    mask = cp.ones((D, H, W), dtype=cp.bool_)
    if W < 12 or H < 8 or D < 8:
        return mask
    x1 = max(2, W // 3)
    x2 = min(W - 3, (2 * W) // 3)
    # Baffle 1 with lower-left opening.
    mask[:, :, x1] = False
    z0, z1 = max(1, D // 8), max(2, D // 8 + D // 3)
    y0, y1 = max(1, H // 8), max(2, H // 8 + H // 3)
    mask[z0:z1, y0:y1, x1] = True
    # Baffle 2 with upper-right opening.
    mask[:, :, x2] = False
    z0b, z1b = max(1, D - D // 8 - D // 3), max(2, D - D // 8)
    y0b, y1b = max(1, H - H // 8 - H // 3), max(2, H - H // 8)
    mask[z0b:z1b, y0b:y1b, x2] = True
    return mask


def rectangular_duct_analytic_ux_gpu(centroid: cp.ndarray, shape_zyx: Tuple[int, int, int], cfg: PB615Config, terms: int = 31) -> cp.ndarray:
    """Analytical fully developed rectangular-duct Stokes velocity under body force.

    Solves -nu*(u_yy+u_zz)=f_x with u=0 on y/z walls and periodic x.
    Coordinates are the cell centroids in x,y,z order.  The formula is a sine
    series over odd modes.  It is used as a benchmark, not as part of the solver.
    """
    D, H, W = [int(v) for v in shape_zyx]
    h = float(cfg.voxel_size)
    Ly = H * h
    Lz = D * h
    y = centroid[:, 1]
    z = centroid[:, 2]
    fx = float(cfg.body_force[0])
    nu = float(cfg.nu)
    out = cp.zeros(centroid.shape[0], dtype=cp.float64)
    max_mode = max(1, int(terms))
    for m in range(1, max_mode + 1, 2):
        sm = cp.sin(math.pi * m * y / Ly)
        for n in range(1, max_mode + 1, 2):
            denom = (m * n) * ((m / Ly) ** 2 + (n / Lz) ** 2)
            out += sm * cp.sin(math.pi * n * z / Lz) / denom
    out *= (16.0 * fx) / (nu * math.pi ** 4)
    return out


def _gpu_weighted_rel_l2_vec(A: cp.ndarray, B: cp.ndarray, weights: cp.ndarray) -> float:
    num = cp.sum(weights * cp.sum((A - B) * (A - B), axis=1))
    den = cp.sum(weights * cp.sum(B * B, axis=1))
    return float(cp.sqrt(num / cp.maximum(den, 1.0e-300)).get())


def _gpu_weighted_rel_l2_scalar(a: cp.ndarray, b: cp.ndarray, weights: cp.ndarray) -> float:
    num = cp.sum(weights * (a - b) ** 2)
    den = cp.sum(weights * b ** 2)
    return float(cp.sqrt(num / cp.maximum(den, 1.0e-300)).get())


def _gpu_rel_inf_scalar(a: cp.ndarray, b: cp.ndarray) -> float:
    return float((cp.max(cp.abs(a - b)) / cp.maximum(cp.max(cp.abs(b)), 1.0e-300)).get())


def run_analytical_poiseuille_duct_comparison_gpu(cfg: Optional[PB615Config] = None,
                                                  shape_zyx: Tuple[int, int, int] = (16, 16, 32),
                                                  stride_zyx: Tuple[int, int, int] = (4, 4, 4),
                                                  terms: int = 41) -> Dict[str, float]:
    """Validation 1: compare computed Ux against the rectangular duct series."""
    cfg = _pb616_clean_cfg(cfg, out_dir=PB617_OUT_DIR, periodic_x=True,
                           enable_convection=True, body_force=(2.0e-3, 0.0, 0.0),
                           initial_velocity_mode="zero", initial_velocity_amplitude=0.0,
                           nu=1.0, dt=2.0, n_steps=600, stop_on_steady=True,
                           raise_on_solver_failure=False)
    mask = make_periodic_channel_mask_gpu(shape_zyx)
    geom = build_geometry_from_mask_stride_gpu(mask, stride_zyx, cfg)
    result = run_velocity_pressure_projection_gpu(geom, cfg, run_label="analytic_duct")
    ux_ref = rectangular_duct_analytic_ux_gpu(geom.centroid, shape_zyx, cfg, terms=terms)
    U_ref = cp.zeros_like(result["U"]); U_ref[:, 0] = ux_ref
    mean_ref = cp.sum(ux_ref * geom.volume) / cp.sum(geom.volume)
    k_ref = float((cfg.nu * mean_ref / float(cfg.body_force[0])).get())
    row = {
        "validation": "analytic_rectangular_duct_poiseuille",
        "shape": str(tuple(shape_zyx)), "stride": str(tuple(stride_zyx)), "terms": int(terms),
        "n_cells": int(geom.n_cells), "n_faces": int(geom.owner.size),
        "steady_converged": bool(result.get("steady_converged", False)),
        "steps_completed": int(result.get("steps_completed", -1)),
        "mass_inf_per_volume": float(result["mass_inf_per_volume"]),
        "steady_momentum_inf": float(result["steady_momentum_inf"]),
        "U_rel_l2": _gpu_weighted_rel_l2_vec(result["U"], U_ref, geom.volume),
        "Ux_rel_inf": _gpu_rel_inf_scalar(result["U"][:, 0], ux_ref),
        "K_eff_x": float(result["K_eff_x"]), "K_ref_x": k_ref,
        "K_rel_err": abs(float(result["K_eff_x"]) - k_ref) / max(abs(k_ref), 1.0e-300),
        "max_cfl": float(result.get("max_cfl", float("nan"))),
        "note": "Analytical comparison is for steady Stokes/fully developed rectangular duct; it does not validate high-Re transient NS."
    }
    out = _pb616_out(cfg)
    with (out / "pb61_7_analytic_poiseuille_duct.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys())); w.writeheader(); w.writerow(row)
    print("[PB61.8-GPU] analytic duct comparison:", row)
    return row


def coarsen_voxel_reference_to_coarse_gpu(ref_geom: GPUFVMGeometry, ref_result: Dict[str, object], coarse_geom: GPUFVMGeometry) -> Tuple[cp.ndarray, cp.ndarray]:
    """Volume-average voxel-grid reference U,p over the coarse Voronoi labels."""
    labels_c = coarse_geom.labels.ravel()
    labels_r = ref_geom.labels.ravel()
    valid = (labels_c >= 0) & (labels_r >= 0) & coarse_geom.mask.ravel()
    labc = labels_c[valid].astype(cp.int32)
    labr = labels_r[valid].astype(cp.int32)
    Uvox = ref_result["U"][labr]
    pvox = ref_result["p"][labr]
    n = coarse_geom.n_cells
    cnt = cp.bincount(labc, minlength=n).astype(cp.float64)
    Uavg_cols = []
    for c in range(3):
        Uavg_cols.append(cp.bincount(labc, weights=Uvox[:, c], minlength=n).astype(cp.float64) / cp.maximum(cnt, 1.0e-300))
    Uavg = cp.stack(Uavg_cols, axis=1)
    pavg = cp.bincount(labc, weights=pvox, minlength=n).astype(cp.float64) / cp.maximum(cnt, 1.0e-300)
    # Gauge-align pressure to mean zero under coarse volumes.
    pavg = pavg - cp.sum(pavg * coarse_geom.volume) / cp.maximum(cp.sum(coarse_geom.volume), 1.0e-300)
    return Uavg, pavg


def run_voxel_grid_fvm_reference_comparison_gpu(cfg: Optional[PB615Config] = None,
                                                shape_zyx: Tuple[int, int, int] = (12, 12, 24),
                                                coarse_stride_zyx: Tuple[int, int, int] = (3, 3, 3)) -> Dict[str, float]:
    """Validation 2: compare coarse solution with a voxel-grid FVM reference.

    The reference is the same GPU solver at one fluid voxel per control volume,
    i.e. a voxel-grid FVM reference rather than a separate external DNS code.
    """
    cfg0 = _pb616_clean_cfg(cfg, out_dir=PB617_OUT_DIR, periodic_x=True,
                            body_force=(2.0e-3, 0.0, 0.0), nu=1.0,
                            enable_convection=True, n_steps=600, dt=2.0,
                            stop_on_steady=True, raise_on_solver_failure=False)
    mask = make_periodic_channel_mask_gpu(shape_zyx)
    ref_geom = build_geometry_from_mask_stride_gpu(mask, (1, 1, 1), cfg0)
    ref = run_velocity_pressure_projection_gpu(ref_geom, cfg0, run_label="voxel_reference")
    coarse_geom = build_geometry_from_mask_stride_gpu(mask, coarse_stride_zyx, cfg0)
    coarse = run_velocity_pressure_projection_gpu(coarse_geom, cfg0, run_label="coarse_vs_voxel_ref")
    Uref_c, pref_c = coarsen_voxel_reference_to_coarse_gpu(ref_geom, ref, coarse_geom)
    pcoarse = coarse["p"] - cp.sum(coarse["p"] * coarse_geom.volume) / cp.maximum(cp.sum(coarse_geom.volume), 1.0e-300)
    row = {
        "validation": "voxel_grid_fvm_reference_comparison",
        "shape": str(tuple(shape_zyx)), "coarse_stride": str(tuple(coarse_stride_zyx)),
        "ref_cells": int(ref_geom.n_cells), "coarse_cells": int(coarse_geom.n_cells),
        "ref_steady": bool(ref.get("steady_converged", False)), "coarse_steady": bool(coarse.get("steady_converged", False)),
        "U_rel_l2_vs_voxel_ref": _gpu_weighted_rel_l2_vec(coarse["U"], Uref_c, coarse_geom.volume),
        "p_rel_l2_vs_voxel_ref": _gpu_weighted_rel_l2_scalar(pcoarse, pref_c, coarse_geom.volume),
        "K_ref_x": float(ref["K_eff_x"]), "K_coarse_x": float(coarse["K_eff_x"]),
        "K_rel_err_vs_voxel_ref": abs(float(coarse["K_eff_x"]) - float(ref["K_eff_x"])) / max(abs(float(ref["K_eff_x"])), 1.0e-300),
        "mass_inf_per_volume": float(coarse["mass_inf_per_volume"]),
        "steady_momentum_inf": float(coarse["steady_momentum_inf"]),
        "note": "Reference is voxel-grid FVM generated by one seed per traversable voxel on the same mask."
    }
    out = _pb616_out(cfg0)
    with (out / "pb61_7_voxel_grid_reference_comparison.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys())); w.writeheader(); w.writerow(row)
    print("[PB61.8-GPU] voxel-grid comparison:", row)
    return row


def _fit_loglog_slope(rows: Sequence[Dict[str, object]], h_key: str, e_key: str) -> float:
    vals = []
    for r in rows:
        h = float(r[h_key]); e = float(r[e_key])
        if h > 0.0 and e > 0.0 and math.isfinite(h) and math.isfinite(e):
            vals.append((math.log(h), math.log(e)))
    if len(vals) < 2:
        return float("nan")
    xm = sum(x for x, _ in vals) / len(vals)
    ym = sum(y for _, y in vals) / len(vals)
    den = sum((x - xm) ** 2 for x, _ in vals)
    if den <= 0:
        return float("nan")
    return sum((x - xm) * (y - ym) for x, y in vals) / den


def run_seed_refinement_convergence_gpu(cfg: Optional[PB615Config] = None,
                                        shape_zyx: Tuple[int, int, int] = (12, 12, 24),
                                        strides: Tuple[Tuple[int, int, int], ...] = ((6, 6, 6), (4, 4, 4), (3, 3, 3), (2, 2, 2))) -> Dict[str, object]:
    """Validation 3 and 7: seed refinement convergence and empirical order."""
    cfg0 = _pb616_clean_cfg(cfg, out_dir=PB617_OUT_DIR, periodic_x=True,
                            body_force=(2.0e-3, 0.0, 0.0), nu=1.0,
                            enable_convection=True, n_steps=600, dt=2.0,
                            stop_on_steady=True, raise_on_solver_failure=False)
    mask = make_periodic_channel_mask_gpu(shape_zyx)
    ref_geom = build_geometry_from_mask_stride_gpu(mask, (1, 1, 1), cfg0)
    ref = run_velocity_pressure_projection_gpu(ref_geom, cfg0, run_label="seed_convergence_voxel_ref")
    rows = []
    for stride in strides:
        geom = build_geometry_from_mask_stride_gpu(mask, stride, cfg0)
        res = run_velocity_pressure_projection_gpu(geom, cfg0, run_label=f"seed_convergence_stride={stride}")
        Uref_c, pref_c = coarsen_voxel_reference_to_coarse_gpu(ref_geom, ref, geom)
        pcoarse = res["p"] - cp.sum(res["p"] * geom.volume) / cp.maximum(cp.sum(geom.volume), 1.0e-300)
        h_eff = float((cp.mean(geom.volume) ** (1.0 / 3.0)).get())
        rows.append({
            "validation": "seed_refinement_convergence",
            "shape": str(tuple(shape_zyx)), "stride": str(tuple(stride)),
            "h_eff": h_eff, "n_cells": int(geom.n_cells), "n_faces": int(geom.owner.size),
            "U_rel_l2": _gpu_weighted_rel_l2_vec(res["U"], Uref_c, geom.volume),
            "p_rel_l2": _gpu_weighted_rel_l2_scalar(pcoarse, pref_c, geom.volume),
            "K_rel_err": abs(float(res["K_eff_x"]) - float(ref["K_eff_x"])) / max(abs(float(ref["K_eff_x"])), 1.0e-300),
            "K_eff_x": float(res["K_eff_x"]), "K_ref_x": float(ref["K_eff_x"]),
            "steady_converged": bool(res.get("steady_converged", False)),
            "mass_inf_per_volume": float(res["mass_inf_per_volume"]),
            "steady_momentum_inf": float(res["steady_momentum_inf"])
        })
    order = {
        "order_U_l2": _fit_loglog_slope(rows, "h_eff", "U_rel_l2"),
        "order_p_l2": _fit_loglog_slope(rows, "h_eff", "p_rel_l2"),
        "order_K": _fit_loglog_slope(rows, "h_eff", "K_rel_err"),
        "note": "Orders are empirical slopes against the one-voxel-per-cell GPU FVM reference."
    }
    out = _pb616_out(cfg0)
    with (out / "pb61_7_seed_refinement_convergence.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); [w.writerow(r) for r in rows]
    with (out / "pb61_7_order_of_accuracy_summary.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(order.keys())); w.writeheader(); w.writerow(order)
    print("[PB61.8-GPU] seed refinement rows:")
    for r in rows: print(r)
    print("[PB61.8-GPU] empirical orders:", order)
    return {"rows": rows, "order": order}


def run_nonlinear_re_sweep_gpu(cfg: Optional[PB615Config] = None,
                               shape_zyx: Tuple[int, int, int] = (16, 16, 32),
                               stride_zyx: Tuple[int, int, int] = (4, 4, 4),
                               body_forces: Tuple[float, ...] = (1e-4, 5e-4, 1e-3, 2e-3, 5e-3, 1e-2),
                               nu: float = 1.0e-3) -> Dict[str, object]:
    """Validation 4: nonlinear/Reynolds sweep on a baffled periodic channel."""
    base = _pb616_clean_cfg(cfg, out_dir=PB617_OUT_DIR, periodic_x=True,
                            enable_convection=True, initial_velocity_mode="zero",
                            nu=nu, dt=0.5, n_steps=500, stop_on_steady=True,
                            steady_tol=1.0e-6, raise_on_solver_failure=False,
                            report_every=100)
    mask = make_baffled_periodic_channel_mask_gpu(shape_zyx)
    geom = build_geometry_from_mask_stride_gpu(mask, stride_zyx, base)
    h = float(base.voxel_size)
    D, H, W = [int(v) for v in shape_zyx]
    # A crude hydraulic scale for reporting only.
    Lh = 2.0 * (D * h) * (H * h) / max((D * h) + (H * h), 1.0e-300)
    rows = []
    for fx in body_forces:
        local = _pb616_clean_cfg(base, body_force=(float(fx), 0.0, 0.0), nu=float(nu))
        print(f"[PB61.8-GPU] Re sweep fx={fx:g}, nu={nu:g}")
        res = run_velocity_pressure_projection_gpu(geom, local, run_label=f"re_sweep_fx={fx:g}")
        umax = float(cp.max(cp.linalg.norm(res["U"], axis=1)).get())
        Re = umax * Lh / max(float(nu), 1.0e-300)
        rows.append({
            "validation": "nonlinear_re_sweep_baffled_channel",
            "body_force_x": float(fx), "nu": float(nu), "Re_est_umax_Lh_over_nu": Re,
            "steady_converged": bool(res.get("steady_converged", False)),
            "solver_failure": str(res.get("solver_failure", "")),
            "steps_completed": int(res.get("steps_completed", -1)),
            "umax": umax, "mean_U_x": float(res["mean_U_x"]), "K_eff_x": float(res["K_eff_x"]),
            "max_cfl": float(res.get("max_cfl", float("nan"))),
            "max_convection_inf": float(res.get("max_convection_inf", float("nan"))),
            "mass_inf_per_volume": float(res["mass_inf_per_volume"]),
            "steady_momentum_inf": float(res["steady_momentum_inf"]),
            "nan_or_inf_detected": bool(res.get("nan_or_inf_detected", False))
        })
    out = _pb616_out(base)
    with (out / "pb61_7_nonlinear_re_sweep.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); [w.writerow(r) for r in rows]
    print("[PB61.8-GPU] Re sweep rows:")
    for r in rows: print(r)
    return {"rows": rows}


def run_standard_transient_ns_benchmarks_gpu(cfg: Optional[PB615Config] = None,
                                             shape_zyx: Tuple[int, int, int] = (16, 16, 32),
                                             stride_zyx: Tuple[int, int, int] = (4, 4, 4)) -> Dict[str, object]:
    """Validation 5: transient N-S regression cases.

    Current cases are (i) manufactured vortex-like advective decay and (ii)
    impulsively forced channel approach to steady state.  These are transient
    solver benchmarks; they are not yet a substitute for canonical external DNS
    benchmarks such as true Taylor-Green vortex with fully periodic boundaries.
    """
    rows = []
    base = _pb616_clean_cfg(cfg, out_dir=PB617_OUT_DIR, periodic_x=True,
                            enable_convection=True, raise_on_solver_failure=False,
                            diagnostics_every_step=True, stop_on_steady=False)
    mask = make_periodic_channel_mask_gpu(shape_zyx)
    geom = build_geometry_from_mask_stride_gpu(mask, stride_zyx, base)
    # Case A: advective manufactured transient decay.
    local = _pb616_clean_cfg(base, body_force=(0.0, 0.0, 0.0), nu=1.0e-2, dt=0.5, n_steps=80,
                             initial_velocity_mode="manufactured_advective", initial_velocity_amplitude=1.0,
                             report_every=20)
    res = run_velocity_pressure_projection_gpu(geom, local, run_label="transient_manufactured_advective_decay")
    rows.append({
        "validation": "standard_transient_ns_regression",
        "case": "manufactured_advective_decay",
        "n_cells": int(geom.n_cells), "dt": float(local.dt), "nu": float(local.nu),
        "energy_initial": float(res["energy_initial"]), "energy_final": float(res["energy_final"]),
        "energy_growth": float(res["energy_max"]) / max(float(res["energy_initial"]), 1.0e-300),
        "max_cfl": float(res["max_cfl"]), "max_convection_inf": float(res["max_convection_inf"]),
        "mass_inf_per_volume": float(res["mass_inf_per_volume"]),
        "nan_or_inf_detected": bool(res["nan_or_inf_detected"]),
        "solver_failure": str(res.get("solver_failure", ""))
    })
    # Case B: impulsively forced channel.
    local = _pb616_clean_cfg(base, body_force=(2.0e-3, 0.0, 0.0), nu=1.0, dt=1.0, n_steps=180,
                             initial_velocity_mode="zero", initial_velocity_amplitude=0.0,
                             report_every=30, stop_on_steady=True)
    res = run_velocity_pressure_projection_gpu(geom, local, run_label="transient_impulsively_forced_channel")
    rows.append({
        "validation": "standard_transient_ns_regression",
        "case": "impulsively_forced_channel_to_steady",
        "n_cells": int(geom.n_cells), "dt": float(local.dt), "nu": float(local.nu),
        "energy_initial": float(res["energy_initial"]), "energy_final": float(res["energy_final"]),
        "energy_growth": float(res["energy_max"]) / max(float(res["energy_initial"]), 1.0e-300),
        "max_cfl": float(res["max_cfl"]), "max_convection_inf": float(res["max_convection_inf"]),
        "mass_inf_per_volume": float(res["mass_inf_per_volume"]),
        "steady_converged": bool(res.get("steady_converged", False)),
        "steady_momentum_inf": float(res.get("steady_momentum_inf", float("nan"))),
        "nan_or_inf_detected": bool(res["nan_or_inf_detected"]),
        "solver_failure": str(res.get("solver_failure", ""))
    })
    out = _pb616_out(base)
    keys = sorted(set(k for r in rows for k in r.keys()))
    with (out / "pb61_7_standard_transient_ns_benchmarks.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); [w.writerow(r) for r in rows]
    print("[PB61.8-GPU] transient benchmark rows:")
    for r in rows: print(r)
    return {"rows": rows}


def build_wall_area_gpu(mask: cp.ndarray, labels: cp.ndarray, n_cells: int, cfg: PB615Config) -> cp.ndarray:
    """Aggregate wall facelet area per coarse cell on GPU."""
    h = float(cfg.voxel_size); area0 = h * h
    D, H, W = [int(v) for v in labels.shape]
    labs = []
    def add(valid, lab_arr):
        if int(valid.sum().get()) > 0:
            labs.append(lab_arr[cp.nonzero(valid)].astype(cp.int32))
    # x exterior walls only if not periodic_x.
    if not cfg.periodic_x:
        add(mask[:, :, 0] & (labels[:, :, 0] >= 0), labels[:, :, 0])
        add(mask[:, :, -1] & (labels[:, :, -1] >= 0), labels[:, :, -1])
    # x interior solid/fluid faces.
    li = labels[:, :, :-1]; lj = labels[:, :, 1:]; mi = mask[:, :, :-1]; mj = mask[:, :, 1:]
    add(mi & (~mj) & (li >= 0), li); add((~mi) & mj & (lj >= 0), lj)
    # y exterior and interior.
    add(mask[:, 0, :] & (labels[:, 0, :] >= 0), labels[:, 0, :])
    add(mask[:, -1, :] & (labels[:, -1, :] >= 0), labels[:, -1, :])
    li = labels[:, :-1, :]; lj = labels[:, 1:, :]; mi = mask[:, :-1, :]; mj = mask[:, 1:, :]
    add(mi & (~mj) & (li >= 0), li); add((~mi) & mj & (lj >= 0), lj)
    # z exterior and interior.
    add(mask[0, :, :] & (labels[0, :, :] >= 0), labels[0, :, :])
    add(mask[-1, :, :] & (labels[-1, :, :] >= 0), labels[-1, :, :])
    li = labels[:-1, :, :]; lj = labels[1:, :, :]; mi = mask[:-1, :, :]; mj = mask[1:, :, :]
    add(mi & (~mj) & (li >= 0), li); add((~mi) & mj & (lj >= 0), lj)
    if not labs:
        return cp.zeros(n_cells, dtype=cp.float64)
    lab = cp.concatenate(labs)
    return cp.bincount(lab, weights=cp.full(lab.shape, area0, dtype=cp.float64), minlength=n_cells).astype(cp.float64)


def run_wall_shear_validation_gpu(cfg: Optional[PB615Config] = None,
                                  shape_zyx: Tuple[int, int, int] = (16, 16, 32),
                                  stride_zyx: Tuple[int, int, int] = (4, 4, 4)) -> Dict[str, float]:
    """Validation 6: wall shear/drag balance audit.

    The current coarse wall closure gives an effective wall shear/drag, not a
    pointwise DNS wall shear map.  This validates global wall-drag balance and
    writes a coarse cell-wise effective shear estimate.
    """
    cfg0 = _pb616_clean_cfg(cfg, out_dir=PB617_OUT_DIR, periodic_x=True,
                            body_force=(2.0e-3, 0.0, 0.0), nu=1.0,
                            enable_convection=True, dt=2.0, n_steps=600,
                            stop_on_steady=True, raise_on_solver_failure=False)
    mask = make_periodic_channel_mask_gpu(shape_zyx)
    geom = build_geometry_from_mask_stride_gpu(mask, stride_zyx, cfg0)
    res = run_velocity_pressure_projection_gpu(geom, cfg0, run_label="wall_shear_validation")
    wall_area = build_wall_area_gpu(mask, geom.labels, geom.n_cells, cfg0)
    speed = cp.linalg.norm(res["U"], axis=1)
    shear_eff = float(cfg0.nu) * geom.twall * speed / cp.maximum(wall_area, 1.0e-300)
    wall_drag_x = float((float(cfg0.nu) * cp.sum(geom.twall * res["U"][:, 0])).get())
    body_drive_x = float(float(cfg0.body_force[0]) * cp.sum(geom.volume).get())
    rel_drag_balance = abs(wall_drag_x - body_drive_x) / max(abs(body_drive_x), 1.0e-300)
    row = {
        "validation": "wall_shear_drag_balance",
        "shape": str(tuple(shape_zyx)), "stride": str(tuple(stride_zyx)),
        "n_cells": int(geom.n_cells), "wall_cells": int(cp.count_nonzero(wall_area > 0).get()),
        "wall_drag_x": wall_drag_x, "body_drive_x": body_drive_x,
        "global_drag_balance_rel_err": rel_drag_balance,
        "mean_effective_wall_shear": float(cp.mean(shear_eff[wall_area > 0]).get()) if int(cp.count_nonzero(wall_area > 0).get()) else float("nan"),
        "max_effective_wall_shear": float(cp.max(shear_eff).get()) if int(shear_eff.size) else float("nan"),
        "steady_converged": bool(res.get("steady_converged", False)),
        "mass_inf_per_volume": float(res["mass_inf_per_volume"]),
        "steady_momentum_inf": float(res["steady_momentum_inf"]),
        "note": "This is an effective coarse wall-shear/drag validation; pointwise DNS wall shear requires a voxel/grid velocity-gradient reference."
    }
    out = _pb616_out(cfg0)
    with (out / "pb61_7_wall_shear_drag_balance.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys())); w.writeheader(); w.writerow(row)
    # Write per-cell effective shear.
    shear_h = cp.asnumpy(shear_eff); area_h = cp.asnumpy(wall_area); tw_h = cp.asnumpy(geom.twall)
    with (out / "pb61_7_wall_shear_cell_map.csv").open("w", newline="") as f:
        w = csv.writer(f); w.writerow(["cell", "wall_area", "twall", "effective_wall_shear"])
        for i in range(len(shear_h)):
            w.writerow([i, area_h[i], tw_h[i], shear_h[i]])
    print("[PB61.8-GPU] wall shear validation:", row)
    return row


def run_cfl_stability_envelope_and_all_cfl_claim_audit_gpu(cfg: Optional[PB615Config] = None,
                                                           dt_values: Tuple[float, ...] = (0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0)) -> Dict[str, object]:
    """Validation 8: CFL stability envelope plus explicit rejection of universal all-CFL claim.

    For the current explicit upwind convection, a universal "stable for all CFL"
    statement is not a valid numerical claim.  This function records the tested
    stability envelope and writes an audit row stating that universal all-CFL
    stability is not proven and should not be claimed for this explicit scheme.
    """
    cfg0 = _pb616_clean_cfg(cfg, out_dir=PB617_OUT_DIR, periodic_x=True,
                            enable_convection=True, stress_dt_values=tuple(dt_values),
                            stress_amplitude=1.0, stress_nu=1.0e-3,
                            stress_n_steps=80, raise_on_solver_failure=False)
    stability = run_time_stability_scan_gpu(cfg0)
    rows = stability["rows"]
    stress_rows = [r for r in rows if str(r.get("family", "")).startswith("high_cfl") or str(r.get("family", "")) == "high_cfl_convection_stress"]
    if not stress_rows:
        stress_rows = rows
    stable_cfl = [float(r.get("max_cfl", float("nan"))) for r in stress_rows if bool(r.get("stable", False)) and math.isfinite(float(r.get("max_cfl", float("nan"))))]
    failed = [r for r in stress_rows if not bool(r.get("stable", False))]
    max_stable = max(stable_cfl) if stable_cfl else float("nan")
    first_failed_initial_cfl = float(failed[0].get("initial_cfl", float("nan"))) if failed else float("nan")
    first_failed_last_finite_cfl = float(failed[0].get("last_finite_cfl", float("nan"))) if failed else float("nan")
    audit = {
        "claim": "The method is stable for all CFL",
        "universal_claim_supported": False,
        "tested_max_stable_cfl": max_stable,
        "first_failed_initial_cfl": first_failed_initial_cfl,
        "first_failed_last_finite_cfl": first_failed_last_finite_cfl,
        "first_failed_or_unvalidated_cfl": first_failed_initial_cfl,
        "failed_cases_observed": bool(len(failed) > 0),
        "recommended_replacement_claim": "The method is stable over the tested CFL envelope; higher-CFL points are recorded as solver/stability failures when observed.",
        "reason": "The current method uses explicit convection. Explicit convection schemes do not provide unconditional all-CFL stability without an additional implicit/nonlinear solve or proof."
    }
    out = _pb616_out(cfg0)
    with (out / "pb61_7_all_cfl_claim_audit.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(audit.keys())); w.writeheader(); w.writerow(audit)
    print("[PB61.8-GPU] CFL/all-CFL claim audit:", audit)
    return {"stability": stability, "audit": audit}


def run_pb616_full_validation_suite_gpu(cfg: Optional[PB615Config] = None) -> Dict[str, object]:
    """Run all PB61.8 validation blocks.

    This is intentionally long.  For development, run individual blocks first.
    """
    base = _pb616_clean_cfg(cfg, out_dir=PB617_OUT_DIR)
    results = {}
    results["analytic_duct"] = run_analytical_poiseuille_duct_comparison_gpu(base)
    results["voxel_grid_reference"] = run_voxel_grid_fvm_reference_comparison_gpu(base)
    results["seed_convergence_order"] = run_seed_refinement_convergence_gpu(base)
    results["nonlinear_re_sweep"] = run_nonlinear_re_sweep_gpu(base)
    results["transient_ns_benchmarks"] = run_standard_transient_ns_benchmarks_gpu(base)
    results["wall_shear"] = run_wall_shear_validation_gpu(base)
    results["cfl_stability_envelope"] = run_cfl_stability_envelope_and_all_cfl_claim_audit_gpu(base)
    out = _pb616_out(base)
    manifest = {"suite": "PB61.8 all-GPU full validation suite", "out_dir": str(out), "blocks": list(results.keys()),
                "interpretation": "All-CFL stability is audited, not asserted universally. Use the stable CFL envelope and failure rows in the manuscript."}
    (out / "pb61_7_validation_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("[PB61.8-GPU] full validation suite complete. Outputs:", out)
    return results


PB616_DEFINITIONS_LOADED = True
print("PB61.8g validation-suite definitions loaded. Use run_pb616_full_validation_suite_gpu() or individual validation blocks.")



def main() -> None:
    cfg = _pb616_clean_cfg(out_dir=PB617_OUT_DIR)
    run_pb616_full_validation_suite_gpu(cfg)



# ---- PB61.8 convenience aliases and one-click full validation runner ----
PB617Config = PB615Config

def _pb617_make_zip(out_dir: str = PB617_OUT_DIR) -> str:
    import shutil
    out = Path(out_dir)
    if not out.exists():
        raise RuntimeError(f"Output directory does not exist: {out}")
    zip_path = shutil.make_archive(str(out), "zip", out)
    print(f"[PB61.8-GPU] zipped outputs: {zip_path}")
    return zip_path


def run_pb617_full_validation_suite_gpu(cfg: Optional[PB615Config] = None, *, zip_outputs: bool = True) -> Dict[str, object]:
    """Run every PB61.8 validation block in sequence and optionally zip the output directory.

    This is the 'all switches on' runner: analytic duct, voxel-grid reference, seed/order
    convergence, nonlinear/Re sweep, transient N-S regressions, wall-drag validation, and
    CFL envelope/all-CFL audit.  It is GPU-only except final CSV/JSON/ZIP output writing.
    """
    base = _pb616_clean_cfg(cfg, out_dir=PB617_OUT_DIR)
    results = run_pb617_full_validation_suite_gpu_nozip(base)
    if zip_outputs:
        results["outputs_zip"] = _pb617_make_zip(base.out_dir)
    return results


def run_pb617_full_validation_suite_gpu_nozip(cfg: Optional[PB615Config] = None) -> Dict[str, object]:
    base = _pb616_clean_cfg(cfg, out_dir=PB617_OUT_DIR)
    results = {}
    print("[PB61.8-GPU] FULL SUITE 1/7 analytic duct / Poiseuille")
    results["analytic_duct"] = run_analytical_poiseuille_duct_comparison_gpu(base)
    print("[PB61.8-GPU] FULL SUITE 2/7 voxel-grid reference comparison")
    results["voxel_grid_reference"] = run_voxel_grid_fvm_reference_comparison_gpu(base)
    print("[PB61.8-GPU] FULL SUITE 3/7 seed refinement + order of accuracy")
    results["seed_convergence_order"] = run_seed_refinement_convergence_gpu(base)
    print("[PB61.8-GPU] FULL SUITE 4/7 nonlinear/Re sweep")
    results["nonlinear_re_sweep"] = run_nonlinear_re_sweep_gpu(base)
    print("[PB61.8-GPU] FULL SUITE 5/7 transient N-S regression benchmarks")
    results["transient_ns_benchmarks"] = run_standard_transient_ns_benchmarks_gpu(base)
    print("[PB61.8-GPU] FULL SUITE 6/7 wall shear / wall-drag validation")
    results["wall_shear"] = run_wall_shear_validation_gpu(base)
    print("[PB61.8-GPU] FULL SUITE 7/7 CFL stability envelope + all-CFL audit")
    results["cfl_stability_envelope"] = run_cfl_stability_envelope_and_all_cfl_claim_audit_gpu(base)
    out = _pb616_out(base)
    manifest = {"suite": "PB61.8 all-GPU full validation suite", "out_dir": str(out), "blocks": list(results.keys()),
                "interpretation": "Velocity/pressure accuracy is given by analytic_duct, voxel_grid_reference and seed_convergence outputs. All-CFL stability is audited, not asserted universally."}
    (out / "pb61_7_validation_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("[PB61.8-GPU] full validation suite complete. Outputs:", out)
    return results

PB617_DEFINITIONS_LOADED = True
print("PB61.8 all-GPU full validation definitions loaded. Run `full = run_pb617_full_validation_suite_gpu(PB617Config())` to execute everything.")



# =============================================================================
# PB61.8g — fixed-seed operator/closure ablation for lowering velocity-field loss
# =============================================================================
PB618_OUT_DIR = "pb61_8_fixed_seed_operator_ablation_outputs"
PB618Config = PB615Config


def _pb618_cfg(cfg: Optional[PB615Config] = None, **updates) -> PB615Config:
    base = _pb616_clean_cfg(cfg, out_dir=PB618_OUT_DIR)
    data = {k: getattr(base, k) for k in PB615Config.__dataclass_fields__.keys()}
    data.update(updates)
    return PB615Config(**data)


def _pb618_out(cfg: Optional[PB615Config] = None) -> Path:
    out = Path((cfg.out_dir if cfg is not None else PB618_OUT_DIR) or PB618_OUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    return out


def _pb618_face_clip_for_mode(mode: str) -> Tuple[float, float]:
    """Return transmissibility_ratio_clip for staged fixed-seed face-closure tests."""
    m = str(mode).lower()
    if m in ("default", "overrelaxed_default"):
        return (0.05, 20.0)
    if m in ("tpfa", "tpfa_fallback", "euclidean_tpfa"):
        # Force the over-relaxed formula to be rejected, yielding A/|d| fallback.
        return (1.0e99, 1.0e100)
    if m in ("tight_clip", "clip_0p5_2"):
        return (0.5, 2.0)
    if m in ("moderate_clip", "clip_0p25_4"):
        return (0.25, 4.0)
    if m in ("raw", "raw_overrelaxed", "unclipped"):
        return (0.0, 1.0e99)
    raise ValueError(f"Unknown face transmissibility mode: {mode}")


def _pb618_clone_geometry_with_scaled_twall(geom: GPUFVMGeometry, wall_beta: float) -> GPUFVMGeometry:
    """Return a geometry with the same topology/faces but scaled wall transmissibility."""
    beta = float(wall_beta)
    return GPUFVMGeometry(mask=geom.mask, labels=geom.labels, dist=geom.dist, n_cells=geom.n_cells,
                          volume=geom.volume, centroid=geom.centroid, owner=geom.owner, neigh=geom.neigh,
                          area=geom.area, avec=geom.avec, face_centroid=geom.face_centroid, dvec=geom.dvec,
                          tproj=geom.tproj, w_owner=geom.w_owner, w_neigh=geom.w_neigh,
                          twall=geom.twall * beta, laplacian=geom.laplacian)


def nonorthogonal_diffusion_correction_gpu(U: cp.ndarray, geom: GPUFVMGeometry, cfg: PB615Config) -> cp.ndarray:
    """Explicit non-orthogonality diffusion correction, all on GPU.

    The implicit matrix contains the orthogonal part and wall drag.  This routine
    computes the explicit correction

        nu/V_i sum_f alpha * k_f dot grad(U)_f,

    with k_f = a_f - T_f d_f.  It is optional and controlled by
    cfg.explicit_nonorthogonal_correction.
    """
    alpha = float(getattr(cfg, "explicit_nonorthogonal_correction", 0.0))
    n = geom.n_cells
    if abs(alpha) <= 0.0 or int(geom.owner.size) == 0:
        return cp.zeros_like(U)
    nf = int(geom.owner.size)
    kvec = geom.avec - geom.tproj[:, None] * geom.dvec
    out = cp.zeros_like(U)
    w_o = geom.w_owner[:, None]
    w_n = geom.w_neigh[:, None]
    for c in range(3):
        grad = lsq_gradient_gpu(U[:, c], geom, cfg)  # [n,3]
        gf = w_o * grad[geom.owner] + w_n * grad[geom.neigh]
        corr_f = alpha * float(cfg.nu) * cp.sum(kvec * gf, axis=1)
        acc = cp.bincount(geom.owner, weights=corr_f, minlength=n).astype(cp.float64)
        acc -= cp.bincount(geom.neigh, weights=corr_f, minlength=n).astype(cp.float64)
        out[:, c] = acc / cp.maximum(geom.volume, 1.0e-300)
    return out


# Override PB61.7/PB61.8 base predictor so PB61.8 can actually ablate explicit NOC.
def implicit_momentum_predictor_gpu(
    U: cp.ndarray,
    p: cp.ndarray,
    phi: cp.ndarray,
    geom: GPUFVMGeometry,
    cfg: PB615Config,
    A_vel,
    M_vel=None,
) -> cp.ndarray:
    """Implicit all-GPU Stokes/N-S predictor with optional explicit nonorthogonal correction."""
    gradp = lsq_gradient_gpu(p, geom, cfg)
    Cterm = convection_term_gpu(U, phi, geom) if bool(cfg.enable_convection) else cp.zeros_like(U)
    Dnoc = nonorthogonal_diffusion_correction_gpu(U, geom, cfg)
    body = cp.asarray(cfg.body_force, dtype=cp.float64)[None, :]
    rhs = (geom.volume[:, None] / float(cfg.dt)) * U + geom.volume[:, None] * (body - gradp / float(cfg.rho) - Cterm + Dnoc)
    Ustar = cp.empty_like(U)
    for c in range(3):
        sol, info = _cg_gpu_tol(A_vel, rhs[:, c], float(cfg.momentum_tol), int(cfg.momentum_maxiter), M=M_vel)
        info_i = int(info)
        if info_i != 0:
            raise RuntimeError(f"GPU momentum CG did not converge for component {c}; info={info_i}")
        Ustar[:, c] = sol
    return Ustar


# Override residual as well, so reported momentum residual includes explicit NOC.
def steady_momentum_residual_gpu(U: cp.ndarray, p: cp.ndarray, phi: cp.ndarray, geom: GPUFVMGeometry, cfg: PB615Config) -> cp.ndarray:
    gradp = lsq_gradient_gpu(p, geom, cfg)
    Dterm = diffusion_term_gpu(U, geom, cfg) + nonorthogonal_diffusion_correction_gpu(U, geom, cfg)
    Cterm = convection_term_gpu(U, phi, geom) if bool(cfg.enable_convection) else cp.zeros_like(U)
    body = cp.asarray(cfg.body_force, dtype=cp.float64)[None, :]
    return -Cterm + Dterm + body - gradp / float(cfg.rho)


def analytic_duct_cell_average_ux_gpu(mask: cp.ndarray, labels: cp.ndarray, n_cells: int,
                                      shape_zyx: Tuple[int, int, int], cfg: PB615Config,
                                      terms: int = 41) -> cp.ndarray:
    """Cell-average analytical duct velocity over the voxels belonging to each coarse cell.

    This is a fairer comparison than evaluating the analytical velocity only at
    the coarse centroid, because the solved velocity is a cell-centered control-
    volume average rather than a pointwise DNS value.
    """
    D, H, W = [int(v) for v in shape_zyx]
    h = float(cfg.voxel_size)
    zz, yy, xx = cp.indices((D, H, W), dtype=cp.float64)
    coords = cp.stack([(xx.ravel() + 0.5) * h, (yy.ravel() + 0.5) * h, (zz.ravel() + 0.5) * h], axis=1)
    valid = mask.ravel() & (labels.ravel() >= 0)
    lab = labels.ravel()[valid].astype(cp.int32)
    ux = rectangular_duct_analytic_ux_gpu(coords[valid], shape_zyx, cfg, terms=terms)
    cnt = cp.bincount(lab, minlength=n_cells).astype(cp.float64)
    avg = cp.bincount(lab, weights=ux, minlength=n_cells).astype(cp.float64) / cp.maximum(cnt, 1.0e-300)
    return avg


def _pb618_weighted_rel_l2_vec_gpu(A: cp.ndarray, B: cp.ndarray, weights: cp.ndarray) -> float:
    return _gpu_weighted_rel_l2_vec(A, B, weights)


def _pb618_candidate_score(row: Dict[str, object]) -> float:
    """Scalar objective for fixed-seed ablation. Smaller is better."""
    penalty = 0.0
    if not bool(row.get("steady_converged", False)):
        penalty += 10.0
    if row.get("solver_failure", ""):
        penalty += 10.0
    for key in ("U_rel_l2_vs_voxel_ref", "U_rel_l2_analytic_cellavg", "K_rel_err_vs_voxel_ref", "K_rel_err_analytic_cellavg"):
        val = float(row.get(key, float("inf")))
        if not math.isfinite(val):
            penalty += 10.0
    return (float(row.get("U_rel_l2_vs_voxel_ref", 1.0e9))
            + float(row.get("U_rel_l2_analytic_cellavg", 1.0e9))
            + 0.50 * float(row.get("K_rel_err_vs_voxel_ref", 1.0e9))
            + 0.25 * float(row.get("K_rel_err_analytic_cellavg", 1.0e9))
            + penalty)


def _pb618_run_candidate_gpu(base: PB615Config, mask: cp.ndarray, shape_zyx: Tuple[int, int, int],
                             stride_zyx: Tuple[int, int, int], ref_geom: GPUFVMGeometry,
                             ref_result: Dict[str, object], *, stage: str, candidate: str,
                             face_mode: str = "overrelaxed_default", wall_beta: float = 1.0,
                             wall_distance_floor: Optional[float] = None,
                             noc_alpha: float = 0.0,
                             reconstruct: bool = False,
                             recon_lambda: float = 1.0e-8,
                             terms: int = 41) -> Dict[str, object]:
    local = _pb618_cfg(base, out_dir=PB618_OUT_DIR, periodic_x=True, body_force=(2.0e-3, 0.0, 0.0),
                       nu=1.0, dt=2.0, n_steps=600, report_every=0, stop_on_steady=True,
                       enable_convection=True, raise_on_solver_failure=False,
                       explicit_nonorthogonal_correction=float(noc_alpha),
                       velocity_reconstruct_from_flux=bool(reconstruct),
                       velocity_reconstruction_lambda=float(recon_lambda),
                       transmissibility_ratio_clip=_pb618_face_clip_for_mode(face_mode))
    if wall_distance_floor is not None:
        local.wall_distance_floor = float(wall_distance_floor)
    geom = build_geometry_from_mask_stride_gpu(mask, stride_zyx, local)
    if abs(float(wall_beta) - 1.0) > 1.0e-15:
        geom = _pb618_clone_geometry_with_scaled_twall(geom, float(wall_beta))
    label = f"pb618_{stage}_{candidate}".replace(" ", "_").replace("/", "_")
    res = run_velocity_pressure_projection_gpu(geom, local, run_label=label)

    # Analytical references: centroid and cell-average.
    ux_centroid = rectangular_duct_analytic_ux_gpu(geom.centroid, shape_zyx, local, terms=terms)
    U_analytic_centroid = cp.zeros_like(res["U"]); U_analytic_centroid[:, 0] = ux_centroid
    ux_avg = analytic_duct_cell_average_ux_gpu(mask, geom.labels, geom.n_cells, shape_zyx, local, terms=terms)
    U_analytic_avg = cp.zeros_like(res["U"]); U_analytic_avg[:, 0] = ux_avg
    mean_avg = cp.sum(ux_avg * geom.volume) / cp.maximum(cp.sum(geom.volume), 1.0e-300)
    k_analytic_avg = float((local.nu * mean_avg / float(local.body_force[0])).get())

    # Voxel-grid reference averaged to this same fixed seed partition.
    Uref_c, pref_c = coarsen_voxel_reference_to_coarse_gpu(ref_geom, ref_result, geom)
    row = {
        "validation": "fixed_seed_operator_ablation",
        "stage": str(stage), "candidate": str(candidate),
        "shape": str(tuple(shape_zyx)), "stride": str(tuple(stride_zyx)),
        "n_cells": int(geom.n_cells), "n_faces": int(geom.owner.size),
        "face_mode": str(face_mode), "wall_beta": float(wall_beta),
        "wall_distance_floor": float(local.wall_distance_floor),
        "noc_alpha": float(noc_alpha), "velocity_reconstruct_from_flux": bool(reconstruct),
        "velocity_reconstruction_lambda": float(recon_lambda),
        "steady_converged": bool(res.get("steady_converged", False)),
        "steps_completed": int(res.get("steps_completed", -1)),
        "solver_failure": str(res.get("solver_failure", "")),
        "mass_inf_per_volume": float(res.get("mass_inf_per_volume", float("nan"))),
        "steady_momentum_inf": float(res.get("steady_momentum_inf", float("nan"))),
        "U_rel_l2_analytic_centroid": _gpu_weighted_rel_l2_vec(res["U"], U_analytic_centroid, geom.volume),
        "U_rel_l2_analytic_cellavg": _gpu_weighted_rel_l2_vec(res["U"], U_analytic_avg, geom.volume),
        "Ux_rel_inf_analytic_cellavg": _gpu_rel_inf_scalar(res["U"][:, 0], ux_avg),
        "U_rel_l2_vs_voxel_ref": _gpu_weighted_rel_l2_vec(res["U"], Uref_c, geom.volume),
        "K_eff_x": float(res["K_eff_x"]),
        "K_ref_voxel_x": float(ref_result["K_eff_x"]),
        "K_rel_err_vs_voxel_ref": abs(float(res["K_eff_x"]) - float(ref_result["K_eff_x"])) / max(abs(float(ref_result["K_eff_x"])), 1.0e-300),
        "K_ref_analytic_cellavg": k_analytic_avg,
        "K_rel_err_analytic_cellavg": abs(float(res["K_eff_x"]) - k_analytic_avg) / max(abs(k_analytic_avg), 1.0e-300),
        "max_cfl": float(res.get("max_cfl", float("nan"))),
        "max_convection_inf": float(res.get("max_convection_inf", float("nan"))),
    }
    row["score"] = _pb618_candidate_score(row)
    print("[PB61.8-GPU] fixed-seed candidate:", row)
    return row


def _pb618_best_row(rows: Sequence[Dict[str, object]]) -> Dict[str, object]:
    valid = [r for r in rows if math.isfinite(float(r.get("score", float("inf"))))]
    return min(valid, key=lambda r: float(r["score"])) if valid else dict(rows[0])


def _pb618_write_rows(out: Path, name: str, rows: Sequence[Dict[str, object]]) -> None:
    if not rows:
        return
    keys = list(rows[0].keys())
    with (out / name).open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)


def run_fixed_seed_operator_ablation_gpu(cfg: Optional[PB615Config] = None,
                                         shape_zyx: Tuple[int, int, int] = (12, 12, 24),
                                         fixed_stride_zyx: Tuple[int, int, int] = (3, 3, 3),
                                         terms: int = 41,
                                         zip_outputs: bool = True) -> Dict[str, object]:
    """PB61.8 main routine: fixed-seed ablation to reduce velocity-field loss.

    This keeps the seed layout fixed and changes only operator/closure choices:
      1. face transmissibility mode;
      2. wall transmissibility scale / wall distance floor;
      3. explicit non-orthogonality correction;
      4. flux-consistent velocity reconstruction.

    It compares every candidate against both a voxel-grid FVM reference and a
    cell-averaged analytical duct reference.  The goal is to separate true seed
    resolution error from closure/operator error.
    """
    base = _pb618_cfg(cfg, out_dir=PB618_OUT_DIR, periodic_x=True, body_force=(2.0e-3, 0.0, 0.0),
                      nu=1.0, dt=2.0, n_steps=600, report_every=0,
                      stop_on_steady=True, enable_convection=True,
                      raise_on_solver_failure=False)
    out = _pb618_out(base)
    mask = make_periodic_channel_mask_gpu(shape_zyx)

    print("[PB61.8-GPU] fixed-seed ablation: building voxel-grid reference")
    ref_geom = build_geometry_from_mask_stride_gpu(mask, (1, 1, 1), base)
    ref = run_velocity_pressure_projection_gpu(ref_geom, base, run_label="pb618_voxel_reference")

    rows = []

    # Baseline candidate.
    baseline = _pb618_run_candidate_gpu(base, mask, shape_zyx, fixed_stride_zyx, ref_geom, ref,
                                        stage="baseline", candidate="default",
                                        face_mode="overrelaxed_default", wall_beta=1.0,
                                        noc_alpha=0.0, reconstruct=False, terms=terms)
    rows.append(baseline)

    # Stage 1: face transmissibility mode.
    face_rows = []
    for mode in ("tpfa_fallback", "tight_clip", "moderate_clip", "overrelaxed_default", "raw_overrelaxed"):
        face_rows.append(_pb618_run_candidate_gpu(base, mask, shape_zyx, fixed_stride_zyx, ref_geom, ref,
                                                  stage="face_mode", candidate=mode,
                                                  face_mode=mode, wall_beta=1.0, noc_alpha=0.0,
                                                  reconstruct=False, terms=terms))
    rows.extend(face_rows)
    best_face = _pb618_best_row(face_rows)
    best_face_mode = str(best_face["face_mode"])
    print(f"[PB61.8-GPU] best face mode: {best_face_mode}, score={best_face['score']}")

    # Stage 2: wall scaling and wall distance floor around best face mode.
    wall_rows = []
    for beta in (0.75, 0.9, 1.0, 1.1, 1.25, 1.5):
        wall_rows.append(_pb618_run_candidate_gpu(base, mask, shape_zyx, fixed_stride_zyx, ref_geom, ref,
                                                  stage="wall_beta", candidate=f"beta_{beta:g}",
                                                  face_mode=best_face_mode, wall_beta=beta,
                                                  noc_alpha=0.0, reconstruct=False, terms=terms))
    for floor in (0.25, 0.35, 0.5, 0.75):
        wall_rows.append(_pb618_run_candidate_gpu(base, mask, shape_zyx, fixed_stride_zyx, ref_geom, ref,
                                                  stage="wall_distance_floor", candidate=f"floor_{floor:g}",
                                                  face_mode=best_face_mode, wall_beta=1.0,
                                                  wall_distance_floor=floor, noc_alpha=0.0,
                                                  reconstruct=False, terms=terms))
    rows.extend(wall_rows)
    best_wall = _pb618_best_row(wall_rows)
    best_wall_beta = float(best_wall["wall_beta"])
    best_wall_floor = float(best_wall["wall_distance_floor"])
    print(f"[PB61.8-GPU] best wall closure: beta={best_wall_beta}, floor={best_wall_floor}, score={best_wall['score']}")

    # Stage 3: explicit non-orthogonality correction.  This is truly active in PB61.8.
    noc_rows = []
    for alpha in (0.0, 0.1, 0.25, 0.5, 0.75, 1.0):
        noc_rows.append(_pb618_run_candidate_gpu(base, mask, shape_zyx, fixed_stride_zyx, ref_geom, ref,
                                                 stage="nonorthogonal_correction", candidate=f"alpha_{alpha:g}",
                                                 face_mode=best_face_mode, wall_beta=best_wall_beta,
                                                 wall_distance_floor=best_wall_floor, noc_alpha=alpha,
                                                 reconstruct=False, terms=terms))
    rows.extend(noc_rows)
    best_noc = _pb618_best_row(noc_rows)
    best_alpha = float(best_noc["noc_alpha"])
    print(f"[PB61.8-GPU] best NOC alpha: {best_alpha}, score={best_noc['score']}")

    # Stage 4: flux-consistent velocity reconstruction.  This can lower U-field
    # comparison error without changing mass-conservative face fluxes.
    recon_rows = []
    recon_rows.append(_pb618_run_candidate_gpu(base, mask, shape_zyx, fixed_stride_zyx, ref_geom, ref,
                                               stage="velocity_reconstruction", candidate="off",
                                               face_mode=best_face_mode, wall_beta=best_wall_beta,
                                               wall_distance_floor=best_wall_floor, noc_alpha=best_alpha,
                                               reconstruct=False, terms=terms))
    for lam in (1.0e-10, 1.0e-8, 1.0e-6, 1.0e-4):
        recon_rows.append(_pb618_run_candidate_gpu(base, mask, shape_zyx, fixed_stride_zyx, ref_geom, ref,
                                                   stage="velocity_reconstruction", candidate=f"on_lambda_{lam:g}",
                                                   face_mode=best_face_mode, wall_beta=best_wall_beta,
                                                   wall_distance_floor=best_wall_floor, noc_alpha=best_alpha,
                                                   reconstruct=True, recon_lambda=lam, terms=terms))
    rows.extend(recon_rows)
    best_final = _pb618_best_row(recon_rows + [best_noc, best_wall, best_face, baseline])

    # Improvement summary.
    summary = {
        "validation": "fixed_seed_operator_ablation_summary",
        "shape": str(tuple(shape_zyx)), "fixed_stride": str(tuple(fixed_stride_zyx)),
        "baseline_score": float(baseline["score"]),
        "best_score": float(best_final["score"]),
        "baseline_U_rel_l2_vs_voxel_ref": float(baseline["U_rel_l2_vs_voxel_ref"]),
        "best_U_rel_l2_vs_voxel_ref": float(best_final["U_rel_l2_vs_voxel_ref"]),
        "baseline_U_rel_l2_analytic_cellavg": float(baseline["U_rel_l2_analytic_cellavg"]),
        "best_U_rel_l2_analytic_cellavg": float(best_final["U_rel_l2_analytic_cellavg"]),
        "baseline_K_rel_err_vs_voxel_ref": float(baseline["K_rel_err_vs_voxel_ref"]),
        "best_K_rel_err_vs_voxel_ref": float(best_final["K_rel_err_vs_voxel_ref"]),
        "best_stage": str(best_final["stage"]),
        "best_candidate": str(best_final["candidate"]),
        "best_face_mode": str(best_final["face_mode"]),
        "best_wall_beta": float(best_final["wall_beta"]),
        "best_wall_distance_floor": float(best_final["wall_distance_floor"]),
        "best_noc_alpha": float(best_final["noc_alpha"]),
        "best_velocity_reconstruct_from_flux": bool(best_final["velocity_reconstruct_from_flux"]),
        "best_velocity_reconstruction_lambda": float(best_final["velocity_reconstruction_lambda"]),
        "note": "Fixed seed: only operator/closure/reconstruction knobs are changed; seed count and seed locations are unchanged."
    }
    _pb618_write_rows(out, "pb61_8_fixed_seed_operator_ablation.csv", rows)
    _pb618_write_rows(out, "pb61_8_fixed_seed_best_summary.csv", [summary])
    (out / "pb61_8_validation_manifest.json").write_text(json.dumps({
        "suite": "PB61.8 fixed-seed operator/closure ablation",
        "out_dir": str(out),
        "shape": tuple(shape_zyx),
        "fixed_stride": tuple(fixed_stride_zyx),
        "interpretation": "This tests whether U-field loss can be lowered without changing seed count or seed positions. Current solver remains one vector velocity per coarse control volume plus conservative face fluxes."
    }, indent=2), encoding="utf-8")
    zip_path = None
    if zip_outputs:
        import shutil
        zip_path = shutil.make_archive(str(out), "zip", out)
        print(f"[PB61.8-GPU] zipped outputs: {zip_path}")
    print("[PB61.8-GPU] fixed-seed ablation summary:", summary)
    return {"rows": rows, "baseline": baseline, "best": best_final, "summary": summary, "outputs_zip": zip_path}


PB618_DEFINITIONS_LOADED = True
print("PB61.8 fixed-seed operator-ablation definitions loaded. Run `out = run_fixed_seed_operator_ablation_gpu(PB618Config())`.")


def main() -> None:
    cfg = PB618Config(out_dir=PB618_OUT_DIR)
    out = run_fixed_seed_operator_ablation_gpu(cfg, zip_outputs=True)
    print("PB61.8 best fixed-seed candidate:", out["best"])



## ABC suite wrappers

The next cell adds case generators, topology/moment audits, coarse/reference flow runs, wall calibration, component study, and CSV/plot output generation.

In [ ]:

# ============================================================
# ABC suite built on the PB61.8 GPU code
# ============================================================
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
from typing import Any, Iterable, List
import csv, json, math, time, shutil
import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

PVFV_OUT_DIR = "pvfv_abc_outputs"

# -----------------------------------------------------------------------------
# Configuration helpers
# -----------------------------------------------------------------------------

def pvfv_clone_cfg(cfg: Optional[PB615Config] = None, **updates) -> PB615Config:
    base = cfg if cfg is not None else PB615Config()
    data = {k: getattr(base, k) for k in PB615Config.__dataclass_fields__.keys()}
    data.update(updates)
    return PB615Config(**data)


def pvfv_base_cfg(out_dir: str = PVFV_OUT_DIR, profile: str = "pilot") -> PB615Config:
    """Return a conservative low-Re/Stokes-like configuration for production sweeps.

    The default profile runs all blocks on modest grids.  Increase the shape and
    strides in `pvfv_case_plan('production')` when moving to paper-scale runs.
    """
    profile = str(profile).lower()
    if profile == "production":
        n_steps = 900
        report_every = 100
        geodesic_max_iters = 8192
    else:
        n_steps = 500
        report_every = 0
        geodesic_max_iters = 4096
    return PB615Config(
        out_dir=out_dir,
        periodic_x=True,
        distance_connectivity=6,  # safest default: consistent with positive-area FV paths
        rho=1.0,
        nu=1.0,
        body_force=(2.0e-3, 0.0, 0.0),
        dt=2.0,
        n_steps=n_steps,
        report_every=report_every,
        projection_tol=1.0e-9,
        projection_maxiter=5000,
        pressure_gauge_eps=1.0e-12,
        enable_convection=False,       # permeability/Stokes production baseline
        implicit_diffusion=True,
        momentum_tol=1.0e-8,
        momentum_maxiter=5000,
        use_jacobi_preconditioner=True,
        raise_on_solver_failure=False,
        steady_tol=1.0e-7,
        steady_min_steps=20,
        stop_on_steady=True,
        explicit_nonorthogonal_correction=0.0,
        velocity_reconstruct_from_flux=False,
        velocity_reconstruction_lambda=1.0e-8,
        tikhonov=1.0e-10,
        transmissibility_floor=1.0e-12,
        transmissibility_ratio_clip=(0.05, 20.0),
        wall_distance_floor=0.35,
        geodesic_max_iters=geodesic_max_iters,
        geodesic_tie_eps=1.0e-12,
        demo_shape_zyx=(16, 16, 32),
        demo_seed_stride_zyx=(4, 4, 4),
    )


def pvfv_case_plan(profile: str = "pilot") -> dict:
    """Case plan for canonical + ABC synthetic runs.

    Strides are in z-y-x order.  The flow direction is x with periodic x.
    """
    profile = str(profile).lower()
    if profile == "production":
        shape = (24, 24, 64)
        strides = [(4, 4, 4), (6, 6, 6), (8, 8, 8), (12, 12, 12)]
        fixed_stride = (6, 6, 6)
        beta_values = (0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.4, 1.6, 2.0)
    else:
        shape = (12, 12, 24)
        strides = [(3, 3, 3), (4, 4, 4), (6, 6, 6)]
        fixed_stride = (3, 3, 3)
        beta_values = (0.75, 0.9, 1.0, 1.1, 1.25, 1.5)
    return {
        "profile": profile,
        "shape_zyx": shape,
        "seed_strides": strides,
        "fixed_stride": fixed_stride,
        "beta_values": beta_values,
        "component_case": "C_maze",
        "component_stride": fixed_stride,
    }

# -----------------------------------------------------------------------------
# Synthetic masks for canonical + ABC cases
# -----------------------------------------------------------------------------

def pvfv_make_mask_gpu(case_name: str, shape_zyx: Tuple[int, int, int]) -> cp.ndarray:
    """Return a binary traversable mask for the requested case.

    Cases:
      - orthogonal_duct: full periodic-x duct;
      - skewed_duct: oblique internal baffle to activate non-orthogonal geometry;
      - A_thin_wall: thin wall parallel to x separating nearby pores;
      - B_narrow_throat: cross-flow baffle with a small pore throat;
      - C_maze: multiple offset baffles forming a tortuous periodic path.
    """
    D, H, W = [int(v) for v in shape_zyx]
    case = str(case_name).lower()
    mask = cp.ones((D, H, W), dtype=cp.bool_)
    if case in ("orthogonal", "orthogonal_duct", "duct", "channel"):
        return mask

    zz, yy, xx = cp.indices((D, H, W))

    if case in ("a_thin_wall", "thin_wall", "case_a"):
        # Thin wall parallel to the flow direction.  It creates two adjacent
        # traversable regions separated by a one-voxel solid sheet, which is
        # useful for demonstrating that Euclidean proximity cannot cross walls.
        ywall = H // 2
        mask[:, ywall, :] = False
        # Make two small windows so that the full domain remains connected for flow.
        z0, z1 = max(1, D // 3), min(D - 1, D // 3 + max(1, D // 6))
        x0, x1 = max(1, W // 4), min(W - 1, W // 4 + max(2, W // 8))
        mask[z0:z1, ywall, x0:x1] = True
        z0b, z1b = max(1, (2 * D) // 3), min(D - 1, (2 * D) // 3 + max(1, D // 6))
        x0b, x1b = max(1, (3 * W) // 4 - max(2, W // 8)), min(W - 1, (3 * W) // 4)
        mask[z0b:z1b, ywall, x0b:x1b] = True
        return mask

    if case in ("b_narrow_throat", "narrow_throat", "case_b"):
        # A solid baffle normal to the flow with a small central aperture.
        xmid = W // 2
        mask[:, :, xmid] = False
        dz = max(1, D // 8)
        dy = max(1, H // 8)
        z0, z1 = max(1, D // 2 - dz), min(D - 1, D // 2 + dz + 1)
        y0, y1 = max(1, H // 2 - dy), min(H - 1, H // 2 + dy + 1)
        mask[z0:z1, y0:y1, xmid] = True
        return mask

    if case in ("c_maze", "maze", "case_c"):
        # Three offset baffles.  Similar spirit to make_baffled_periodic_channel_mask_gpu,
        # but with one more obstacle to generate a more tortuous path.
        mask = cp.ones((D, H, W), dtype=cp.bool_)
        if W < 12 or H < 8 or D < 8:
            return make_baffled_periodic_channel_mask_gpu(shape_zyx)
        xs = [max(2, W // 4), max(3, W // 2), min(W - 3, (3 * W) // 4)]
        windows = [
            (slice(max(1, D // 8), max(2, D // 8 + D // 3)),
             slice(max(1, H // 8), max(2, H // 8 + H // 3))),
            (slice(max(1, D // 2 - D // 6), min(D - 1, D // 2 + D // 6)),
             slice(max(1, H // 2 - H // 6), min(H - 1, H // 2 + H // 6))),
            (slice(max(1, D - D // 8 - D // 3), max(2, D - D // 8)),
             slice(max(1, H - H // 8 - H // 3), max(2, H - H // 8))),
        ]
        for x, (zs, ys) in zip(xs, windows):
            mask[:, :, x] = False
            mask[zs, ys, x] = True
        return mask

    if case in ("skewed_duct", "oblique", "skewed"):
        # An oblique thin obstacle sheet with a broad passage.  It is not meant as
        # a complex porous medium; it is a canonical non-orthogonal geometry test.
        x0 = W // 3
        slab = (xx == ((x0 + (yy - H // 2) // max(1, H // 6)) % W))
        mask[slab] = False
        # Create a diagonal/open central window.
        window = (cp.abs(yy - H // 2) <= max(1, H // 6)) & (cp.abs(zz - D // 2) <= max(1, D // 6))
        mask[slab & window] = True
        return mask

    raise ValueError(f"Unknown mask case: {case_name}")

# -----------------------------------------------------------------------------
# Face-connected component split and topology diagnostics
# -----------------------------------------------------------------------------

def _asnumpy_bool(a) -> np.ndarray:
    return cp.asnumpy(a).astype(bool, copy=False)


def _asnumpy_i32(a) -> np.ndarray:
    return cp.asnumpy(a).astype(np.int32, copy=False)


def pvfv_face_connected_reindex_cpu(mask_gpu: cp.ndarray, labels_gpu: cp.ndarray) -> Tuple[cp.ndarray, dict]:
    """Split each label into 6-face-connected components and return reindexed GPU labels.

    This is a diagnostic/assembly step.  It transfers labels to CPU because it is
    cheap compared with the flow solve for the pilot/ABC sizes.  For large images,
    replace with a GPU connected-component kernel.
    """
    mask = _asnumpy_bool(mask_gpu)
    labels = _asnumpy_i32(labels_gpu)
    D, H, W = labels.shape
    new = np.full(labels.shape, -1, dtype=np.int32)
    visited = np.zeros(labels.shape, dtype=bool)
    dirs = [(1,0,0), (-1,0,0), (0,1,0), (0,-1,0), (0,0,1), (0,0,-1)]
    comp_count = 0
    orig_component_counts = {}
    from collections import deque
    coords = np.argwhere(mask & (labels >= 0))
    for z, y, x in coords:
        if visited[z, y, x]:
            continue
        lab = int(labels[z, y, x])
        q = deque([(int(z), int(y), int(x))])
        visited[z, y, x] = True
        new[z, y, x] = comp_count
        while q:
            cz, cy, cx = q.popleft()
            for dz, dy, dx in dirs:
                nz, ny, nx = cz + dz, cy + dy, cx + dx
                if nz < 0 or nz >= D or ny < 0 or ny >= H:
                    continue
                if nx < 0 or nx >= W:
                    # periodic x is handled as wrap for connectivity because the solver uses periodic x
                    nx %= W
                if visited[nz, ny, nx] or (not mask[nz, ny, nx]) or int(labels[nz, ny, nx]) != lab:
                    continue
                visited[nz, ny, nx] = True
                new[nz, ny, nx] = comp_count
                q.append((nz, ny, nx))
        orig_component_counts[lab] = orig_component_counts.get(lab, 0) + 1
        comp_count += 1
    n_orig = len(orig_component_counts)
    n_split_extra = int(comp_count - n_orig)
    n_disconnected_labels = int(sum(1 for v in orig_component_counts.values() if v > 1))
    info = {
        "n_original_labels": int(n_orig),
        "n_cv": int(comp_count),
        "n_split_extra": n_split_extra,
        "n_face_disconnected_labels": n_disconnected_labels,
        "face_connected_fraction": float((n_orig - n_disconnected_labels) / max(n_orig, 1)),
    }
    return cp.asarray(new, dtype=cp.int32), info


def pvfv_zero_area_candidate_count_cpu(mask_gpu: cp.ndarray, labels_gpu: cp.ndarray) -> int:
    """Count unique label pairs touching in 26-neighborhood but not by a face."""
    mask = _asnumpy_bool(mask_gpu)
    labels = _asnumpy_i32(labels_gpu)
    D, H, W = labels.shape
    pairs26 = set()
    pairs6 = set()
    offsets = []
    for dz in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                if dz == dy == dx == 0:
                    continue
                # one-sided offsets only to avoid duplicates
                if (dz, dy, dx) <= (0, 0, 0):
                    continue
                offsets.append((dz, dy, dx))
    for z in range(D):
        for y in range(H):
            for x in range(W):
                if not mask[z, y, x] or labels[z, y, x] < 0:
                    continue
                li = int(labels[z, y, x])
                for dz, dy, dx in offsets:
                    nz, ny, nx = z + dz, y + dy, x + dx
                    if nz < 0 or nz >= D or ny < 0 or ny >= H:
                        continue
                    if nx < 0 or nx >= W:
                        nx %= W
                    if not mask[nz, ny, nx] or labels[nz, ny, nx] < 0:
                        continue
                    lj = int(labels[nz, ny, nx])
                    if li == lj:
                        continue
                    pair = (min(li, lj), max(li, lj))
                    pairs26.add(pair)
                    if abs(dz) + abs(dy) + abs(dx) == 1:
                        pairs6.add(pair)
    return int(len(pairs26 - pairs6))


def pvfv_build_geometry_from_labels_ncells_gpu(mask: cp.ndarray, labels: cp.ndarray, dist: cp.ndarray, n_cells: int, cfg: PB615Config) -> GPUFVMGeometry:
    """Build geometry when labels may have been reindexed after component splitting."""
    volume, centroid = build_cells_gpu(mask, labels, int(n_cells), cfg)
    faces = build_positive_area_faces_gpu(mask, labels, int(n_cells), cfg)
    owner = faces["owner"]
    neigh = faces["neigh"]
    area = faces["area"].astype(cp.float64)
    avec = cp.stack([faces["ax"], faces["ay"], faces["az"]], axis=1).astype(cp.float64)
    face_centroid = cp.stack([faces["x0"], faces["y0"], faces["z0"]], axis=1).astype(cp.float64)
    ci = centroid[owner]
    cj = centroid[neigh]
    dvec = cj - ci
    if cfg.periodic_x:
        Lx = int(mask.shape[2]) * float(cfg.voxel_size)
        dx = dvec[:, 0]
        dvec[:, 0] = cp.where(dx > 0.5 * Lx, dx - Lx, cp.where(dx < -0.5 * Lx, dx + Lx, dx))
    xi = ci.copy(); xj = cj.copy(); xf = face_centroid.copy()
    if cfg.periodic_x:
        Lx = int(mask.shape[2]) * float(cfg.voxel_size)
        dfi = xf[:, 0] - xi[:, 0]
        dfj = xf[:, 0] - xj[:, 0]
        dfi = cp.where(dfi > 0.5 * Lx, dfi - Lx, cp.where(dfi < -0.5 * Lx, dfi + Lx, dfi))
        dfj = cp.where(dfj > 0.5 * Lx, dfj - Lx, cp.where(dfj < -0.5 * Lx, dfj + Lx, dfj))
        ri = cp.sqrt(dfi*dfi + (xf[:,1]-xi[:,1])**2 + (xf[:,2]-xi[:,2])**2)
        rj = cp.sqrt(dfj*dfj + (xf[:,1]-xj[:,1])**2 + (xf[:,2]-xj[:,2])**2)
    else:
        ri = cp.linalg.norm(face_centroid - ci, axis=1)
        rj = cp.linalg.norm(face_centroid - cj, axis=1)
    denom = cp.maximum(ri + rj, 1.0e-300)
    w_owner = rj / denom
    w_neigh = 1.0 - w_owner
    adotd = cp.sum(avec * dvec, axis=1)
    anorm2 = cp.sum(avec * avec, axis=1)
    dnorm = cp.maximum(cp.linalg.norm(dvec, axis=1), 1.0e-300)
    fallback = area / dnorm
    raw = anorm2 / cp.maximum(adotd, 1.0e-300)
    lo, hi = cfg.transmissibility_ratio_clip
    ratio = raw / cp.maximum(fallback, 1.0e-300)
    tproj = cp.where((adotd > 0.0) & (ratio >= lo) & (ratio <= hi), raw, fallback)
    tproj = cp.maximum(tproj, float(cfg.transmissibility_floor))
    twall = build_wall_moments_gpu(mask, labels, centroid, int(n_cells), cfg)
    nf = int(owner.size)
    rows = cp.concatenate([owner, owner, neigh, neigh]).astype(cp.int32)
    cols = cp.concatenate([owner, neigh, neigh, owner]).astype(cp.int32)
    vals = cp.concatenate([tproj, -tproj, tproj, -tproj]).astype(cp.float64)
    L = cpx_sp.coo_matrix((vals, (rows, cols)), shape=(int(n_cells), int(n_cells))).tocsr()
    if cfg.pressure_gauge_eps > 0:
        L = L + float(cfg.pressure_gauge_eps) * cpx_sp.eye(int(n_cells), dtype=cp.float64, format="csr")
    return GPUFVMGeometry(mask=mask, labels=labels, dist=dist, n_cells=int(n_cells),
                          volume=volume, centroid=centroid, owner=owner, neigh=neigh,
                          area=area, avec=avec, face_centroid=face_centroid, dvec=dvec,
                          tproj=tproj, w_owner=w_owner, w_neigh=w_neigh, twall=twall,
                          laplacian=L)


def pvfv_build_geometry_timed(mask: cp.ndarray, stride_zyx: Tuple[int, int, int], cfg: PB615Config,
                               *, split_face_components: bool = True,
                               label_mode: str = "exact_geodesic") -> Tuple[GPUFVMGeometry, dict]:
    """Build labels and moments, returning geometry plus timing/topology metadata."""
    t0 = time.perf_counter()
    seed_flat = make_regular_seeds_gpu(mask, stride_zyx)
    cp.cuda.Stream.null.synchronize()
    t_seed = time.perf_counter() - t0

    if label_mode != "exact_geodesic":
        raise NotImplementedError("This notebook currently provides exact_geodesic labels. Drop ROI--JFA here when available.")
    t1 = time.perf_counter()
    dist, labels = geodesic_voronoi_labels_gpu(mask, seed_flat, cfg)
    cp.cuda.Stream.null.synchronize()
    t_label = time.perf_counter() - t1

    n_seed = int(seed_flat.size)
    t_split0 = time.perf_counter()
    split_info = {
        "n_original_labels": n_seed,
        "n_cv": n_seed,
        "n_split_extra": 0,
        "n_face_disconnected_labels": 0,
        "face_connected_fraction": 1.0,
    }
    if split_face_components:
        labels, split_info = pvfv_face_connected_reindex_cpu(mask, labels)
    cp.cuda.Stream.null.synchronize()
    t_split = time.perf_counter() - t_split0

    n0a = pvfv_zero_area_candidate_count_cpu(mask, labels)

    t2 = time.perf_counter()
    geom = pvfv_build_geometry_from_labels_ncells_gpu(mask, labels, dist, int(split_info["n_cv"]), cfg)
    cp.cuda.Stream.null.synchronize()
    t_mom = time.perf_counter() - t2
    meta = {
        "S": n_seed,
        "N_cv": int(geom.n_cells),
        "N_fl": int(mask.sum().get()),
        "C_comp": float(int(mask.sum().get()) / max(int(geom.n_cells), 1)),
        "t_seed_s": t_seed,
        "t_label_s": t_label,
        "t_split_s": t_split,
        "t_part_s": t_seed + t_label + t_split,
        "t_mom_s": t_mom,
        "N_split": int(split_info["n_split_extra"]),
        "N_face_disconnected_labels": int(split_info["n_face_disconnected_labels"]),
        "face_connected_fraction": float(split_info["face_connected_fraction"]),
        "N_0A": int(n0a),
        "label_mode": label_mode,
        "distance_connectivity": int(cfg.distance_connectivity),
    }
    return geom, meta

# -----------------------------------------------------------------------------
# Reference flux aggregation and metrics
# -----------------------------------------------------------------------------

def pvfv_weighted_rel_l2_vec_gpu(A: cp.ndarray, B: cp.ndarray, weights: cp.ndarray) -> float:
    num = cp.sqrt(cp.sum(weights * cp.sum((A - B) ** 2, axis=1)))
    den = cp.sqrt(cp.sum(weights * cp.sum(B ** 2, axis=1)))
    return float((num / cp.maximum(den, 1.0e-300)).get())


def pvfv_phi_ref_on_coarse_faces_cpu(ref_geom: GPUFVMGeometry, ref_result: Dict[str, Any], coarse_geom: GPUFVMGeometry) -> np.ndarray:
    """Sum voxel-reference face fluxes over each coarse facelet set.

    This uses the voxel reference's stored oriented face flux dictionary and scans
    the coarse label field across 6-neighbor voxel faces.  The returned array is
    aligned with `coarse_geom.owner/neigh`, whose orientation is owner -> neigh.
    """
    mask = _asnumpy_bool(coarse_geom.mask)
    c_lab = _asnumpy_i32(coarse_geom.labels)
    r_lab = _asnumpy_i32(ref_geom.labels)
    D, H, W = c_lab.shape
    ref_owner = cp.asnumpy(ref_geom.owner).astype(np.int64)
    ref_neigh = cp.asnumpy(ref_geom.neigh).astype(np.int64)
    ref_phi = cp.asnumpy(ref_result["phi"]).astype(float)
    ref_dict = {(int(o), int(n)): float(v) for o, n, v in zip(ref_owner, ref_neigh, ref_phi)}
    coarse_sum = {}

    def add_face(z, y, x, nz, ny, nx):
        if not (mask[z, y, x] and mask[nz, ny, nx]):
            return
        li = int(c_lab[z, y, x]); lj = int(c_lab[nz, ny, nx])
        if li < 0 or lj < 0 or li == lj:
            return
        clo, chi = (li, lj) if li < lj else (lj, li)
        ri = int(r_lab[z, y, x]); rj = int(r_lab[nz, ny, nx])
        if ri < 0 or rj < 0 or ri == rj:
            return
        rlo, rhi = (ri, rj) if ri < rj else (rj, ri)
        val = ref_dict.get((rlo, rhi), 0.0)
        # Convert reference orientation to p -> q.
        sign_ref_pq = 1.0 if ri == rlo else -1.0
        # Convert p -> q to coarse owner(min label) -> neigh(max label).
        sign_coarse = 1.0 if li == clo else -1.0
        coarse_sum[(clo, chi)] = coarse_sum.get((clo, chi), 0.0) + sign_coarse * sign_ref_pq * val

    for z in range(D):
        for y in range(H):
            for x in range(W):
                # +x with periodic wrap
                add_face(z, y, x, z, y, (x + 1) % W)
                if y + 1 < H:
                    add_face(z, y, x, z, y + 1, x)
                if z + 1 < D:
                    add_face(z, y, x, z + 1, y, x)
    co_owner = cp.asnumpy(coarse_geom.owner).astype(np.int64)
    co_neigh = cp.asnumpy(coarse_geom.neigh).astype(np.int64)
    out = np.zeros(len(co_owner), dtype=float)
    for k, (o, n) in enumerate(zip(co_owner, co_neigh)):
        out[k] = coarse_sum.get((int(o), int(n)), 0.0)
    return out


def pvfv_flux_rel_error(ref_geom: GPUFVMGeometry, ref_result: Dict[str, Any], coarse_geom: GPUFVMGeometry, coarse_result: Dict[str, Any]) -> float:
    phi_ref = pvfv_phi_ref_on_coarse_faces_cpu(ref_geom, ref_result, coarse_geom)
    phi = cp.asnumpy(coarse_result["phi"]).astype(float)
    den = np.linalg.norm(phi_ref)
    return float(np.linalg.norm(phi - phi_ref) / max(den, 1.0e-300))


def pvfv_run_flow_case(case_name: str, mask: cp.ndarray, stride: Tuple[int, int, int], cfg: PB615Config,
                        ref_geom: GPUFVMGeometry, ref_result: Dict[str, Any],
                        *, wall_beta: float = 1.0,
                        face_mode: str = "overrelaxed_default",
                        noc_alpha: float = 0.0,
                        reconstruct: bool = False,
                        split_face_components: bool = True,
                        run_tag: str = "coarse") -> Tuple[dict, GPUFVMGeometry, Dict[str, Any]]:
    """Build and solve one coarse case; return row, geometry, result."""
    local = pvfv_clone_cfg(cfg,
                            transmissibility_ratio_clip=_pb618_face_clip_for_mode(face_mode),
                            explicit_nonorthogonal_correction=float(noc_alpha),
                            velocity_reconstruct_from_flux=bool(reconstruct))
    geom, meta = pvfv_build_geometry_timed(mask, stride, local, split_face_components=split_face_components)
    if abs(float(wall_beta) - 1.0) > 1.0e-15:
        geom = _pb618_clone_geometry_with_scaled_twall(geom, float(wall_beta))
    res = run_velocity_pressure_projection_gpu(geom, local, run_label=f"pvfv_{case_name}_{run_tag}_{stride}")
    Uref_c, pref_c = coarsen_voxel_reference_to_coarse_gpu(ref_geom, ref_result, geom)
    e_u = pvfv_weighted_rel_l2_vec_gpu(res["U"], Uref_c, geom.volume)
    e_k = abs(float(res["K_eff_x"]) - float(ref_result["K_eff_x"])) / max(abs(float(ref_result["K_eff_x"])), 1.0e-300)
    try:
        e_phi = pvfv_flux_rel_error(ref_geom, ref_result, geom, res)
    except Exception as exc:
        print(f"[flow] flux error failed for {case_name}/{stride}: {exc}")
        e_phi = float("nan")
    ref_total = float(ref_result.get("pvfv_total_time_s", ref_result.get("elapsed_s", float("nan"))))
    t_total = float(meta["t_part_s"] + meta["t_mom_s"] + float(res.get("elapsed_s", 0.0)))
    row = {
        "case": case_name,
        "run_tag": run_tag,
        "stride_zyx": str(tuple(stride)),
        "S": int(meta["S"]),
        "N_cv": int(meta["N_cv"]),
        "N_fl": int(meta["N_fl"]),
        "C_comp": float(meta["C_comp"]),
        "N_split": int(meta["N_split"]),
        "N_0A": int(meta["N_0A"]),
        "face_connected_fraction": float(meta["face_connected_fraction"]),
        "face_mode": str(face_mode),
        "wall_beta": float(wall_beta),
        "wall_distance_floor": float(local.wall_distance_floor),
        "noc_alpha": float(noc_alpha),
        "reconstruct": bool(reconstruct),
        "e_K": float(e_k),
        "e_u": float(e_u),
        "e_phi": float(e_phi),
        "mass_inf_per_volume": float(res.get("mass_inf_per_volume", float("nan"))),
        "mass_l2_per_volume": float(res.get("mass_l2_per_volume", float("nan"))),
        "steady_momentum_inf": float(res.get("steady_momentum_inf", float("nan"))),
        "steady_converged": bool(res.get("steady_converged", False)),
        "steps_completed": int(res.get("steps_completed", -1)),
        "solver_failure": str(res.get("solver_failure", "")),
        "K_eff_x": float(res.get("K_eff_x", float("nan"))),
        "K_ref_x": float(ref_result.get("K_eff_x", float("nan"))),
        "t_part_s": float(meta["t_part_s"]),
        "t_mom_s": float(meta["t_mom_s"]),
        "t_flow_s": float(res.get("elapsed_s", float("nan"))),
        "t_total_s": float(t_total),
        "t_ref_total_s": float(ref_total),
        "S_pipe": float(ref_total / t_total) if math.isfinite(ref_total) and t_total > 0 else float("nan"),
        "label_mode": meta["label_mode"],
        "distance_connectivity": int(meta["distance_connectivity"]),
    }
    return row, geom, res


def pvfv_write_csv(path: Path, rows: List[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    keys = []
    for r in rows:
        for k in r.keys():
            if k not in keys:
                keys.append(k)
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)

# -----------------------------------------------------------------------------
# Suite blocks
# -----------------------------------------------------------------------------

def pvfv_build_reference(case_name: str, mask: cp.ndarray, cfg: PB615Config) -> Tuple[GPUFVMGeometry, Dict[str, Any], dict]:
    """Voxel-grid reference: stride=(1,1,1)."""
    print(f"[flow] building voxel reference for {case_name}")
    ref_geom, ref_meta = pvfv_build_geometry_timed(mask, (1, 1, 1), cfg, split_face_components=False)
    ref_res = run_velocity_pressure_projection_gpu(ref_geom, cfg, run_label=f"pvfv_ref_{case_name}")
    ref_total = ref_meta["t_part_s"] + ref_meta["t_mom_s"] + float(ref_res.get("elapsed_s", 0.0))
    ref_res["pvfv_total_time_s"] = ref_total
    ref_row = {
        "case": case_name,
        "N_fl": int(ref_meta["N_fl"]),
        "N_ref_cells": int(ref_geom.n_cells),
        "K_ref_x": float(ref_res.get("K_eff_x", float("nan"))),
        "mass_inf_per_volume": float(ref_res.get("mass_inf_per_volume", float("nan"))),
        "steady_momentum_inf": float(ref_res.get("steady_momentum_inf", float("nan"))),
        "steady_converged": bool(ref_res.get("steady_converged", False)),
        "steps_completed": int(ref_res.get("steps_completed", -1)),
        "t_ref_part_s": float(ref_meta["t_part_s"]),
        "t_ref_mom_s": float(ref_meta["t_mom_s"]),
        "t_ref_flow_s": float(ref_res.get("elapsed_s", float("nan"))),
        "t_ref_total_s": float(ref_total),
    }
    return ref_geom, ref_res, ref_row


def pvfv_run_wall_calibration(cfg: PB615Config, plan: dict, out: Path) -> dict:
    """Fixed-seed duct wall beta sweep.  Returns selected beta and rows."""
    shape = plan["shape_zyx"]
    fixed_stride = plan["fixed_stride"]
    mask = pvfv_make_mask_gpu("orthogonal_duct", shape)
    ref_geom, ref_res, ref_row = pvfv_build_reference("orthogonal_duct", mask, cfg)
    rows = []
    for beta in plan["beta_values"]:
        row, _, _ = pvfv_run_flow_case("orthogonal_duct", mask, fixed_stride, cfg, ref_geom, ref_res,
                                        wall_beta=float(beta), face_mode="tpfa_fallback",
                                        run_tag=f"beta_{beta:g}")
        row["stage"] = "wall_beta_sweep"
        rows.append(row)
    # Select by permeability first, velocity second.
    valid = [r for r in rows if math.isfinite(float(r["e_K"]))]
    best = min(valid, key=lambda r: float(r["e_K"]) + 0.2 * float(r.get("e_u", 0.0))) if valid else rows[0]
    summary = {
        "beta_star": float(best["wall_beta"]),
        "best_e_K": float(best["e_K"]),
        "best_e_u": float(best["e_u"]),
        "fixed_stride": str(tuple(fixed_stride)),
        "case": "orthogonal_duct",
    }
    pvfv_write_csv(out / "pvfv_beta_sweep.csv", rows)
    pvfv_write_csv(out / "pvfv_reference_rows.csv", [ref_row])
    (out / "pvfv_wall_calibration_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("[flow] wall calibration summary:", summary)
    return {"beta_star": float(best["wall_beta"]), "rows": rows, "ref_geom": ref_geom, "ref_res": ref_res, "ref_row": ref_row}


def pvfv_run_canonical_and_abc(cfg: PB615Config, plan: dict, beta_star: float, out: Path) -> dict:
    """Run reference + coarse sweeps for canonical and ABC cases."""
    shape = plan["shape_zyx"]
    strides = list(plan["seed_strides"])
    cases = ["orthogonal_duct", "skewed_duct", "A_thin_wall", "B_narrow_throat", "C_maze"]
    all_ref_rows, all_rows = [], []
    ref_cache = {}
    for case in cases:
        print(f"\n[flow] ===== case {case} =====")
        mask = pvfv_make_mask_gpu(case, shape)
        ref_geom, ref_res, ref_row = pvfv_build_reference(case, mask, cfg)
        all_ref_rows.append(ref_row)
        ref_cache[case] = (mask, ref_geom, ref_res)
        for stride in strides:
            row, _, _ = pvfv_run_flow_case(case, mask, stride, cfg, ref_geom, ref_res,
                                            wall_beta=float(beta_star), face_mode="overrelaxed_default",
                                            run_tag="baseline")
            row["stage"] = "canonical_or_abc_baseline"
            all_rows.append(row)
    pvfv_write_csv(out / "pvfv_reference_rows_all.csv", all_ref_rows)
    pvfv_write_csv(out / "pvfv_flow_sweep_abc.csv", all_rows)
    return {"rows": all_rows, "ref_rows": all_ref_rows, "ref_cache": ref_cache}


def pvfv_run_component_study(cfg: PB615Config, plan: dict, beta_star: float, ref_cache: dict, out: Path) -> dict:
    """Compact component study on one ABC case."""
    case = plan.get("component_case", "C_maze")
    stride = tuple(plan.get("component_stride", plan["fixed_stride"]))
    if case not in ref_cache:
        mask = pvfv_make_mask_gpu(case, plan["shape_zyx"])
        ref_geom, ref_res, _ = pvfv_build_reference(case, mask, cfg)
    else:
        mask, ref_geom, ref_res = ref_cache[case]
    variants = [
        {"variant": "proposed", "face_mode": "overrelaxed_default", "wall_beta": beta_star, "noc_alpha": 0.0, "reconstruct": False, "status": "assembled"},
        {"variant": "tpfa_only", "face_mode": "tpfa_fallback", "wall_beta": beta_star, "noc_alpha": 0.0, "reconstruct": False, "status": "assembled"},
        {"variant": "raw_overrelaxed", "face_mode": "raw_overrelaxed", "wall_beta": beta_star, "noc_alpha": 0.0, "reconstruct": False, "status": "assembled"},
        {"variant": "no_wall_closure", "face_mode": "overrelaxed_default", "wall_beta": 0.0, "noc_alpha": 0.0, "reconstruct": False, "status": "assembled"},
        {"variant": "noc_alpha_0p5", "face_mode": "overrelaxed_default", "wall_beta": beta_star, "noc_alpha": 0.5, "reconstruct": False, "status": "assembled"},
        {"variant": "flux_reconstruction", "face_mode": "overrelaxed_default", "wall_beta": beta_star, "noc_alpha": 0.0, "reconstruct": True, "status": "assembled"},
    ]
    rows = []
    for v in variants:
        row, _, _ = pvfv_run_flow_case(case, mask, stride, cfg, ref_geom, ref_res,
                                        wall_beta=float(v["wall_beta"]),
                                        face_mode=str(v["face_mode"]),
                                        noc_alpha=float(v["noc_alpha"]),
                                        reconstruct=bool(v["reconstruct"]),
                                        run_tag=f"component_{v['variant']}")
        row["variant"] = v["variant"]
        row["status"] = v["status"]
        row["stage"] = "component_study"
        rows.append(row)
    # Add topology-only invalid variant line for no-positive-area filter.
    rows.append({
        "case": case,
        "variant": "no_positive_area_filter",
        "status": "topology_only_not_assembled",
        "mechanism": "Flux-interface admissibility; zero-area contacts are not assembled as FV faces.",
        "e_K": float("nan"), "e_u": float("nan"), "e_phi": float("nan"),
    })
    pvfv_write_csv(out / "pvfv_component_study.csv", rows)
    return {"rows": rows}


def pvfv_make_final_tables(out: Path) -> dict:
    """Combine CSV outputs into manuscript-facing tables."""
    if pd is None:
        print("pandas not available; skipping combined tables")
        return {}
    flow_path = out / "pvfv_flow_sweep_abc.csv"
    beta_path = out / "pvfv_beta_sweep.csv"
    comp_path = out / "pvfv_component_study.csv"
    outputs = {}
    if flow_path.exists():
        df = pd.read_csv(flow_path)
        # Best/highest-compression row per case by score: keep one representative per case for Table 5.
        df["score"] = df["e_K"].astype(float) + 0.25 * df["e_phi"].astype(float).fillna(0.0) + 0.1 * df["e_u"].astype(float).fillna(0.0)
        idx = df.groupby("case")["score"].idxmin()
        final = df.loc[idx].sort_values("case")
        final.to_csv(out / "table5_final_accuracy_cost_summary.csv", index=False)
        outputs["table5"] = str(out / "table5_final_accuracy_cost_summary.csv")
    if comp_path.exists():
        comp = pd.read_csv(comp_path)
        comp.to_csv(out / "table6_component_study_summary.csv", index=False)
        outputs["table6"] = str(out / "table6_component_study_summary.csv")
    if beta_path.exists():
        beta = pd.read_csv(beta_path)
        beta.to_csv(out / "fig5_beta_sweep_data.csv", index=False)
        outputs["beta"] = str(out / "fig5_beta_sweep_data.csv")
    return outputs


def pvfv_make_quick_plots(out: Path) -> None:
    if pd is None or plt is None:
        print("pandas/matplotlib not available; skipping plots")
        return
    figdir = out / "quick_figures"
    figdir.mkdir(parents=True, exist_ok=True)
    flow_path = out / "pvfv_flow_sweep_abc.csv"
    beta_path = out / "pvfv_beta_sweep.csv"
    comp_path = out / "pvfv_component_study.csv"
    if beta_path.exists():
        df = pd.read_csv(beta_path)
        plt.figure()
        plt.plot(df["wall_beta"], df["e_K"], marker="o")
        plt.xlabel(r"$\\beta_w$")
        plt.ylabel(r"$e_K$")
        plt.title("Fixed-seed duct wall-closure sweep")
        plt.tight_layout()
        plt.savefig(figdir / "fig5_beta_sweep_quick.png", dpi=200)
        plt.close()
    if flow_path.exists():
        df = pd.read_csv(flow_path)
        for metric, fname in [("e_K", "fig6_eK_vs_compression_quick.png"), ("e_phi", "fig6_ephi_vs_compression_quick.png"), ("S_pipe", "fig6_speedup_vs_compression_quick.png")]:
            plt.figure()
            for case, g in df.groupby("case"):
                plt.plot(g["C_comp"], g[metric], marker="o", label=case)
            plt.xscale("log")
            if metric.startswith("e_"):
                plt.yscale("log")
            plt.xlabel(r"$C_{comp}=N_{fl}/N_{cv}$")
            plt.ylabel(metric)
            plt.legend(fontsize=7)
            plt.tight_layout()
            plt.savefig(figdir / fname, dpi=200)
            plt.close()
    if comp_path.exists():
        df = pd.read_csv(comp_path)
        df = df[df.get("status", "assembled") == "assembled"] if "status" in df else df
        if len(df):
            plt.figure(figsize=(7, 3))
            plt.bar(df["variant"].astype(str), df["e_K"].astype(float))
            plt.ylabel(r"$e_K$")
            plt.xticks(rotation=35, ha="right")
            plt.tight_layout()
            plt.savefig(figdir / "fig7_component_eK_quick.png", dpi=200)
            plt.close()


def pvfv_run_complete_abc_suite(profile: str = "pilot", zip_outputs: bool = True) -> dict:
    """Run the complete current simulation suite for canonical + ABC cases.

    Outputs are written to PVFV_OUT_DIR and aligned with manuscript needs:
      - pvfv_beta_sweep.csv
      - pvfv_flow_sweep_abc.csv
      - pvfv_component_study.csv
      - table5_final_accuracy_cost_summary.csv
      - table6_component_study_summary.csv
      - quick_figures/*.png
      - manifest JSON and optional ZIP archive
    """
    require_cuda_gpu()
    out = Path(PVFV_OUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    cfg = pvfv_base_cfg(out_dir=str(out), profile=profile)
    plan = pvfv_case_plan(profile)
    manifest = {
        "suite": "PoreVoronoi-FV ABC numerical suite",
        "profile": profile,
        "out_dir": str(out),
        "case_plan": plan,
        "cfg": {k: str(getattr(cfg, k)) for k in PB615Config.__dataclass_fields__.keys()},
        "labeler_scope_note": "Current notebook uses exact GPU geodesic relaxation as the label source for flow simulations. ROI--JFA can replace this through the same labels/distances contract for partition-performance claims.",
    }
    (out / "pvfv_suite_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    wall = pvfv_run_wall_calibration(cfg, plan, out)
    beta_star = float(wall["beta_star"])
    sweeps = pvfv_run_canonical_and_abc(cfg, plan, beta_star, out)
    comp = pvfv_run_component_study(cfg, plan, beta_star, sweeps["ref_cache"], out)
    tables = pvfv_make_final_tables(out)
    pvfv_make_quick_plots(out)

    zip_path = None
    if zip_outputs:
        zip_path = shutil.make_archive(str(out), "zip", out)
        print(f"[flow] zipped outputs: {zip_path}")
    summary = {
        "out_dir": str(out),
        "zip_path": zip_path,
        "beta_star": beta_star,
        "n_flow_rows": len(sweeps["rows"]),
        "n_component_rows": len(comp["rows"]),
        "tables": tables,
    }
    (out / "pvfv_suite_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("[flow] suite summary:", summary)
    return summary

print("ABC suite definitions loaded. Run `summary = pvfv_run_complete_abc_suite(profile='pilot')`.")


## Full data-export overrides

This cell overrides the suite functions so that running the final cell exports the raw flow-side data products: per-run profiles, velocity slices, mass residual histograms, face-flux comparisons, coarse arrays, reference arrays, panel CSVs, and a data-products manifest.

In [ ]:
# ============================================================
# Full-data export overrides for the ABC suite
# ============================================================
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import csv, json, math, shutil, time
import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


def pvfv_safe_tag(*parts: Any) -> str:
    """Return a filesystem-safe tag for data products."""
    raw = "__".join(str(p) for p in parts if p is not None and str(p) != "")
    out = []
    for ch in raw:
        if ch.isalnum() or ch in ("-", "_", "."):
            out.append(ch)
        else:
            out.append("_")
    return "".join(out).strip("_") or "run"


def pvfv_ensure_dirs(out: Path) -> Dict[str, Path]:
    dirs = {
        "fig5": out / "fig_data" / "fig5_canonical_wall",
        "fig6": out / "fig_data" / "fig6_synthetic_pareto",
        "fig7": out / "fig_data" / "fig7_component_study",
        "flow": out / "flow_run_data",
        "reference": out / "reference_data",
        "quick": out / "quick_figures",
    }
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)
    return dirs


def pvfv_to_numpy(a: Any) -> np.ndarray:
    """Convert CuPy/NumPy arrays to NumPy without assuming the caller's type."""
    try:
        return cp.asnumpy(a)
    except Exception:
        return np.asarray(a)


def pvfv_cell_vector_to_voxel_np(geom: GPUFVMGeometry, U: Any) -> np.ndarray:
    """Map a cell vector field [n,3] to a voxel field [D,H,W,3] using geom.labels."""
    labels = pvfv_to_numpy(geom.labels).astype(np.int64, copy=False)
    mask = pvfv_to_numpy(geom.mask).astype(bool, copy=False)
    U_np = pvfv_to_numpy(U).astype(float, copy=False)
    out = np.full(labels.shape + (U_np.shape[1],), np.nan, dtype=float)
    valid = mask & (labels >= 0) & (labels < U_np.shape[0])
    out[valid] = U_np[labels[valid]]
    return out


def pvfv_cell_scalar_to_voxel_np(geom: GPUFVMGeometry, a: Any) -> np.ndarray:
    """Map a cell scalar field [n] to a voxel field [D,H,W]."""
    labels = pvfv_to_numpy(geom.labels).astype(np.int64, copy=False)
    mask = pvfv_to_numpy(geom.mask).astype(bool, copy=False)
    arr = pvfv_to_numpy(a).astype(float, copy=False)
    out = np.full(labels.shape, np.nan, dtype=float)
    valid = mask & (labels >= 0) & (labels < arr.shape[0])
    out[valid] = arr[labels[valid]]
    return out


def pvfv_best_slice_z(mask_np: np.ndarray) -> int:
    counts = mask_np.reshape(mask_np.shape[0], -1).sum(axis=1)
    if np.all(counts == 0):
        return int(mask_np.shape[0] // 2)
    return int(np.argmax(counts))


def pvfv_profile_y_rows(mask_np: np.ndarray, ref_Uvox: np.ndarray, coarse_Uvox: np.ndarray) -> List[dict]:
    """Return y-profile rows averaging over z and x for Ux."""
    rows: List[dict] = []
    D, H, W = mask_np.shape
    for y in range(H):
        m = mask_np[:, y, :]
        ref = ref_Uvox[:, y, :, 0]
        crs = coarse_Uvox[:, y, :, 0]
        if np.any(m):
            ref_vals = ref[m]
            crs_vals = crs[m]
            err_vals = crs_vals - ref_vals
            rows.append({
                "y_index": int(y),
                "n_voxels": int(np.sum(m)),
                "Ux_ref_mean": float(np.nanmean(ref_vals)),
                "Ux_coarse_mean": float(np.nanmean(crs_vals)),
                "Ux_error_mean": float(np.nanmean(err_vals)),
                "Ux_abs_error_mean": float(np.nanmean(np.abs(err_vals))),
            })
        else:
            rows.append({"y_index": int(y), "n_voxels": 0, "Ux_ref_mean": float("nan"),
                         "Ux_coarse_mean": float("nan"), "Ux_error_mean": float("nan"),
                         "Ux_abs_error_mean": float("nan")})
    return rows


def pvfv_write_csv(path: Path, rows: List[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    keys: List[str] = []
    for r in rows:
        for k in r.keys():
            if k not in keys:
                keys.append(k)
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow(r)


def pvfv_save_reference_products(out: Path, case_name: str, ref_geom: GPUFVMGeometry, ref_res: Dict[str, Any], ref_row: dict) -> str:
    """Save full voxel-reference arrays needed for later comparison."""
    dirs = pvfv_ensure_dirs(out)
    tag = pvfv_safe_tag(case_name, "reference")
    path = dirs["reference"] / f"{tag}.npz"
    np.savez_compressed(
        path,
        case=str(case_name),
        mask=pvfv_to_numpy(ref_geom.mask).astype(np.bool_),
        labels=pvfv_to_numpy(ref_geom.labels).astype(np.int32),
        owner=pvfv_to_numpy(ref_geom.owner).astype(np.int32),
        neigh=pvfv_to_numpy(ref_geom.neigh).astype(np.int32),
        area=pvfv_to_numpy(ref_geom.area).astype(float),
        avec=pvfv_to_numpy(ref_geom.avec).astype(float),
        face_centroid=pvfv_to_numpy(ref_geom.face_centroid).astype(float),
        dvec=pvfv_to_numpy(ref_geom.dvec).astype(float),
        tproj=pvfv_to_numpy(ref_geom.tproj).astype(float),
        twall=pvfv_to_numpy(ref_geom.twall).astype(float),
        volume=pvfv_to_numpy(ref_geom.volume).astype(float),
        centroid=pvfv_to_numpy(ref_geom.centroid).astype(float),
        U=pvfv_to_numpy(ref_res["U"]).astype(float),
        p=pvfv_to_numpy(ref_res["p"]).astype(float),
        phi=pvfv_to_numpy(ref_res["phi"]).astype(float),
        div=pvfv_to_numpy(ref_res["div"]).astype(float),
        K_eff_x=float(ref_res.get("K_eff_x", float("nan"))),
        metadata=json.dumps(ref_row),
    )
    return str(path)


def pvfv_save_flow_products(out: Path, case_name: str, stride: Tuple[int, int, int], run_tag: str,
                             ref_geom: GPUFVMGeometry, ref_res: Dict[str, Any],
                             geom: GPUFVMGeometry, res: Dict[str, Any], row: dict,
                             *, panel_hint: str = "flow") -> Dict[str, str]:
    """Save all raw arrays needed for Fig. 5/6/7 panels and Table 5/6 rows."""
    dirs = pvfv_ensure_dirs(out)
    tag = pvfv_safe_tag(case_name, run_tag, "stride", "x".join(map(str, stride)))
    flow_dir = dirs["flow"] / tag
    flow_dir.mkdir(parents=True, exist_ok=True)

    mask_np = pvfv_to_numpy(geom.mask).astype(np.bool_)
    labels_np = pvfv_to_numpy(geom.labels).astype(np.int32)
    ref_Uvox = pvfv_cell_vector_to_voxel_np(ref_geom, ref_res["U"])
    coarse_Uvox = pvfv_cell_vector_to_voxel_np(geom, res["U"])
    err_Uvox = coarse_Uvox - ref_Uvox
    z_idx = pvfv_best_slice_z(mask_np)

    # 1) Velocity profile data for Fig. 5(a), and general profile diagnostics.
    profile_path = flow_dir / "velocity_profile_y.csv"
    pvfv_write_csv(profile_path, pvfv_profile_y_rows(mask_np, ref_Uvox, coarse_Uvox))

    # 2) Mid-slice arrays for Fig. 6(e) and real/diagnostic panels.
    slice_path = flow_dir / "velocity_slice_midz.npz"
    np.savez_compressed(
        slice_path,
        case=str(case_name),
        stride_zyx=np.asarray(stride, dtype=np.int32),
        run_tag=str(run_tag),
        z_index=np.asarray(z_idx, dtype=np.int32),
        mask_slice=mask_np[z_idx],
        label_slice=labels_np[z_idx],
        Ux_ref_slice=ref_Uvox[z_idx, :, :, 0],
        Ux_coarse_slice=coarse_Uvox[z_idx, :, :, 0],
        Ux_error_slice=err_Uvox[z_idx, :, :, 0],
        Uy_ref_slice=ref_Uvox[z_idx, :, :, 1],
        Uy_coarse_slice=coarse_Uvox[z_idx, :, :, 1],
        Uz_ref_slice=ref_Uvox[z_idx, :, :, 2],
        Uz_coarse_slice=coarse_Uvox[z_idx, :, :, 2],
    )

    # 3) Mass-residual raw values and histograms for Fig. 5(c) and Table 5.
    div = pvfv_to_numpy(res["div"]).astype(float)
    volume = pvfv_to_numpy(geom.volume).astype(float)
    r_vol = div / np.maximum(volume, 1.0e-300)
    counts, bins = np.histogram(r_vol, bins=80)
    log_abs = np.log10(np.maximum(np.abs(r_vol), 1.0e-300))
    counts_log, bins_log = np.histogram(log_abs, bins=80)
    mass_path = flow_dir / "mass_residual_histogram.npz"
    np.savez_compressed(
        mass_path,
        div=div,
        volume=volume,
        residual_per_volume=r_vol,
        hist_counts=counts,
        hist_bins=bins,
        log10_abs_residual=log_abs,
        log_hist_counts=counts_log,
        log_hist_bins=bins_log,
        mass_inf_per_volume=float(row.get("mass_inf_per_volume", np.nan)),
        mass_l2_per_volume=float(row.get("mass_l2_per_volume", np.nan)),
    )

    # 4) Flux data for e_phi and FV-specific comparisons.
    flux_path = flow_dir / "face_flux_comparison.npz"
    try:
        phi_ref = pvfv_phi_ref_on_coarse_faces_cpu(ref_geom, ref_res, geom)
    except Exception as exc:
        print(f"[flow-data] flux reference aggregation failed for {tag}: {exc}")
        phi_ref = np.full(int(geom.owner.size), np.nan, dtype=float)
    np.savez_compressed(
        flux_path,
        owner=pvfv_to_numpy(geom.owner).astype(np.int32),
        neigh=pvfv_to_numpy(geom.neigh).astype(np.int32),
        area=pvfv_to_numpy(geom.area).astype(float),
        avec=pvfv_to_numpy(geom.avec).astype(float),
        face_centroid=pvfv_to_numpy(geom.face_centroid).astype(float),
        dvec=pvfv_to_numpy(geom.dvec).astype(float),
        tproj=pvfv_to_numpy(geom.tproj).astype(float),
        phi=pvfv_to_numpy(res["phi"]).astype(float),
        phi_ref=phi_ref.astype(float),
    )

    # 5) Full compact run object for archiving and reproduction of tables.
    run_path = flow_dir / "coarse_run_arrays.npz"
    np.savez_compressed(
        run_path,
        case=str(case_name),
        stride_zyx=np.asarray(stride, dtype=np.int32),
        run_tag=str(run_tag),
        mask=mask_np,
        labels=labels_np,
        dist=pvfv_to_numpy(geom.dist).astype(float),
        volume=pvfv_to_numpy(geom.volume).astype(float),
        centroid=pvfv_to_numpy(geom.centroid).astype(float),
        owner=pvfv_to_numpy(geom.owner).astype(np.int32),
        neigh=pvfv_to_numpy(geom.neigh).astype(np.int32),
        area=pvfv_to_numpy(geom.area).astype(float),
        avec=pvfv_to_numpy(geom.avec).astype(float),
        face_centroid=pvfv_to_numpy(geom.face_centroid).astype(float),
        dvec=pvfv_to_numpy(geom.dvec).astype(float),
        tproj=pvfv_to_numpy(geom.tproj).astype(float),
        twall=pvfv_to_numpy(geom.twall).astype(float),
        U=pvfv_to_numpy(res["U"]).astype(float),
        p=pvfv_to_numpy(res["p"]).astype(float),
        phi=pvfv_to_numpy(res["phi"]).astype(float),
        div=div,
        row_metadata=json.dumps(row),
    )

    # 6) Human-readable row-level manifest.
    manifest = {
        "case": str(case_name),
        "stride_zyx": list(map(int, stride)),
        "run_tag": str(run_tag),
        "panel_hint": str(panel_hint),
        "profile_csv": str(profile_path),
        "slice_npz": str(slice_path),
        "mass_residual_npz": str(mass_path),
        "flux_npz": str(flux_path),
        "run_arrays_npz": str(run_path),
    }
    manifest_path = flow_dir / "run_data_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


def pvfv_run_flow_case(case_name: str, mask: cp.ndarray, stride: Tuple[int, int, int], cfg: PB615Config,
                        ref_geom: GPUFVMGeometry, ref_result: Dict[str, Any],
                        *, wall_beta: float = 1.0,
                        face_mode: str = "overrelaxed_default",
                        noc_alpha: float = 0.0,
                        reconstruct: bool = False,
                        split_face_components: bool = True,
                        run_tag: str = "coarse",
                        out: Optional[Path] = None,
                        export_data: bool = False,
                        panel_hint: str = "flow") -> Tuple[dict, GPUFVMGeometry, Dict[str, Any]]:
    """Build and solve one coarse case; optionally export all figure/table arrays."""
    local = pvfv_clone_cfg(cfg,
                            transmissibility_ratio_clip=_pb618_face_clip_for_mode(face_mode),
                            explicit_nonorthogonal_correction=float(noc_alpha),
                            velocity_reconstruct_from_flux=bool(reconstruct))
    geom, meta = pvfv_build_geometry_timed(mask, stride, local, split_face_components=split_face_components)
    if abs(float(wall_beta) - 1.0) > 1.0e-15:
        geom = _pb618_clone_geometry_with_scaled_twall(geom, float(wall_beta))
    res = run_velocity_pressure_projection_gpu(geom, local, run_label=f"pvfv_{case_name}_{run_tag}_{stride}")
    Uref_c, pref_c = coarsen_voxel_reference_to_coarse_gpu(ref_geom, ref_result, geom)
    e_u = pvfv_weighted_rel_l2_vec_gpu(res["U"], Uref_c, geom.volume)
    e_k = abs(float(res["K_eff_x"]) - float(ref_result["K_eff_x"])) / max(abs(float(ref_result["K_eff_x"])), 1.0e-300)
    try:
        e_phi = pvfv_flux_rel_error(ref_geom, ref_result, geom, res)
    except Exception as exc:
        print(f"[flow] flux error failed for {case_name}/{stride}: {exc}")
        e_phi = float("nan")
    ref_total = float(ref_result.get("pvfv_total_time_s", ref_result.get("elapsed_s", float("nan"))))
    t_total = float(meta["t_part_s"] + meta["t_mom_s"] + float(res.get("elapsed_s", 0.0)))
    row = {
        "case": case_name,
        "run_tag": run_tag,
        "stride_zyx": str(tuple(stride)),
        "S": int(meta["S"]),
        "N_cv": int(meta["N_cv"]),
        "N_fl": int(meta["N_fl"]),
        "C_comp": float(meta["C_comp"]),
        "N_split": int(meta["N_split"]),
        "N_0A": int(meta["N_0A"]),
        "face_connected_fraction": float(meta["face_connected_fraction"]),
        "face_mode": str(face_mode),
        "wall_beta": float(wall_beta),
        "wall_distance_floor": float(local.wall_distance_floor),
        "noc_alpha": float(noc_alpha),
        "reconstruct": bool(reconstruct),
        "e_K": float(e_k),
        "e_u": float(e_u),
        "e_phi": float(e_phi),
        "mass_inf_per_volume": float(res.get("mass_inf_per_volume", float("nan"))),
        "mass_l2_per_volume": float(res.get("mass_l2_per_volume", float("nan"))),
        "steady_momentum_inf": float(res.get("steady_momentum_inf", float("nan"))),
        "steady_converged": bool(res.get("steady_converged", False)),
        "steps_completed": int(res.get("steps_completed", -1)),
        "solver_failure": str(res.get("solver_failure", "")),
        "K_eff_x": float(res.get("K_eff_x", float("nan"))),
        "K_ref_x": float(ref_result.get("K_eff_x", float("nan"))),
        "t_part_s": float(meta["t_part_s"]),
        "t_mom_s": float(meta["t_mom_s"]),
        "t_flow_s": float(res.get("elapsed_s", float("nan"))),
        "t_total_s": float(t_total),
        "t_ref_total_s": float(ref_total),
        "S_pipe": float(ref_total / t_total) if math.isfinite(ref_total) and t_total > 0 else float("nan"),
        "label_mode": meta["label_mode"],
        "distance_connectivity": int(meta["distance_connectivity"]),
        "data_manifest": "",
        "profile_csv": "",
        "slice_npz": "",
        "mass_residual_npz": "",
        "flux_npz": "",
        "run_arrays_npz": "",
    }
    if export_data and out is not None:
        products = pvfv_save_flow_products(Path(out), case_name, tuple(stride), run_tag, ref_geom, ref_result, geom, res, row, panel_hint=panel_hint)
        row.update(products)
        row["data_manifest"] = str(Path(products["run_arrays_npz"]).parent / "run_data_manifest.json")
    return row, geom, res


def pvfv_build_reference(case_name: str, mask: cp.ndarray, cfg: PB615Config, out: Optional[Path] = None) -> Tuple[GPUFVMGeometry, Dict[str, Any], dict]:
    """Voxel-grid reference: stride=(1,1,1), with optional full data export."""
    print(f"[flow] building voxel reference for {case_name}")
    ref_geom, ref_meta = pvfv_build_geometry_timed(mask, (1, 1, 1), cfg, split_face_components=False)
    ref_res = run_velocity_pressure_projection_gpu(ref_geom, cfg, run_label=f"pvfv_ref_{case_name}")
    ref_total = ref_meta["t_part_s"] + ref_meta["t_mom_s"] + float(ref_res.get("elapsed_s", 0.0))
    ref_res["pvfv_total_time_s"] = ref_total
    ref_row = {
        "case": case_name,
        "N_fl": int(ref_meta["N_fl"]),
        "N_ref_cells": int(ref_geom.n_cells),
        "K_ref_x": float(ref_res.get("K_eff_x", float("nan"))),
        "mass_inf_per_volume": float(ref_res.get("mass_inf_per_volume", float("nan"))),
        "steady_momentum_inf": float(ref_res.get("steady_momentum_inf", float("nan"))),
        "steady_converged": bool(ref_res.get("steady_converged", False)),
        "steps_completed": int(ref_res.get("steps_completed", -1)),
        "t_ref_part_s": float(ref_meta["t_part_s"]),
        "t_ref_mom_s": float(ref_meta["t_mom_s"]),
        "t_ref_flow_s": float(ref_res.get("elapsed_s", float("nan"))),
        "t_ref_total_s": float(ref_total),
        "reference_npz": "",
    }
    if out is not None:
        ref_row["reference_npz"] = pvfv_save_reference_products(Path(out), case_name, ref_geom, ref_res, ref_row)
    return ref_geom, ref_res, ref_row


def pvfv_run_wall_calibration(cfg: PB615Config, plan: dict, out: Path) -> dict:
    """Fixed-seed duct wall beta sweep; exports all data needed for Fig. 5."""
    shape = plan["shape_zyx"]
    fixed_stride = tuple(plan["fixed_stride"])
    mask = pvfv_make_mask_gpu("orthogonal_duct", shape)
    ref_geom, ref_res, ref_row = pvfv_build_reference("orthogonal_duct", mask, cfg, out=out)
    rows: List[dict] = []
    for beta in plan["beta_values"]:
        row, _, _ = pvfv_run_flow_case(
            "orthogonal_duct", mask, fixed_stride, cfg, ref_geom, ref_res,
            wall_beta=float(beta), face_mode="tpfa_fallback",
            run_tag=f"beta_{beta:g}", out=out, export_data=True,
            panel_hint="Fig5_wall_beta_sweep"
        )
        row["stage"] = "wall_beta_sweep"
        rows.append(row)
    valid = [r for r in rows if math.isfinite(float(r["e_K"]))]
    best = min(valid, key=lambda r: float(r["e_K"]) + 0.2 * float(r.get("e_u", 0.0))) if valid else rows[0]
    summary = {
        "beta_star": float(best["wall_beta"]),
        "best_e_K": float(best["e_K"]),
        "best_e_u": float(best["e_u"]),
        "fixed_stride": str(tuple(fixed_stride)),
        "case": "orthogonal_duct",
        "best_profile_csv": best.get("profile_csv", ""),
        "best_mass_residual_npz": best.get("mass_residual_npz", ""),
        "best_slice_npz": best.get("slice_npz", ""),
    }
    pvfv_write_csv(out / "pvfv_beta_sweep.csv", rows)
    pvfv_write_csv(out / "pvfv_reference_rows.csv", [ref_row])
    # Dedicated Fig. 5 panel data products.
    pvfv_write_csv(out / "fig_data" / "fig5_canonical_wall" / "fig5_beta_sweep_data.csv", rows)
    (out / "pvfv_wall_calibration_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("[flow] wall calibration summary:", summary)
    return {"beta_star": float(best["wall_beta"]), "rows": rows, "ref_geom": ref_geom, "ref_res": ref_res, "ref_row": ref_row, "summary": summary}


def pvfv_run_canonical_and_abc(cfg: PB615Config, plan: dict, beta_star: float, out: Path) -> dict:
    """Run reference + coarse sweeps for canonical and ABC cases; export all field products."""
    shape = plan["shape_zyx"]
    strides = list(plan["seed_strides"])
    cases = ["orthogonal_duct", "skewed_duct", "A_thin_wall", "B_narrow_throat", "C_maze"]
    all_ref_rows: List[dict] = []
    all_rows: List[dict] = []
    ref_cache: Dict[str, Tuple[Any, Any, Any]] = {}
    for case in cases:
        print(f"\n[flow] ===== case {case} =====")
        mask = pvfv_make_mask_gpu(case, shape)
        ref_geom, ref_res, ref_row = pvfv_build_reference(case, mask, cfg, out=out)
        all_ref_rows.append(ref_row)
        ref_cache[case] = (mask, ref_geom, ref_res)
        for stride in strides:
            hint = "Fig5_refinement_or_transfer" if case in ("orthogonal_duct", "skewed_duct") else "Fig6_synthetic_pareto"
            row, _, _ = pvfv_run_flow_case(
                case, mask, tuple(stride), cfg, ref_geom, ref_res,
                wall_beta=float(beta_star), face_mode="overrelaxed_default",
                run_tag="baseline", out=out, export_data=True, panel_hint=hint
            )
            row["stage"] = "canonical_or_abc_baseline"
            all_rows.append(row)
    pvfv_write_csv(out / "pvfv_reference_rows_all.csv", all_ref_rows)
    pvfv_write_csv(out / "pvfv_flow_sweep_abc.csv", all_rows)
    # Panel-specific CSVs for figures.
    pvfv_write_csv(out / "fig_data" / "fig5_canonical_wall" / "fig5_refinement_transfer_data.csv",
                    [r for r in all_rows if r["case"] in ("orthogonal_duct", "skewed_duct")])
    pvfv_write_csv(out / "fig_data" / "fig6_synthetic_pareto" / "fig6_pareto_data.csv",
                    [r for r in all_rows if r["case"] in ("A_thin_wall", "B_narrow_throat", "C_maze")])
    return {"rows": all_rows, "ref_rows": all_ref_rows, "ref_cache": ref_cache}


def pvfv_run_component_study(cfg: PB615Config, plan: dict, beta_star: float, ref_cache: dict, out: Path) -> dict:
    """Compact FV-side component study with full data exports for Fig. 7/Table 6."""
    case = plan.get("component_case", "C_maze")
    stride = tuple(plan.get("component_stride", plan["fixed_stride"]))
    if case not in ref_cache:
        mask = pvfv_make_mask_gpu(case, plan["shape_zyx"])
        ref_geom, ref_res, _ = pvfv_build_reference(case, mask, cfg, out=out)
    else:
        mask, ref_geom, ref_res = ref_cache[case]
    variants = [
        {"variant": "proposed", "face_mode": "overrelaxed_default", "wall_beta": beta_star, "noc_alpha": 0.0, "reconstruct": False, "status": "assembled"},
        {"variant": "tpfa_only", "face_mode": "tpfa_fallback", "wall_beta": beta_star, "noc_alpha": 0.0, "reconstruct": False, "status": "assembled"},
        {"variant": "raw_overrelaxed", "face_mode": "raw_overrelaxed", "wall_beta": beta_star, "noc_alpha": 0.0, "reconstruct": False, "status": "assembled"},
        {"variant": "no_wall_closure", "face_mode": "overrelaxed_default", "wall_beta": 0.0, "noc_alpha": 0.0, "reconstruct": False, "status": "assembled"},
        {"variant": "noc_alpha_0p5", "face_mode": "overrelaxed_default", "wall_beta": beta_star, "noc_alpha": 0.5, "reconstruct": False, "status": "assembled"},
        {"variant": "flux_reconstruction", "face_mode": "overrelaxed_default", "wall_beta": beta_star, "noc_alpha": 0.0, "reconstruct": True, "status": "assembled"},
    ]
    rows: List[dict] = []
    for v in variants:
        row, _, _ = pvfv_run_flow_case(
            case, mask, stride, cfg, ref_geom, ref_res,
            wall_beta=float(v["wall_beta"]),
            face_mode=str(v["face_mode"]),
            noc_alpha=float(v["noc_alpha"]),
            reconstruct=bool(v["reconstruct"]),
            run_tag=f"component_{v['variant']}",
            out=out, export_data=True, panel_hint="Fig7_component_study"
        )
        row["variant"] = v["variant"]
        row["status"] = v["status"]
        row["mechanism"] = {
            "proposed": "Complete FV-side chain",
            "tpfa_only": "TPFA coefficient branch",
            "raw_overrelaxed": "Unclipped over-relaxed coefficient",
            "no_wall_closure": "Under-resolved no-slip drag",
            "noc_alpha_0p5": "Explicit non-orthogonal correction",
            "flux_reconstruction": "Flux-consistent velocity reconstruction",
        }.get(v["variant"], "")
        row["stage"] = "component_study"
        rows.append(row)
    rows.append({
        "case": case,
        "variant": "no_positive_area_filter",
        "status": "topology_only_not_assembled",
        "mechanism": "Flux-interface admissibility; zero-area contacts are not assembled as FV faces.",
        "e_K": float("nan"), "e_u": float("nan"), "e_phi": float("nan"),
        "mass_inf_per_volume": float("nan"), "mass_l2_per_volume": float("nan"),
    })
    pvfv_write_csv(out / "pvfv_component_study.csv", rows)
    pvfv_write_csv(out / "fig_data" / "fig7_component_study" / "fig7_component_data.csv", rows)
    return {"rows": rows}


def pvfv_make_final_tables(out: Path) -> dict:
    """Combine CSV outputs into manuscript-facing tables and panel datasets."""
    if pd is None:
        print("pandas not available; skipping combined tables")
        return {}
    flow_path = out / "pvfv_flow_sweep_abc.csv"
    beta_path = out / "pvfv_beta_sweep.csv"
    comp_path = out / "pvfv_component_study.csv"
    outputs: Dict[str, str] = {}
    if flow_path.exists():
        df = pd.read_csv(flow_path)
        df.to_csv(out / "table5_all_flow_rows.csv", index=False)
        outputs["table5_all"] = str(out / "table5_all_flow_rows.csv")
        # Representative row per case by a transparent accuracy score.
        for col in ["e_K", "e_phi", "e_u"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df["score"] = df["e_K"] + 0.25 * df["e_phi"].fillna(0.0) + 0.1 * df["e_u"].fillna(0.0)
        idx = df.groupby("case")["score"].idxmin()
        final = df.loc[idx].sort_values("case")
        final.to_csv(out / "table5_final_accuracy_cost_summary.csv", index=False)
        outputs["table5"] = str(out / "table5_final_accuracy_cost_summary.csv")
        df[df["case"].isin(["orthogonal_duct", "skewed_duct"])].to_csv(out / "fig_data" / "fig5_canonical_wall" / "fig5_refinement_transfer_data.csv", index=False)
        df[df["case"].isin(["A_thin_wall", "B_narrow_throat", "C_maze"])].to_csv(out / "fig_data" / "fig6_synthetic_pareto" / "fig6_pareto_data.csv", index=False)
        outputs["fig5_refinement"] = str(out / "fig_data" / "fig5_canonical_wall" / "fig5_refinement_transfer_data.csv")
        outputs["fig6_pareto"] = str(out / "fig_data" / "fig6_synthetic_pareto" / "fig6_pareto_data.csv")
    if comp_path.exists():
        comp = pd.read_csv(comp_path)
        comp.to_csv(out / "table6_component_study_summary.csv", index=False)
        comp.to_csv(out / "fig_data" / "fig7_component_study" / "fig7_component_data.csv", index=False)
        outputs["table6"] = str(out / "table6_component_study_summary.csv")
        outputs["fig7_component"] = str(out / "fig_data" / "fig7_component_study" / "fig7_component_data.csv")
    if beta_path.exists():
        beta = pd.read_csv(beta_path)
        beta.to_csv(out / "fig5_beta_sweep_data.csv", index=False)
        beta.to_csv(out / "fig_data" / "fig5_canonical_wall" / "fig5_beta_sweep_data.csv", index=False)
        outputs["beta"] = str(out / "fig5_beta_sweep_data.csv")
    return outputs


def pvfv_make_quick_plots(out: Path) -> None:
    """Generate non-publication quick-look plots from the exported panel datasets."""
    if pd is None or plt is None:
        print("pandas/matplotlib not available; skipping plots")
        return
    dirs = pvfv_ensure_dirs(out)
    figdir = dirs["quick"]
    flow_path = out / "pvfv_flow_sweep_abc.csv"
    beta_path = out / "pvfv_beta_sweep.csv"
    comp_path = out / "pvfv_component_study.csv"
    if beta_path.exists():
        df = pd.read_csv(beta_path)
        plt.figure()
        plt.plot(df["wall_beta"], df["e_K"], marker="o")
        plt.xlabel(r"$\beta_w$")
        plt.ylabel(r"$e_K$")
        plt.title("Fixed-seed duct wall-closure sweep")
        plt.tight_layout()
        plt.savefig(figdir / "fig5_beta_sweep_quick.png", dpi=200)
        plt.close()
        # Profile and mass histogram for beta-star/best-score row.
        try:
            df["score"] = pd.to_numeric(df["e_K"], errors="coerce") + 0.2 * pd.to_numeric(df["e_u"], errors="coerce").fillna(0.0)
            best = df.loc[df["score"].idxmin()]
            prof_path = Path(str(best.get("profile_csv", "")))
            if prof_path.exists():
                prof = pd.read_csv(prof_path)
                plt.figure()
                plt.plot(prof["Ux_ref_mean"], prof["y_index"], marker="o", label="reference")
                plt.plot(prof["Ux_coarse_mean"], prof["y_index"], marker="s", label="coarse")
                plt.xlabel(r"$\langle U_x\rangle_{z,x}$")
                plt.ylabel("y index")
                plt.legend()
                plt.tight_layout()
                plt.savefig(figdir / "fig5_velocity_profile_quick.png", dpi=200)
                plt.close()
            mass_path = Path(str(best.get("mass_residual_npz", "")))
            if mass_path.exists():
                dat = np.load(mass_path)
                plt.figure()
                plt.step(dat["log_hist_bins"][:-1], dat["log_hist_counts"], where="post")
                plt.xlabel(r"$\log_{10}(|D\phi|/V)$")
                plt.ylabel("count")
                plt.tight_layout()
                plt.savefig(figdir / "fig5_mass_residual_hist_quick.png", dpi=200)
                plt.close()
        except Exception as exc:
            print("[flow quick plots] Fig5 profile/hist failed:", exc)
    if flow_path.exists():
        df = pd.read_csv(flow_path)
        for metric, fname in [("e_K", "fig6_eK_vs_compression_quick.png"), ("e_phi", "fig6_ephi_vs_compression_quick.png"), ("S_pipe", "fig6_speedup_vs_compression_quick.png")]:
            plt.figure()
            for case, g in df.groupby("case"):
                plt.plot(g["C_comp"], g[metric], marker="o", label=case)
            plt.xscale("log")
            if metric.startswith("e_"):
                plt.yscale("log")
            plt.xlabel(r"$C_{comp}=N_{fl}/N_{cv}$")
            plt.ylabel(metric)
            plt.legend(fontsize=7)
            plt.tight_layout()
            plt.savefig(figdir / fname, dpi=200)
            plt.close()
        # Slice quick plots for best representative synthetic rows.
        try:
            syn = df[df["case"].isin(["A_thin_wall", "B_narrow_throat", "C_maze"])].copy()
            for col in ["e_K", "e_phi", "e_u"]:
                syn[col] = pd.to_numeric(syn[col], errors="coerce")
            syn["score"] = syn["e_K"] + 0.25 * syn["e_phi"].fillna(0.0) + 0.1 * syn["e_u"].fillna(0.0)
            for case, g in syn.groupby("case"):
                best = g.loc[g["score"].idxmin()]
                sp = Path(str(best.get("slice_npz", "")))
                if not sp.exists():
                    continue
                dat = np.load(sp)
                for key, suffix in [("Ux_ref_slice", "ref"), ("Ux_coarse_slice", "coarse"), ("Ux_error_slice", "error")]:
                    plt.figure()
                    arr = dat[key]
                    plt.imshow(np.where(dat["mask_slice"], arr, np.nan), origin="lower")
                    plt.colorbar()
                    plt.title(f"{case} {suffix}")
                    plt.tight_layout()
                    plt.savefig(figdir / f"fig6_{case}_{suffix}_slice_quick.png", dpi=200)
                    plt.close()
        except Exception as exc:
            print("[flow quick plots] Fig6 slices failed:", exc)
    if comp_path.exists():
        df = pd.read_csv(comp_path)
        if "status" in df:
            df = df[df["status"] == "assembled"]
        if len(df):
            plt.figure(figsize=(7, 3))
            plt.bar(df["variant"].astype(str), pd.to_numeric(df["e_K"], errors="coerce"))
            plt.ylabel(r"$e_K$")
            plt.xticks(rotation=35, ha="right")
            plt.tight_layout()
            plt.savefig(figdir / "fig7_component_eK_quick.png", dpi=200)
            plt.close()


def pvfv_write_data_products_manifest(out: Path, tables: dict, beta_summary: dict) -> str:
    """Write a figure/table product manifest so no required data are implicit."""
    manifest = {
        "scope": "Flow-simulation data products for Fig. 5, Fig. 6, Fig. 7, Table 5, and Table 6. ROI--JFA-specific Fig. 2/Fig. 4/Table 4 are handled by the partition benchmark notebook.",
        "figures": {
            "Fig5_canonical_wall_transfer": {
                "beta_sweep_csv": str(Path(out) / "fig_data" / "fig5_canonical_wall" / "fig5_beta_sweep_data.csv"),
                "refinement_transfer_csv": str(Path(out) / "fig_data" / "fig5_canonical_wall" / "fig5_refinement_transfer_data.csv"),
                "beta_star_profile_csv": beta_summary.get("best_profile_csv", ""),
                "beta_star_mass_hist_npz": beta_summary.get("best_mass_residual_npz", ""),
                "beta_star_slice_npz": beta_summary.get("best_slice_npz", ""),
            },
            "Fig6_synthetic_pareto": {
                "pareto_csv": str(Path(out) / "fig_data" / "fig6_synthetic_pareto" / "fig6_pareto_data.csv"),
                "slice_npz_paths": "See slice_npz column in pareto_csv for each run; representative rows are chosen by table5 score.",
            },
            "Fig7_component_study": {
                "component_csv": str(Path(out) / "fig_data" / "fig7_component_study" / "fig7_component_data.csv"),
                "per_variant_raw_arrays": "See run_arrays_npz, flux_npz, mass_residual_npz, slice_npz columns for assembled variants.",
            },
        },
        "tables": {
            "Table5_final_accuracy_cost_summary": tables.get("table5", ""),
            "Table5_all_flow_rows": tables.get("table5_all", ""),
            "Table6_component_study_summary": tables.get("table6", ""),
        },
        "notes": [
            "All per-run rows include profile_csv, slice_npz, mass_residual_npz, flux_npz, and run_arrays_npz columns.",
            "S_pipe is whole-pipeline speedup relative to voxel FV reference, not ROI--JFA partition speedup.",
            "C_comp is N_fl/N_cv after face-connected component splitting.",
        ],
    }
    path = Path(out) / "pvfv_data_products_manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return str(path)


def pvfv_run_complete_abc_suite(profile: str = "pilot", zip_outputs: bool = True) -> dict:
    """Run the complete current simulation suite and export all data for its assigned figures/tables.

    This notebook is responsible for:
      - Fig. 5: canonical FV verification and wall-closure transfer;
      - Fig. 6: synthetic ABC accuracy--compression--runtime Pareto;
      - Fig. 7: FV-side component study;
      - Table 5: final accuracy--cost summary;
      - Table 6: component-study summary.
    It does not generate ROI--JFA-specific Fig. 2/Fig. 4/Table 4 data.
    """
    require_cuda_gpu()
    out = Path(PVFV_OUT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    pvfv_ensure_dirs(out)
    cfg = pvfv_base_cfg(out_dir=str(out), profile=profile)
    plan = pvfv_case_plan(profile)
    manifest = {
        "suite": "PoreVoronoi-FV ABC full-data flow simulation suite",
        "profile": profile,
        "out_dir": str(out),
        "case_plan": plan,
        "cfg": {k: str(getattr(cfg, k)) for k in PB615Config.__dataclass_fields__.keys()},
        "assigned_figures_tables": ["Fig5", "Fig6", "Fig7", "Table5", "Table6"],
        "labeler_scope_note": "Current notebook uses exact GPU geodesic relaxation as the label source for flow simulations. ROI--JFA partition-performance claims are produced by a separate partition benchmark notebook and can use the same labels/distances/moments contract.",
    }
    (out / "pvfv_suite_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    wall = pvfv_run_wall_calibration(cfg, plan, out)
    beta_star = float(wall["beta_star"])
    sweeps = pvfv_run_canonical_and_abc(cfg, plan, beta_star, out)
    comp = pvfv_run_component_study(cfg, plan, beta_star, sweeps["ref_cache"], out)
    tables = pvfv_make_final_tables(out)
    pvfv_make_quick_plots(out)
    products_manifest = pvfv_write_data_products_manifest(out, tables, wall.get("summary", {}))

    zip_path = None
    if zip_outputs:
        zip_path = shutil.make_archive(str(out), "zip", out)
        print(f"[flow] zipped outputs: {zip_path}")
    summary = {
        "out_dir": str(out),
        "zip_path": zip_path,
        "beta_star": beta_star,
        "n_flow_rows": len(sweeps["rows"]),
        "n_component_rows": len(comp["rows"]),
        "tables": tables,
        "data_products_manifest": products_manifest,
        "figures_tables_fully_supported_by_this_notebook": ["Fig5", "Fig6", "FV-side Fig7", "Table5", "FV-side Table6"],
        "figures_tables_not_supported_by_this_notebook": ["Fig1", "Fig2", "Fig3 schematic only", "Fig4", "Table1", "Table2", "Table3", "Table4"],
    }
    (out / "pvfv_suite_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("[flow] suite summary:", summary)
    return summary

print("Full-data export overrides loaded. Run `summary = pvfv_run_complete_abc_suite(profile='pilot')`.")



## Seed-influence suite

This block extends the flow-simulation suite so that seed effects are measured explicitly.  It runs density sweeps, stride-offset sensitivity, random/wall-biased seed placement comparisons, canonical wall calibration, ABC synthetic Pareto data, and FV-side component studies.  It uses exact GPU geodesic labels for the flow-simulation side; ROI--JFA partition benchmarks are not part of this notebook.


In [ ]:
# ============================================================
# Seed-influence full-data suite
# Built on the definitions in the cells above
# ============================================================
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import math, time, json, shutil
import numpy as np

PVFV_SEED_OUT_DIR = "pvfv_abc_seed_influence_outputs"


def pvfv_seed_influence_case_plan(profile: str = "pilot") -> dict:
    """Case plan that explicitly measures seed-density and seed-placement effects.

    The plan keeps the available canonical + A/B/C cases but avoids relying on a
    single stride family.  It reports both density sweeps and placement sensitivity.
    """
    profile = str(profile).lower()
    if profile in ("production_dense", "dense"):
        shape = (24, 24, 64)
        density_strides = [(2, 2, 2), (3, 3, 3), (4, 4, 4), (5, 5, 5), (6, 6, 6)]
        fixed_stride = (3, 3, 3)
        beta_values = (0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.4, 1.6, 2.0)
        random_repeats = 3
    elif profile == "production":
        shape = (24, 24, 64)
        density_strides = [(3, 3, 3), (4, 4, 4), (5, 5, 5), (6, 6, 6)]
        fixed_stride = (3, 3, 3)
        beta_values = (0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.4, 1.6, 2.0)
        random_repeats = 3
    else:
        shape = (12, 12, 24)
        density_strides = [(2, 2, 2), (3, 3, 3), (4, 4, 4)]
        fixed_stride = (3, 3, 3)
        beta_values = (0.75, 0.9, 1.0, 1.1, 1.25, 1.5)
        random_repeats = 2

    return {
        "profile": profile,
        "shape_zyx": shape,
        "density_strides": density_strides,
        "fixed_stride": fixed_stride,
        "beta_values": beta_values,
        "cases": ["orthogonal_duct", "skewed_duct", "A_thin_wall", "B_narrow_throat", "C_maze"],
        "synthetic_cases": ["A_thin_wall", "B_narrow_throat", "C_maze"],
        "canonical_cases": ["orthogonal_duct", "skewed_duct"],
        "component_case": "skewed_duct",
        "component_stride": fixed_stride,
        "random_repeats": int(random_repeats),
        "offset_modes": ["origin", "half"],
    }


def pvfv_stride_offset_seed_flat_gpu(mask: cp.ndarray, stride_zyx: Tuple[int, int, int],
                                      offset_zyx: Tuple[int, int, int]) -> cp.ndarray:
    """Regular stride seeds with an explicit phase/offset."""
    D, H, W = [int(v) for v in mask.shape]
    sz, sy, sx = [max(1, int(v)) for v in stride_zyx]
    oz, oy, ox = [int(v) for v in offset_zyx]
    zz, yy, xx = cp.indices(mask.shape)
    sel = mask & (((zz - oz) % sz) == 0) & (((yy - oy) % sy) == 0) & (((xx - ox) % sx) == 0)
    seed_flat = cp.flatnonzero(sel.ravel()).astype(cp.int64)
    if int(seed_flat.size) == 0:
        seed_flat = make_regular_seeds_gpu(mask, stride_zyx).astype(cp.int64)
    return seed_flat


def pvfv_offset_from_mode(stride_zyx: Tuple[int, int, int], mode: str) -> Tuple[int, int, int]:
    sz, sy, sx = [max(1, int(v)) for v in stride_zyx]
    mode = str(mode).lower()
    if mode == "origin":
        return (0, 0, 0)
    if mode == "half":
        return (sz // 2, sy // 2, sx // 2)
    if mode == "quarter":
        return (max(0, sz // 4), max(0, sy // 4), max(0, sx // 4))
    return (0, 0, 0)


def pvfv_random_seed_flat_gpu(mask: cp.ndarray, n_seed: int, rng_seed: int = 0) -> cp.ndarray:
    """Uniform random seeds over traversable voxels."""
    valid = pvfv_to_numpy(cp.flatnonzero(mask.ravel())).astype(np.int64)
    if valid.size == 0:
        raise ValueError("mask contains no traversable voxels")
    n = int(max(1, min(int(n_seed), int(valid.size))))
    rng = np.random.default_rng(int(rng_seed))
    chosen = rng.choice(valid, size=n, replace=False)
    return cp.asarray(chosen.astype(np.int64))


def pvfv_wall_biased_seed_flat_gpu(mask: cp.ndarray, n_seed: int, rng_seed: int = 0,
                                    wall_fraction: float = 0.65) -> cp.ndarray:
    """Wall-biased seed set: sample many seeds near fluid-solid faces plus a background set.

    This is a sensitivity/placement study only; it is not the baseline method.
    """
    m = pvfv_to_numpy(mask).astype(bool)
    D, H, W = m.shape
    wall_adj = np.zeros_like(m, dtype=bool)
    dirs = [(1,0,0), (-1,0,0), (0,1,0), (0,-1,0), (0,0,1), (0,0,-1)]
    for dz, dy, dx in dirs:
        neigh = np.zeros_like(m, dtype=bool)
        if dx != 0:
            neigh = np.roll(m, shift=dx, axis=2)  # periodic x for this flow setup
        else:
            zsrc = slice(max(0, -dz), min(D, D-dz))
            zdst = slice(max(0, dz), min(D, D+dz))
            ysrc = slice(max(0, -dy), min(H, H-dy))
            ydst = slice(max(0, dy), min(H, H+dy))
            neigh[zdst, ydst, :] = m[zsrc, ysrc, :]
        wall_adj |= (m & (~neigh))
    valid_all = np.flatnonzero(m.ravel()).astype(np.int64)
    valid_wall = np.flatnonzero((m & wall_adj).ravel()).astype(np.int64)
    valid_bg = valid_all
    rng = np.random.default_rng(int(rng_seed))
    n = int(max(1, min(int(n_seed), int(valid_all.size))))
    n_wall = int(round(float(wall_fraction) * n))
    n_wall = min(n_wall, int(valid_wall.size))
    chosen = []
    if n_wall > 0:
        chosen.extend(rng.choice(valid_wall, size=n_wall, replace=False).tolist())
    remaining = n - len(chosen)
    if remaining > 0:
        pool = np.setdiff1d(valid_bg, np.asarray(chosen, dtype=np.int64), assume_unique=False)
        if pool.size == 0:
            pool = valid_bg
        chosen.extend(rng.choice(pool, size=min(remaining, pool.size), replace=False).tolist())
    return cp.asarray(np.asarray(chosen, dtype=np.int64))


def pvfv_build_geometry_from_seed_flat_timed(mask: cp.ndarray, seed_flat: cp.ndarray, cfg: PB615Config,
                                              *, split_face_components: bool = True,
                                              label_mode: str = "exact_geodesic",
                                              seed_spec: str = "custom") -> Tuple[GPUFVMGeometry, dict]:
    """Build labels/moments from an explicit seed list, with timing and topology metadata."""
    if label_mode != "exact_geodesic":
        raise NotImplementedError("Current flow suite uses exact geodesic labels; ROI--JFA timing is handled separately.")
    t0 = time.perf_counter()
    seed_flat = cp.asarray(seed_flat, dtype=cp.int64)
    cp.cuda.Stream.null.synchronize()
    t_seed = time.perf_counter() - t0

    t1 = time.perf_counter()
    dist, labels = geodesic_voronoi_labels_gpu(mask, seed_flat, cfg)
    cp.cuda.Stream.null.synchronize()
    t_label = time.perf_counter() - t1

    n_seed = int(seed_flat.size)
    t_split0 = time.perf_counter()
    split_info = {
        "n_original_labels": n_seed,
        "n_cv": n_seed,
        "n_split_extra": 0,
        "n_face_disconnected_labels": 0,
        "face_connected_fraction": 1.0,
    }
    if split_face_components:
        labels, split_info = pvfv_face_connected_reindex_cpu(mask, labels)
    cp.cuda.Stream.null.synchronize()
    t_split = time.perf_counter() - t_split0

    n0a = pvfv_zero_area_candidate_count_cpu(mask, labels)

    t2 = time.perf_counter()
    geom = pvfv_build_geometry_from_labels_ncells_gpu(mask, labels, dist, int(split_info["n_cv"]), cfg)
    cp.cuda.Stream.null.synchronize()
    t_mom = time.perf_counter() - t2

    n_fl = int(mask.sum().get())
    meta = {
        "S": n_seed,
        "N_cv": int(geom.n_cells),
        "N_fl": n_fl,
        "C_comp": float(n_fl / max(int(geom.n_cells), 1)),
        "t_seed_s": float(t_seed),
        "t_label_s": float(t_label),
        "t_split_s": float(t_split),
        "t_part_s": float(t_seed + t_label + t_split),
        "t_mom_s": float(t_mom),
        "N_split": int(split_info["n_split_extra"]),
        "N_face_disconnected_labels": int(split_info["n_face_disconnected_labels"]),
        "face_connected_fraction": float(split_info["face_connected_fraction"]),
        "N_0A": int(n0a),
        "label_mode": label_mode,
        "distance_connectivity": int(cfg.distance_connectivity),
        "seed_spec": str(seed_spec),
    }
    return geom, meta


def pvfv_quality_flags(row: dict, *, mass_tol: float = 1.0e-6,
                        max_main_eK: float = 1.0, max_main_ephi: float = 1.5) -> dict:
    """Add physical/plot-quality flags without hiding the raw results."""
    reasons = []
    for key in ("e_K", "e_u", "e_phi", "K_eff_x", "S_pipe"):
        try:
            if not math.isfinite(float(row.get(key, float("nan")))):
                reasons.append(f"nonfinite_{key}")
        except Exception:
            reasons.append(f"bad_{key}")
    try:
        if float(row.get("K_eff_x", float("nan"))) <= 0:
            reasons.append("nonpositive_K_eff")
    except Exception:
        pass
    try:
        if abs(float(row.get("mass_inf_per_volume", 0.0))) > float(mass_tol):
            reasons.append("high_mass_residual")
    except Exception:
        pass
    try:
        if str(row.get("solver_failure", "")) not in ("", "None", "nan"):
            reasons.append("solver_failure")
    except Exception:
        pass
    physical = len(reasons) == 0
    main_reasons = list(reasons)
    try:
        if float(row.get("e_K", float("inf"))) > max_main_eK:
            main_reasons.append("large_e_K")
    except Exception:
        main_reasons.append("large_e_K")
    try:
        if float(row.get("e_phi", float("inf"))) > max_main_ephi:
            main_reasons.append("large_e_phi")
    except Exception:
        main_reasons.append("large_e_phi")
    out = dict(row)
    out["valid_physical"] = bool(physical)
    out["valid_for_main_pareto"] = bool(len(main_reasons) == 0)
    out["quality_reasons"] = ";".join(main_reasons)
    return out


def pvfv_run_flow_case_seeded(case_name: str, mask: cp.ndarray, seed_flat: cp.ndarray, seed_spec: dict,
                               cfg: PB615Config, ref_geom: GPUFVMGeometry, ref_result: Dict[str, Any],
                               *, wall_beta: float = 1.0,
                               face_mode: str = "overrelaxed_default",
                               noc_alpha: float = 0.0,
                               reconstruct: bool = False,
                               split_face_components: bool = True,
                               run_tag: str = "seeded",
                               out: Optional[Path] = None,
                               export_data: bool = True,
                               panel_hint: str = "seeded") -> Tuple[dict, GPUFVMGeometry, Dict[str, Any]]:
    """Build/solve one case from an explicit seed list and export all data."""
    local = pvfv_clone_cfg(cfg,
                            transmissibility_ratio_clip=_pb618_face_clip_for_mode(face_mode),
                            explicit_nonorthogonal_correction=float(noc_alpha),
                            velocity_reconstruct_from_flux=bool(reconstruct))
    seed_desc = str(seed_spec.get("seed_id", seed_spec.get("family", "custom")))
    geom, meta = pvfv_build_geometry_from_seed_flat_timed(mask, seed_flat, local,
                                                           split_face_components=split_face_components,
                                                           seed_spec=seed_desc)
    if abs(float(wall_beta) - 1.0) > 1.0e-15:
        geom = _pb618_clone_geometry_with_scaled_twall(geom, float(wall_beta))
    res = run_velocity_pressure_projection_gpu(geom, local, run_label=f"pvfv_{case_name}_{run_tag}")
    Uref_c, pref_c = coarsen_voxel_reference_to_coarse_gpu(ref_geom, ref_result, geom)
    e_u = pvfv_weighted_rel_l2_vec_gpu(res["U"], Uref_c, geom.volume)
    e_k = abs(float(res["K_eff_x"]) - float(ref_result["K_eff_x"])) / max(abs(float(ref_result["K_eff_x"])), 1.0e-300)
    try:
        e_phi = pvfv_flux_rel_error(ref_geom, ref_result, geom, res)
    except Exception as exc:
        print(f"[flow-seed] flux error failed for {case_name}/{run_tag}: {exc}")
        e_phi = float("nan")
    ref_total = float(ref_result.get("pvfv_total_time_s", ref_result.get("elapsed_s", float("nan"))))
    t_total = float(meta["t_part_s"] + meta["t_mom_s"] + float(res.get("elapsed_s", 0.0)))
    row = {
        "case": case_name,
        "run_tag": run_tag,
        "stage": str(seed_spec.get("stage", "seed_sweep")),
        "seed_family": str(seed_spec.get("family", "custom")),
        "seed_id": str(seed_spec.get("seed_id", run_tag)),
        "stride_zyx": str(seed_spec.get("stride_zyx", "")),
        "offset_zyx": str(seed_spec.get("offset_zyx", "")),
        "repeat": int(seed_spec.get("repeat", 0)),
        "S": int(meta["S"]),
        "N_cv": int(meta["N_cv"]),
        "N_fl": int(meta["N_fl"]),
        "C_comp": float(meta["C_comp"]),
        "N_split": int(meta["N_split"]),
        "N_0A": int(meta["N_0A"]),
        "face_connected_fraction": float(meta["face_connected_fraction"]),
        "face_mode": str(face_mode),
        "wall_beta": float(wall_beta),
        "wall_distance_floor": float(local.wall_distance_floor),
        "noc_alpha": float(noc_alpha),
        "reconstruct": bool(reconstruct),
        "e_K": float(e_k),
        "e_u": float(e_u),
        "e_phi": float(e_phi),
        "mass_inf_per_volume": float(res.get("mass_inf_per_volume", float("nan"))),
        "mass_l2_per_volume": float(res.get("mass_l2_per_volume", float("nan"))),
        "steady_momentum_inf": float(res.get("steady_momentum_inf", float("nan"))),
        "steady_converged": bool(res.get("steady_converged", False)),
        "steps_completed": int(res.get("steps_completed", -1)),
        "solver_failure": str(res.get("solver_failure", "")),
        "K_eff_x": float(res.get("K_eff_x", float("nan"))),
        "K_ref_x": float(ref_result.get("K_eff_x", float("nan"))),
        "t_part_s": float(meta["t_part_s"]),
        "t_mom_s": float(meta["t_mom_s"]),
        "t_flow_s": float(res.get("elapsed_s", float("nan"))),
        "t_total_s": float(t_total),
        "t_ref_total_s": float(ref_total),
        "S_pipe": float(ref_total / t_total) if math.isfinite(ref_total) and t_total > 0 else float("nan"),
        "label_mode": meta["label_mode"],
        "distance_connectivity": int(meta["distance_connectivity"]),
        "data_manifest": "",
        "profile_csv": "",
        "slice_npz": "",
        "mass_residual_npz": "",
        "flux_npz": "",
        "run_arrays_npz": "",
    }
    row = pvfv_quality_flags(row)
    if export_data and out is not None:
        pseudo_stride = tuple(seed_spec.get("stride_zyx", (0, 0, 0))) if seed_spec.get("stride_zyx", None) else (0, 0, 0)
        products = pvfv_save_flow_products(Path(out), case_name, tuple(pseudo_stride), run_tag, ref_geom, ref_result, geom, res, row, panel_hint=panel_hint)
        row.update(products)
        row["data_manifest"] = str(Path(products["run_arrays_npz"]).parent / "run_data_manifest.json")
    return row, geom, res


def pvfv_seed_specs_for_density(mask: cp.ndarray, stride: Tuple[int, int, int], offset_modes: List[str]) -> List[Tuple[dict, cp.ndarray]]:
    specs = []
    for mode in offset_modes:
        off = pvfv_offset_from_mode(stride, mode)
        seed_flat = pvfv_stride_offset_seed_flat_gpu(mask, stride, off)
        spec = {
            "stage": "seed_density_sweep",
            "family": "stride_offset",
            "seed_id": f"stride_{'x'.join(map(str,stride))}_offset_{mode}",
            "stride_zyx": tuple(stride),
            "offset_zyx": tuple(off),
            "repeat": 0,
        }
        specs.append((spec, seed_flat))
    return specs


def pvfv_seed_specs_for_strategy(mask: cp.ndarray, fixed_stride: Tuple[int, int, int],
                                  random_repeats: int, base_seed_count: Optional[int] = None) -> List[Tuple[dict, cp.ndarray]]:
    if base_seed_count is None:
        base_seed_count = int(pvfv_stride_offset_seed_flat_gpu(mask, fixed_stride, (0,0,0)).size)
    specs: List[Tuple[dict, cp.ndarray]] = []
    # deterministic stride phases
    for mode in ["origin", "half"]:
        off = pvfv_offset_from_mode(fixed_stride, mode)
        specs.append(({"stage": "seed_strategy_sweep", "family": "stride_offset", "seed_id": f"strategy_stride_{mode}",
                       "stride_zyx": tuple(fixed_stride), "offset_zyx": tuple(off), "repeat": 0},
                      pvfv_stride_offset_seed_flat_gpu(mask, fixed_stride, off)))
    # random repeated seeds
    for r in range(int(random_repeats)):
        specs.append(({"stage": "seed_strategy_sweep", "family": "random", "seed_id": f"random_r{r}",
                       "stride_zyx": tuple(fixed_stride), "offset_zyx": "", "repeat": r},
                      pvfv_random_seed_flat_gpu(mask, base_seed_count, rng_seed=1000 + r)))
    # wall-biased repeated seeds
    for r in range(max(1, int(random_repeats)//2)):
        specs.append(({"stage": "seed_strategy_sweep", "family": "wall_biased", "seed_id": f"wall_biased_r{r}",
                       "stride_zyx": tuple(fixed_stride), "offset_zyx": "", "repeat": r},
                      pvfv_wall_biased_seed_flat_gpu(mask, base_seed_count, rng_seed=2000 + r)))
    return specs


def pvfv_group_summary_csv(df, group_cols: List[str], out_csv: Path) -> None:
    if pd is None or df is None or len(df) == 0:
        return
    metrics = ["C_comp", "e_K", "e_u", "e_phi", "S_pipe", "mass_inf_per_volume", "N_cv", "N_split", "N_0A"]
    rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {c: k for c, k in zip(group_cols, keys)}
        row["n_runs"] = int(len(g))
        row["n_valid_physical"] = int(pd.to_numeric(g.get("valid_physical", False), errors="coerce").fillna(False).astype(bool).sum()) if "valid_physical" in g else 0
        row["n_valid_main"] = int(pd.to_numeric(g.get("valid_for_main_pareto", False), errors="coerce").fillna(False).astype(bool).sum()) if "valid_for_main_pareto" in g else 0
        for m in metrics:
            if m in g:
                vals = pd.to_numeric(g[m], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
                if len(vals):
                    row[f"{m}_median"] = float(vals.median())
                    row[f"{m}_min"] = float(vals.min())
                    row[f"{m}_max"] = float(vals.max())
                    row[f"{m}_q25"] = float(vals.quantile(0.25))
                    row[f"{m}_q75"] = float(vals.quantile(0.75))
                else:
                    row[f"{m}_median"] = float("nan")
        rows.append(row)
    pvfv_write_csv(out_csv, rows)


def pvfv_make_seed_influence_tables(out: Path) -> dict:
    outputs = {}
    if pd is None:
        return outputs
    density_path = out / "pvfv_seed_density_sweep.csv"
    strategy_path = out / "pvfv_seed_strategy_sweep.csv"
    component_path = out / "pvfv_component_study.csv"
    if density_path.exists():
        df = pd.read_csv(density_path)
        # Keep all rows for table traceability.
        df.to_csv(out / "table5_all_flow_rows.csv", index=False)
        outputs["table5_all"] = str(out / "table5_all_flow_rows.csv")
        # Select representative physically valid rows; fall back to all rows if no valid rows in a case.
        for c in ["e_K", "e_phi", "e_u"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["score"] = df["e_K"] + 0.25 * df["e_phi"].fillna(0.0) + 0.1 * df["e_u"].fillna(0.0)
        reps = []
        for case, g in df.groupby("case"):
            gv = g[g.get("valid_physical", False).astype(bool)] if "valid_physical" in g else g
            if len(gv) == 0:
                gv = g
            reps.append(gv.loc[gv["score"].idxmin()])
        final = pd.DataFrame(reps).sort_values("case")
        final.to_csv(out / "table5_final_accuracy_cost_summary.csv", index=False)
        outputs["table5"] = str(out / "table5_final_accuracy_cost_summary.csv")
        # Figure data files.
        fig6 = out / "fig_data" / "fig6_synthetic_pareto"
        fig6.mkdir(parents=True, exist_ok=True)
        df.to_csv(fig6 / "fig6_pareto_data.csv", index=False)
        outputs["fig6_pareto"] = str(fig6 / "fig6_pareto_data.csv")
        pvfv_group_summary_csv(df, ["case", "stride_zyx"], fig6 / "fig6_seed_density_summary.csv")
        outputs["fig6_density_summary"] = str(fig6 / "fig6_seed_density_summary.csv")
        # Fig5 canonical subset.
        fig5 = out / "fig_data" / "fig5_canonical_wall"
        fig5.mkdir(parents=True, exist_ok=True)
        df[df["case"].isin(["orthogonal_duct", "skewed_duct"])].to_csv(fig5 / "fig5_refinement_transfer_data.csv", index=False)
        outputs["fig5_refinement"] = str(fig5 / "fig5_refinement_transfer_data.csv")
    if strategy_path.exists():
        strat = pd.read_csv(strategy_path)
        fig6 = out / "fig_data" / "fig6_synthetic_pareto"
        fig6.mkdir(parents=True, exist_ok=True)
        strat.to_csv(fig6 / "fig6_seed_strategy_data.csv", index=False)
        outputs["fig6_strategy"] = str(fig6 / "fig6_seed_strategy_data.csv")
        pvfv_group_summary_csv(strat, ["case", "seed_family"], fig6 / "fig6_seed_strategy_summary.csv")
        outputs["fig6_strategy_summary"] = str(fig6 / "fig6_seed_strategy_summary.csv")
    if component_path.exists():
        comp = pd.read_csv(component_path)
        comp.to_csv(out / "table6_component_study_summary.csv", index=False)
        fig7 = out / "fig_data" / "fig7_component_study"
        fig7.mkdir(parents=True, exist_ok=True)
        comp.to_csv(fig7 / "fig7_component_data.csv", index=False)
        outputs["table6"] = str(out / "table6_component_study_summary.csv")
        outputs["fig7_component"] = str(fig7 / "fig7_component_data.csv")
    beta_path = out / "pvfv_beta_sweep.csv"
    if beta_path.exists():
        fig5 = out / "fig_data" / "fig5_canonical_wall"
        fig5.mkdir(parents=True, exist_ok=True)
        beta = pd.read_csv(beta_path)
        beta.to_csv(fig5 / "fig5_beta_sweep_data.csv", index=False)
        beta.to_csv(out / "fig5_beta_sweep_data.csv", index=False)
        outputs["beta"] = str(fig5 / "fig5_beta_sweep_data.csv")
    return outputs


def pvfv_make_seed_influence_quick_plots(out: Path) -> None:
    if pd is None or plt is None:
        return
    pvfv_make_quick_plots(out)
    figdir = Path(out) / "quick_figures"
    figdir.mkdir(parents=True, exist_ok=True)
    density_path = Path(out) / "pvfv_seed_density_sweep.csv"
    strategy_path = Path(out) / "pvfv_seed_strategy_sweep.csv"
    if density_path.exists():
        df = pd.read_csv(density_path)
        for metric, fname in [("e_K", "seed_density_eK_all_offsets.png"), ("e_phi", "seed_density_ephi_all_offsets.png"), ("S_pipe", "seed_density_speedup_all_offsets.png")]:
            plt.figure()
            for case, g in df.groupby("case"):
                x = pd.to_numeric(g["C_comp"], errors="coerce")
                y = pd.to_numeric(g[metric], errors="coerce")
                valid = np.isfinite(x) & np.isfinite(y)
                plt.scatter(x[valid], y[valid], label=case, s=24)
            plt.xscale("log")
            if metric.startswith("e_"):
                plt.yscale("log")
            plt.xlabel(r"$C_{comp}=N_{fl}/N_{cv}$")
            plt.ylabel(metric)
            plt.legend(fontsize=7)
            plt.tight_layout()
            plt.savefig(figdir / fname, dpi=200)
            plt.close()
    if strategy_path.exists():
        df = pd.read_csv(strategy_path)
        plt.figure(figsize=(7, 3))
        if "seed_family" in df:
            families = list(df["seed_family"].dropna().unique())
            vals = [pd.to_numeric(df[df["seed_family"] == f]["e_K"], errors="coerce").dropna().values for f in families]
            if vals:
                plt.boxplot(vals, labels=families, showfliers=True)
                plt.ylabel(r"$e_K$")
                plt.xticks(rotation=20, ha="right")
                plt.tight_layout()
                plt.savefig(figdir / "seed_strategy_eK_boxplot.png", dpi=200)
                plt.close()


def pvfv_write_seed_influence_manifest(out: Path, tables: dict, beta_summary: dict) -> str:
    manifest = {
        "scope": "Full flow-simulation data products with explicit seed-density and seed-placement influence for Fig. 5, Fig. 6, FV-side Fig. 7, Table 5, and Table 6.",
        "figures": {
            "Fig5_canonical_wall_transfer": {
                "beta_sweep_csv": str(Path(out) / "fig_data" / "fig5_canonical_wall" / "fig5_beta_sweep_data.csv"),
                "refinement_transfer_csv": str(Path(out) / "fig_data" / "fig5_canonical_wall" / "fig5_refinement_transfer_data.csv"),
                "beta_star_profile_csv": beta_summary.get("best_profile_csv", ""),
                "beta_star_mass_hist_npz": beta_summary.get("best_mass_residual_npz", ""),
            },
            "Fig6_synthetic_pareto": {
                "pareto_csv": str(Path(out) / "fig_data" / "fig6_synthetic_pareto" / "fig6_pareto_data.csv"),
                "seed_density_summary_csv": str(Path(out) / "fig_data" / "fig6_synthetic_pareto" / "fig6_seed_density_summary.csv"),
                "seed_strategy_csv": str(Path(out) / "fig_data" / "fig6_synthetic_pareto" / "fig6_seed_strategy_data.csv"),
                "seed_strategy_summary_csv": str(Path(out) / "fig_data" / "fig6_synthetic_pareto" / "fig6_seed_strategy_summary.csv"),
            },
            "Fig7_component_study": {
                "component_csv": str(Path(out) / "fig_data" / "fig7_component_study" / "fig7_component_data.csv"),
            },
        },
        "tables": tables,
        "notes": [
            "Seed-density rows contain multiple stride offsets so seed phase sensitivity can be shown as scatter/ribbons rather than hidden.",
            "Seed-strategy rows compare regular stride, random, and wall-biased seed placement at comparable seed counts.",
            "valid_physical and valid_for_main_pareto flags mark nonphysical or over-compressed runs without deleting raw data.",
            "This flow suite still uses exact GPU geodesic labels; ROI--JFA-specific Fig. 2/Fig. 4/Table 4 are handled by a separate partition benchmark notebook.",
        ],
    }
    path = Path(out) / "pvfv_seed_influence_data_products_manifest.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return str(path)


def pvfv_run_complete_seed_influence_suite(profile: str = "pilot", zip_outputs: bool = True,
                                             include_strategy: bool = True,
                                             clean_output: bool = True) -> dict:
    """Run comprehensive canonical + ABC flow simulations with seed influence.

    Responsible for all flow-side products: Fig. 5, Fig. 6, FV-side Fig. 7,
    Table 5, and Table 6.  The ROI--JFA partition-specific products remain a
    separate notebook.
    """
    require_cuda_gpu()
    out = Path(PVFV_SEED_OUT_DIR)
    if clean_output and out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True, exist_ok=True)
    pvfv_ensure_dirs(out)
    cfg = pvfv_base_cfg(out_dir=str(out), profile=profile)
    plan = pvfv_seed_influence_case_plan(profile)
    # keep beta values consistent with this expanded plan
    base_plan = dict(plan)
    base_plan["seed_strides"] = plan["density_strides"]
    manifest = {
        "suite": "PoreVoronoi-FV ABC seed-influence full-data flow simulation suite",
        "profile": profile,
        "out_dir": str(out),
        "case_plan": plan,
        "cfg": {k: str(getattr(cfg, k)) for k in PB615Config.__dataclass_fields__.keys()},
        "assigned_figures_tables": ["Fig5", "Fig6", "FV-side Fig7", "Table5", "Table6"],
        "labeler_scope_note": "This suite uses exact GPU geodesic labels as the label source for flow simulations. ROI--JFA partition-performance claims are produced by a separate partition benchmark notebook.",
    }
    (out / "pvfv_suite_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    print("[flow-seed] wall calibration")
    wall_plan = {
        "shape_zyx": plan["shape_zyx"],
        "fixed_stride": plan["fixed_stride"],
        "beta_values": plan["beta_values"],
    }
    wall = pvfv_run_wall_calibration(cfg, wall_plan, out)
    beta_star = float(wall["beta_star"])
    print(f"[flow-seed] beta_star = {beta_star}")

    # Build references once per case.
    ref_cache: Dict[str, Tuple[cp.ndarray, GPUFVMGeometry, Dict[str, Any]]] = {}
    ref_rows = []
    for case in plan["cases"]:
        print(f"\n[flow-seed] reference for {case}")
        mask = pvfv_make_mask_gpu(case, plan["shape_zyx"])
        ref_geom, ref_res, ref_row = pvfv_build_reference(case, mask, cfg, out=out)
        ref_cache[case] = (mask, ref_geom, ref_res)
        ref_rows.append(ref_row)
    pvfv_write_csv(out / "pvfv_reference_rows_all.csv", ref_rows)

    # Density + offset sweep.
    density_rows: List[dict] = []
    for case in plan["cases"]:
        mask, ref_geom, ref_res = ref_cache[case]
        for stride in plan["density_strides"]:
            for spec, seed_flat in pvfv_seed_specs_for_density(mask, tuple(stride), list(plan["offset_modes"])):
                run_tag = f"density_{spec['seed_id']}"
                print(f"[flow-seed] {case} {run_tag} S={int(seed_flat.size)}")
                row, _, _ = pvfv_run_flow_case_seeded(case, mask, seed_flat, spec, cfg, ref_geom, ref_res,
                                                        wall_beta=beta_star,
                                                        face_mode="overrelaxed_default",
                                                        run_tag=run_tag,
                                                        out=out,
                                                        export_data=True,
                                                        panel_hint="density")
                density_rows.append(row)
    pvfv_write_csv(out / "pvfv_seed_density_sweep.csv", density_rows)
    # Backwards-compatible name for existing figure/table code.
    pvfv_write_csv(out / "pvfv_flow_sweep_abc.csv", density_rows)

    # Seed-placement strategy sweep on ABC synthetic cases.
    strategy_rows: List[dict] = []
    if include_strategy:
        for case in plan["synthetic_cases"]:
            mask, ref_geom, ref_res = ref_cache[case]
            base_count = int(pvfv_stride_offset_seed_flat_gpu(mask, tuple(plan["fixed_stride"]), (0,0,0)).size)
            for spec, seed_flat in pvfv_seed_specs_for_strategy(mask, tuple(plan["fixed_stride"]), int(plan["random_repeats"]), base_seed_count=base_count):
                run_tag = f"strategy_{spec['seed_id']}"
                print(f"[flow-seed] {case} {run_tag} S={int(seed_flat.size)}")
                row, _, _ = pvfv_run_flow_case_seeded(case, mask, seed_flat, spec, cfg, ref_geom, ref_res,
                                                        wall_beta=beta_star,
                                                        face_mode="overrelaxed_default",
                                                        run_tag=run_tag,
                                                        out=out,
                                                        export_data=True,
                                                        panel_hint="strategy")
                strategy_rows.append(row)
        pvfv_write_csv(out / "pvfv_seed_strategy_sweep.csv", strategy_rows)

    # Component study: use skewed duct by default for non-orthogonal/FV-side variants.
    comp_plan = dict(plan)
    comp_plan["seed_strides"] = plan["density_strides"]
    comp_plan["component_case"] = plan.get("component_case", "skewed_duct")
    comp_plan["component_stride"] = plan.get("component_stride", plan["fixed_stride"])
    comp = pvfv_run_component_study(cfg, comp_plan, beta_star, ref_cache, out)

    tables = pvfv_make_seed_influence_tables(out)
    pvfv_make_seed_influence_quick_plots(out)
    products_manifest = pvfv_write_seed_influence_manifest(out, tables, wall.get("summary", {}))

    zip_path = None
    if zip_outputs:
        zip_path = shutil.make_archive(str(out), "zip", out)
        print(f"[flow-seed] zipped outputs: {zip_path}")
    summary = {
        "out_dir": str(out),
        "zip_path": zip_path,
        "profile": profile,
        "beta_star": beta_star,
        "n_density_rows": len(density_rows),
        "n_strategy_rows": len(strategy_rows),
        "n_component_rows": len(comp.get("rows", [])) if isinstance(comp, dict) else None,
        "tables": tables,
        "data_products_manifest": products_manifest,
        "figures_tables_fully_supported_by_this_notebook": [
            "Fig5 canonical/wall-transfer data",
            "Fig6 synthetic Pareto including seed density/placement influence",
            "FV-side Fig7 component study",
            "Table5 final accuracy-cost summary",
            "FV-side Table6 component-study summary",
        ],
        "figures_tables_not_supported_by_this_notebook": [
            "Fig1 schematic",
            "Fig2 ROI--JFA pipeline",
            "Fig4 ROI--JFA correctness/runtime",
            "Table4 ROI--JFA diagnostics",
        ],
    }
    (out / "pvfv_suite_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


## Half-offset + geodesic-farthest-point admissible seed protocol

This block replaces the constrained-random/wall-band seed study of the cell above for the flow-side simulations.

Main density curves use two stable particle-seed families:

1. `regular_half_offset`: regular half-offset stride seeds, used as the conservative baseline.
2. `gfps_admissible`: clearance-aware, component-covered, geodesic-farthest-point style seeds with comparable seed count.

The suite uses exact GPU geodesic labels for flow-side validation. ROI--JFA partition figures/tables are not part of this notebook.

In [ ]:

# ============================================================
# PoreVoronoi-FV half-offset + geodesic-farthest-point seed protocol
# ============================================================
# Execute this cell after the PB61.8 GPU code, full-data export overrides,
# and the seed-influence suite definitions have been loaded.
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple
from collections import deque
import json, math, shutil
import numpy as np

try:
    from scipy import ndimage as _hg_ndi
except Exception:
    _hg_ndi = None

PVFV_HALF_GFPS_OUT_DIR = "pvfv_abc_half_gfps_seed_outputs"


def pvfv_half_gfps_case_plan(profile: str = "pilot") -> dict:
    """Production-oriented case plan for stable seed families.

    The main manuscript curves compare regular half-offset seeds with a
    clearance-aware geodesic farthest-point style family. Naive origin, naive
    random, and naive wall-biased placements are intentionally excluded from the
    main suite because pilot runs showed boundary-aligned and degenerate cells.
    """
    profile = str(profile).lower()
    if profile in ("production_broad", "broad"):
        shape = (24, 24, 64)
        density_strides = [(2, 2, 2), (3, 3, 3), (4, 4, 4), (5, 5, 5)]
        fixed_stride = (3, 3, 3)
        beta_values = (0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.4, 1.6, 2.0)
        gfps_repeats = 2
    elif profile in ("production", "prod"):
        shape = (24, 24, 64)
        density_strides = [(2, 2, 2), (3, 3, 3), (4, 4, 4)]
        fixed_stride = (3, 3, 3)
        beta_values = (0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.4, 1.6, 2.0)
        gfps_repeats = 2
    elif profile in ("production_dense", "dense"):
        shape = (24, 24, 64)
        density_strides = [(2, 2, 2), (3, 3, 3), (4, 4, 4)]
        fixed_stride = (2, 2, 2)
        beta_values = (0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.4, 1.6, 2.0)
        gfps_repeats = 2
    else:
        shape = (12, 12, 24)
        density_strides = [(2, 2, 2), (3, 3, 3), (4, 4, 4)]
        fixed_stride = (3, 3, 3)
        beta_values = (0.75, 0.9, 1.0, 1.1, 1.25, 1.5)
        gfps_repeats = 1

    return {
        "profile": profile,
        "shape_zyx": shape,
        "density_strides": density_strides,
        "fixed_stride": fixed_stride,
        "beta_values": beta_values,
        "cases": ["orthogonal_duct", "skewed_duct", "A_thin_wall", "B_narrow_throat", "C_maze"],
        "synthetic_cases": ["A_thin_wall", "B_narrow_throat", "C_maze"],
        "canonical_cases": ["orthogonal_duct", "skewed_duct"],
        "component_case": "skewed_duct",
        "component_stride": fixed_stride,
        # The seed-influence framework expects this key. Here it controls GFPS repeats.
        "random_repeats": int(gfps_repeats),
        "offset_modes": ["half"],
        "seed_protocol": {
            "baseline_density_family": "regular half-offset stride seeds",
            "secondary_density_family": "clearance-aware geodesic farthest-point style seeds",
            "strategy_families": ["stride_half_admissible", "gfps_admissible"],
            "gfps_graph": "6-neighbor face graph for seed placement",
            "min_clearance_vox": 1.0,
            "min_sep_factor": 0.50,
            "gfps_batch_rounds": 10,
            "excluded_main_families": ["origin_offset", "naive_random", "naive_wall_biased"],
        },
    }


def pvfv_hg_to_numpy_mask(mask: Any) -> np.ndarray:
    """Convert a CuPy/NumPy boolean mask to a NumPy boolean array."""
    try:
        return cp.asnumpy(mask).astype(bool)
    except Exception:
        return np.asarray(mask).astype(bool)


def pvfv_hg_clearance_voxels_np(mask_np: np.ndarray) -> np.ndarray:
    """Voxel clearance to solid in voxel units, with a simple fallback."""
    m = np.asarray(mask_np, dtype=bool)
    if _hg_ndi is not None:
        return _hg_ndi.distance_transform_edt(m).astype(np.float32)
    # Fallback: wall-adjacent fluid voxels get clearance 1, others 2.
    D, H, W = m.shape
    clear = np.where(m, 2.0, 0.0).astype(np.float32)
    for dz, dy, dx in [(1,0,0), (-1,0,0), (0,1,0), (0,-1,0), (0,0,1), (0,0,-1)]:
        shifted = np.zeros_like(m)
        z0, z1 = max(0, dz), D + min(0, dz)
        y0, y1 = max(0, dy), H + min(0, dy)
        x0, x1 = max(0, dx), W + min(0, dx)
        shifted[z0:z1, y0:y1, x0:x1] = m[z0-dz:z1-dz, y0-dy:y1-dy, x0-dx:x1-dx]
        clear[m & (~shifted)] = 1.0
    return clear


def pvfv_hg_components_np(mask_np: np.ndarray) -> Tuple[np.ndarray, int]:
    """Face-connected components of the traversable phase."""
    m = np.asarray(mask_np, dtype=bool)
    if _hg_ndi is not None:
        struct = np.zeros((3,3,3), dtype=np.uint8)
        struct[1,1,1] = 1
        struct[0,1,1] = struct[2,1,1] = 1
        struct[1,0,1] = struct[1,2,1] = 1
        struct[1,1,0] = struct[1,1,2] = 1
        lab, n = _hg_ndi.label(m, structure=struct)
        return lab.astype(np.int32), int(n)
    # BFS fallback.
    D, H, W = m.shape
    lab = np.zeros(m.shape, dtype=np.int32)
    comp = 0
    dirs = [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]
    for start in np.flatnonzero(m.ravel()):
        z = start // (H*W); rem = start % (H*W); y = rem // W; x = rem % W
        if lab[z,y,x] != 0:
            continue
        comp += 1
        lab[z,y,x] = comp
        q = deque([(z,y,x)])
        while q:
            zz, yy, xx = q.popleft()
            for dz, dy, dx in dirs:
                nz, ny, nx = zz+dz, yy+dy, xx+dx
                if 0 <= nz < D and 0 <= ny < H and 0 <= nx < W and m[nz,ny,nx] and lab[nz,ny,nx] == 0:
                    lab[nz,ny,nx] = comp
                    q.append((nz,ny,nx))
    return lab, comp


def pvfv_hg_flat_to_zyx(flat: np.ndarray, shape: Tuple[int, int, int]) -> np.ndarray:
    flat = np.asarray(flat, dtype=np.int64).ravel()
    D, H, W = shape
    z = flat // (H * W)
    rem = flat % (H * W)
    y = rem // W
    x = rem % W
    return np.stack([z, y, x], axis=1).astype(np.float32)


def pvfv_hg_zyx_to_flat(coords: np.ndarray, shape: Tuple[int, int, int]) -> np.ndarray:
    coords = np.asarray(coords, dtype=np.int64)
    D, H, W = shape
    return ((coords[:,0] * H + coords[:,1]) * W + coords[:,2]).astype(np.int64)


def pvfv_hg_multisource_bfs_distance_np(mask_np: np.ndarray, seed_flat: Sequence[int]) -> np.ndarray:
    """Multi-source 6-neighbor geodesic distance in voxel steps.

    This is used only for seed placement. The flow simulations still build exact
    geodesic labels using the main GPU label routine.
    """
    m = np.asarray(mask_np, dtype=bool)
    D, H, W = m.shape
    N = D * H * W
    dist = np.full(N, np.inf, dtype=np.float32)
    q = deque()
    flat_m = m.ravel()
    for s in np.asarray(seed_flat, dtype=np.int64).ravel():
        if 0 <= int(s) < N and flat_m[int(s)] and not np.isfinite(dist[int(s)]):
            dist[int(s)] = 0.0
            q.append(int(s))
    dirs = [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]
    while q:
        f = q.popleft()
        z = f // (H*W); rem = f % (H*W); y = rem // W; x = rem % W
        nd = dist[f] + 1.0
        for dz, dy, dx in dirs:
            nz, ny, nx = z+dz, y+dy, x+dx
            if 0 <= nz < D and 0 <= ny < H and 0 <= nx < W and m[nz,ny,nx]:
                nf = (nz * H + ny) * W + nx
                if nd < dist[nf]:
                    dist[nf] = nd
                    q.append(nf)
    return dist.reshape(m.shape)


def pvfv_hg_select_by_minsep(flat_ordered: np.ndarray,
                              shape: Tuple[int, int, int],
                              k: int,
                              min_sep: float,
                              existing_flat: Optional[Sequence[int]] = None,
                              max_scan_factor: int = 40) -> List[int]:
    """Select up to k ordered candidates while enforcing a Euclidean separation."""
    flat_ordered = np.asarray(flat_ordered, dtype=np.int64).ravel()
    if k <= 0 or flat_ordered.size == 0:
        return []
    existing_coords = None
    if existing_flat is not None and len(existing_flat):
        existing_coords = pvfv_hg_flat_to_zyx(np.asarray(existing_flat, dtype=np.int64), shape)
    selected: List[int] = []
    selected_coords: List[np.ndarray] = []
    min_sep2 = float(min_sep) ** 2
    max_scan = min(int(flat_ordered.size), max(int(k) * int(max_scan_factor), int(k)))
    for f in flat_ordered[:max_scan]:
        c = pvfv_hg_flat_to_zyx(np.asarray([f], dtype=np.int64), shape)[0]
        ok = True
        if existing_coords is not None and existing_coords.size:
            if np.min(np.sum((existing_coords - c) ** 2, axis=1)) < min_sep2:
                ok = False
        if ok and selected_coords:
            sc = np.vstack(selected_coords)
            if np.min(np.sum((sc - c) ** 2, axis=1)) < min_sep2:
                ok = False
        if ok:
            selected.append(int(f))
            selected_coords.append(c)
            if len(selected) >= int(k):
                break
    return selected


def pvfv_gfps_seed_flat_gpu(mask: Any,
                              target_count: int,
                              *,
                              rng_seed: int = 0,
                              min_clearance: float = 1.0,
                              min_sep: float = 1.0,
                              batch_rounds: int = 10) -> Any:
    """Clearance-aware geodesic farthest-point style seed selection.

    The algorithm is a batched k-center heuristic on the face-connected voxel
    graph. It first inserts one clearance-maximizing seed per connected component,
    then repeatedly computes the 6-neighbor geodesic distance to the current seed
    set and inserts a batch of farthest admissible candidates.
    """
    mask_np = pvfv_hg_to_numpy_mask(mask)
    shape = tuple(mask_np.shape)
    D, H, W = shape
    N = D * H * W
    target_count = int(max(1, min(int(target_count), int(mask_np.sum()))))
    rng = np.random.default_rng(int(rng_seed))
    clearance = pvfv_hg_clearance_voxels_np(mask_np)
    cand = mask_np & (clearance >= float(min_clearance))
    if int(cand.sum()) < max(1, min(target_count, 8)):
        cand = mask_np & (clearance >= max(0.0, float(min_clearance) * 0.5))
    if int(cand.sum()) < target_count:
        cand = mask_np.copy()

    labels, ncomp = pvfv_hg_components_np(mask_np)
    seeds: List[int] = []
    # One seed per component, choosing high-clearance candidate when possible.
    for comp in range(1, int(ncomp) + 1):
        comp_mask = labels == comp
        comp_cand = comp_mask & cand
        if not comp_cand.any():
            comp_cand = comp_mask
        flats = np.flatnonzero(comp_cand.ravel())
        if flats.size == 0:
            continue
        clear_vals = clearance.ravel()[flats]
        max_clear = clear_vals.max()
        top = flats[np.flatnonzero(clear_vals >= max_clear - 1.0e-6)]
        seeds.append(int(rng.choice(top)))
    if len(seeds) == 0:
        flats = np.flatnonzero(mask_np.ravel())
        seeds = [int(rng.choice(flats))]
    # If there are more components than target seeds, keep largest-clearance seeds.
    if len(seeds) > target_count:
        clear_vals = clearance.ravel()[np.asarray(seeds, dtype=np.int64)]
        order = np.argsort(-clear_vals)
        seeds = [seeds[i] for i in order[:target_count]]

    cand_flat_all = np.flatnonzero(cand.ravel()).astype(np.int64)
    seed_set = set(int(s) for s in seeds)
    rounds = max(1, int(batch_rounds))
    while len(seeds) < target_count:
        remaining = target_count - len(seeds)
        batch = max(1, int(math.ceil(remaining / rounds)))
        dist = pvfv_hg_multisource_bfs_distance_np(mask_np, seeds).ravel()
        # Restrict to admissible candidates not already selected.
        scores = dist[cand_flat_all]
        finite = np.isfinite(scores)
        cand_flat = cand_flat_all[finite]
        scores = scores[finite]
        if cand_flat.size == 0:
            break
        not_sel = np.array([int(f) not in seed_set for f in cand_flat], dtype=bool)
        cand_flat = cand_flat[not_sel]
        scores = scores[not_sel]
        if cand_flat.size == 0:
            break
        # Break exact score ties reproducibly with tiny jitter.
        jitter = rng.random(cand_flat.size) * 1.0e-6
        order = np.argsort(-(scores + jitter))
        ordered = cand_flat[order]
        new = pvfv_hg_select_by_minsep(ordered, shape, batch, min_sep, existing_flat=seeds)
        if len(new) == 0 and min_sep > 0.25:
            new = pvfv_hg_select_by_minsep(ordered, shape, batch, 0.5 * min_sep, existing_flat=seeds)
        if len(new) == 0:
            new = [int(ordered[0])]
        for f in new:
            if int(f) not in seed_set:
                seeds.append(int(f))
                seed_set.add(int(f))
                if len(seeds) >= target_count:
                    break
        if len(new) == 0:
            break
    seeds_arr = np.asarray(seeds[:target_count], dtype=np.int64)
    return cp.asarray(seeds_arr, dtype=cp.int64)


def pvfv_half_gfps_seed_specs_for_density(mask: Any, stride: Tuple[int, int, int],
                                            offset_modes: List[str]) -> List[Tuple[dict, Any]]:
    """Main density curve: regular half-offset and GFPS-style seeds."""
    specs: List[Tuple[dict, Any]] = []
    off = pvfv_offset_from_mode(stride, "half")
    seed_half = pvfv_stride_offset_seed_flat_gpu(mask, stride, off)
    target = int(seed_half.size)
    specs.append(({
        "stage": "half_gfps_density_sweep",
        "family": "stride_half_admissible",
        "seed_id": f"stride_{'x'.join(map(str, stride))}_half",
        "stride_zyx": tuple(stride),
        "offset_zyx": tuple(off),
        "repeat": 0,
        "admissible_seed_protocol": "regular_half_offset",
        "target_seed_count": target,
    }, seed_half))
    sx = max(1, int(min(stride)))
    min_sep = max(1.0, 0.50 * float(sx))
    seed_gfps = pvfv_gfps_seed_flat_gpu(mask, target, rng_seed=5100 + 101 * sx,
                                         min_clearance=1.0, min_sep=min_sep, batch_rounds=10)
    specs.append(({
        "stage": "half_gfps_density_sweep",
        "family": "gfps_admissible",
        "seed_id": f"gfps_target_stride_{'x'.join(map(str, stride))}",
        "stride_zyx": tuple(stride),
        "offset_zyx": "gfps",
        "repeat": 0,
        "admissible_seed_protocol": "clearance_aware_batched_geodesic_farthest_point",
        "target_seed_count": target,
        "min_sep": min_sep,
    }, seed_gfps))
    return specs


def pvfv_half_gfps_seed_specs_for_strategy(mask: Any, fixed_stride: Tuple[int, int, int],
                                             random_repeats: int,
                                             base_seed_count: Optional[int] = None) -> List[Tuple[dict, Any]]:
    """Seed strategy comparison restricted to stable manuscript families."""
    off = pvfv_offset_from_mode(fixed_stride, "half")
    seed_half = pvfv_stride_offset_seed_flat_gpu(mask, fixed_stride, off)
    if base_seed_count is None:
        base_seed_count = int(seed_half.size)
    sx = max(1, int(min(fixed_stride)))
    min_sep = max(1.0, 0.50 * float(sx))
    specs: List[Tuple[dict, Any]] = []
    specs.append(({
        "stage": "half_gfps_seed_strategy_sweep",
        "family": "stride_half_admissible",
        "seed_id": "strategy_stride_half",
        "stride_zyx": tuple(fixed_stride),
        "offset_zyx": tuple(off),
        "repeat": 0,
        "admissible_seed_protocol": "regular_half_offset",
        "target_seed_count": int(base_seed_count),
    }, seed_half))
    for r in range(max(1, int(random_repeats))):
        specs.append(({
            "stage": "half_gfps_seed_strategy_sweep",
            "family": "gfps_admissible",
            "seed_id": f"gfps_r{r}",
            "stride_zyx": tuple(fixed_stride),
            "offset_zyx": "gfps",
            "repeat": r,
            "admissible_seed_protocol": "clearance_aware_batched_geodesic_farthest_point",
            "target_seed_count": int(base_seed_count),
            "min_sep": min_sep,
        }, pvfv_gfps_seed_flat_gpu(mask, int(base_seed_count), rng_seed=6200 + r,
                                    min_clearance=1.0, min_sep=min_sep, batch_rounds=10)))
    return specs


def pvfv_run_complete_half_gfps_seed_suite(profile: str = "pilot", zip_outputs: bool = True,
                                            include_strategy: bool = True,
                                            clean_output: bool = True) -> dict:
    """Run the flow-side suite with half-offset and GFPS-style seed families.

    Responsible for Fig. 5, Fig. 6, FV-side Fig. 7, Table 5, and FV-side
    Table 6. ROI--JFA-specific Fig. 2/Fig. 4/Table 4 remain separate.
    """
    global PVFV_SEED_OUT_DIR, pvfv_seed_influence_case_plan, pvfv_seed_specs_for_density, pvfv_seed_specs_for_strategy
    old_out_dir = PVFV_SEED_OUT_DIR
    old_plan = pvfv_seed_influence_case_plan
    old_density = pvfv_seed_specs_for_density
    old_strategy = pvfv_seed_specs_for_strategy
    try:
        PVFV_SEED_OUT_DIR = PVFV_HALF_GFPS_OUT_DIR
        pvfv_seed_influence_case_plan = pvfv_half_gfps_case_plan
        pvfv_seed_specs_for_density = pvfv_half_gfps_seed_specs_for_density
        pvfv_seed_specs_for_strategy = pvfv_half_gfps_seed_specs_for_strategy
        summary = pvfv_run_complete_seed_influence_suite(profile=profile,
                                                          zip_outputs=zip_outputs,
                                                          include_strategy=include_strategy,
                                                          clean_output=clean_output)
        out = Path(summary["out_dir"])
        protocol = pvfv_half_gfps_case_plan(profile).get("seed_protocol", {})
        (out / "pvfv_half_gfps_seed_protocol.json").write_text(json.dumps(protocol, indent=2), encoding="utf-8")
        summary["suite"] = "PoreVoronoi-FV ABC half-offset + GFPS admissible-seed flow simulation suite"
        summary["half_gfps_seed_protocol"] = protocol
        (out / "pvfv_suite_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
        if zip_outputs:
            zip_path = shutil.make_archive(str(out), "zip", out)
            summary["zip_path"] = zip_path
            print(f"[flow-half-gfps] re-zipped outputs with protocol file: {zip_path}")
        return summary
    finally:
        PVFV_SEED_OUT_DIR = old_out_dir
        pvfv_seed_influence_case_plan = old_plan
        pvfv_seed_specs_for_density = old_density
        pvfv_seed_specs_for_strategy = old_strategy


## Execute half-offset + GFPS admissible seed suite

Start with `profile='pilot'`. For full-scale runs use `profile='production'` or `profile='production_broad'`.

The output directory is `pvfv_abc_half_gfps_seed_outputs/` and, with `zip_outputs=True`, the notebook writes `pvfv_abc_half_gfps_seed_outputs.zip`.

In [ ]:
# ============================================================
# Manuscript-aware run control for half-offset + GFPS suite
# ============================================================
# This cell is intentionally the only place where the final run profile and
# output root are selected.  Keep the algorithm cells above unchanged.
from __future__ import annotations
from pathlib import Path
import os, json, time

NOTEBOOK_DIR = Path.cwd().resolve()
WORK_PACKAGE_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR
MANUSCRIPT_RUN_ROOT = Path(os.environ.get(
    "PVFV_MANUSCRIPT_RUN_ROOT",
    str(WORK_PACKAGE_ROOT / "production_runs")
)).resolve()
MANUSCRIPT_RUN_ROOT.mkdir(parents=True, exist_ok=True)

# For a quick environment check set PVFV_FLOW_PROFILE=pilot.
# For manuscript data products set PVFV_FLOW_PROFILE=production or production_broad.
PVFV_FLOW_PROFILE = str(os.environ.get(
    "PVFV_FLOW_PROFILE",
    globals().get("PVFV_FLOW_PROFILE", "pilot")
)).lower()

PVFV_INCLUDE_STRATEGY = str(os.environ.get(
    "PVFV_INCLUDE_STRATEGY",
    globals().get("PVFV_INCLUDE_STRATEGY", "0")
)).lower() not in {"0", "false", "no"}

PVFV_CLEAN_OUTPUT = str(os.environ.get(
    "PVFV_CLEAN_OUTPUT",
    globals().get("PVFV_CLEAN_OUTPUT", "1")
)).lower() not in {"0", "false", "no"}

PVFV_ZIP_OUTPUTS = str(os.environ.get(
    "PVFV_ZIP_OUTPUTS",
    globals().get("PVFV_ZIP_OUTPUTS", "0")
)).lower() not in {"0", "false", "no"}

PVFV_SKIP_INLINE_QUICK_PLOTS = str(os.environ.get(
    "PVFV_SKIP_INLINE_QUICK_PLOTS",
    globals().get("PVFV_SKIP_INLINE_QUICK_PLOTS", "1")
)).lower() not in {"0", "false", "no"}

PVFV_HALF_GFPS_OUT_DIR = str(
    MANUSCRIPT_RUN_ROOT / f"pvfv_abc_half_gfps_seed_outputs__{PVFV_FLOW_PROFILE}"
)

print("[flow-run] work package root:", WORK_PACKAGE_ROOT)
print("[flow-run] run root:", MANUSCRIPT_RUN_ROOT)
print("[flow-run] profile:", PVFV_FLOW_PROFILE)
print("[flow-run] output dir:", PVFV_HALF_GFPS_OUT_DIR)
print("[flow-run] include strategy:", PVFV_INCLUDE_STRATEGY)
print("[flow-run] zip outputs:", PVFV_ZIP_OUTPUTS)
print("[flow-run] skip inline quick plots:", PVFV_SKIP_INLINE_QUICK_PLOTS)

if PVFV_SKIP_INLINE_QUICK_PLOTS:
    def pvfv_make_seed_influence_quick_plots(out):
        print("[flow-run] inline quick plots skipped; use package builder for paper figures.")
        return None

# Fast manuscript defaults: pilot showed GFPS rows are mostly invalid for the
# main Pareto table, while half-offset rows are stable.  Keep GFPS available by
# setting PVFV_DENSITY_FAMILIES=all when a sensitivity run is needed.
PVFV_DENSITY_FAMILIES = str(os.environ.get(
    "PVFV_DENSITY_FAMILIES",
    globals().get("PVFV_DENSITY_FAMILIES", "half")
)).lower().strip()

if PVFV_DENSITY_FAMILIES in {"half", "stride_half", "half_only", "regular"}:
    _pvfv_original_half_gfps_seed_specs_for_density = pvfv_half_gfps_seed_specs_for_density
    def pvfv_half_gfps_seed_specs_for_density(mask, stride_zyx, *args, **kwargs):
        try:
            specs = _pvfv_original_half_gfps_seed_specs_for_density(mask, stride_zyx, *args, **kwargs)
        except TypeError:
            specs = _pvfv_original_half_gfps_seed_specs_for_density(mask, stride_zyx)
        return [item for item in specs if str(item[0].get("family", "")).lower() == "stride_half_admissible"]
    print("[flow-run] density families: half-offset only")
elif PVFV_DENSITY_FAMILIES in {"all", "half+gfps", "gfps"}:
    print("[flow-run] density families: half-offset + GFPS")
else:
    raise ValueError("PVFV_DENSITY_FAMILIES must be 'half' or 'all'")

# Optional label backend.  The default exact_geodesic backend is retained for
# manuscript comparability.  Set PVFV_LABEL_BACKEND=roi_jfa to use the exported
# ROI-JFA backend module from the work-package notebook directory.
PVFV_LABEL_BACKEND = str(os.environ.get(
    "PVFV_LABEL_BACKEND",
    globals().get("PVFV_LABEL_BACKEND", "exact_geodesic")
)).lower().strip()

if PVFV_LABEL_BACKEND in {"roi_jfa", "roi-jfa", "roijfa"}:
    import sys
    if str(NOTEBOOK_DIR) not in sys.path:
        sys.path.insert(0, str(NOTEBOOK_DIR))
    import roi_jfa_backend as _pvfv_roi_jfa_backend
    _pvfv_original_build_geometry_from_seed_flat_timed = pvfv_build_geometry_from_seed_flat_timed

    def pvfv_build_geometry_from_seed_flat_timed(mask: cp.ndarray, seed_flat: cp.ndarray, cfg: PB615Config,
                                                  *, split_face_components: bool = True,
                                                  label_mode: str = "exact_geodesic",
                                                  seed_spec: str = "custom") -> Tuple[GPUFVMGeometry, dict]:
        requested = PVFV_LABEL_BACKEND if label_mode == "exact_geodesic" else str(label_mode).lower().strip()
        if requested not in {"roi_jfa", "roi-jfa", "roijfa"}:
            return _pvfv_original_build_geometry_from_seed_flat_timed(
                mask, seed_flat, cfg,
                split_face_components=split_face_components,
                label_mode=label_mode,
                seed_spec=seed_spec,
            )

        t0 = time.perf_counter()
        seed_flat = cp.asarray(seed_flat, dtype=cp.int64)
        cp.cuda.Stream.null.synchronize()
        t_seed = time.perf_counter() - t0

        D, H, W = [int(v) for v in mask.shape]
        z = seed_flat // cp.int64(H * W)
        rem = seed_flat - z * cp.int64(H * W)
        y = rem // cp.int64(W)
        x = rem - y * cp.int64(W)
        seeds_zyx = cp.stack([z, y, x], axis=1).astype(cp.int32, copy=False)

        t1 = time.perf_counter()
        labels, dist, _tile_roi, _roi_mask = _pvfv_roi_jfa_backend.geodesic_voronoi_roi_jfa(
            mask.astype(cp.uint8, copy=False),
            seeds_zyx,
            tile_size=tuple(int(v) for v in os.environ.get("PVFV_ROIJFA_TILE", "8,8,16").split(",")),
            delta_r=float(os.environ.get("PVFV_ROIJFA_DELTA_R", "1.0")),
            eta_max=float(os.environ.get("PVFV_ROIJFA_ETA_MAX", "0.8")),
            r_tile=int(os.environ.get("PVFV_ROIJFA_R_TILE", "1")),
            verbose=False,
            viz_policy="none",
            n_relax_after=int(os.environ.get("PVFV_ROIJFA_RELAX_AFTER", "1")),
            profile_gpu=True,
            return_records=False,
        )
        labels = labels.astype(cp.int32, copy=False)
        dist = dist.astype(cp.float64, copy=False)
        cp.cuda.Stream.null.synchronize()
        t_label = time.perf_counter() - t1

        n_seed = int(seed_flat.size)
        t_split0 = time.perf_counter()
        split_info = {
            "n_original_labels": n_seed,
            "n_cv": n_seed,
            "n_split_extra": 0,
            "n_face_disconnected_labels": 0,
            "face_connected_fraction": 1.0,
        }
        if split_face_components:
            labels, split_info = pvfv_face_connected_reindex_cpu(mask, labels)
        cp.cuda.Stream.null.synchronize()
        t_split = time.perf_counter() - t_split0

        n0a = pvfv_zero_area_candidate_count_cpu(mask, labels)
        t2 = time.perf_counter()
        geom = pvfv_build_geometry_from_labels_ncells_gpu(mask, labels, dist, int(split_info["n_cv"]), cfg)
        cp.cuda.Stream.null.synchronize()
        t_mom = time.perf_counter() - t2

        n_fl = int(mask.sum().get())
        meta = {
            "S": n_seed,
            "N_cv": int(geom.n_cells),
            "N_fl": n_fl,
            "C_comp": float(n_fl / max(int(geom.n_cells), 1)),
            "t_seed_s": float(t_seed),
            "t_label_s": float(t_label),
            "t_split_s": float(t_split),
            "t_part_s": float(t_seed + t_label + t_split),
            "t_mom_s": float(t_mom),
            "N_split": int(split_info["n_split_extra"]),
            "N_face_disconnected_labels": int(split_info["n_face_disconnected_labels"]),
            "face_connected_fraction": float(split_info["face_connected_fraction"]),
            "N_0A": int(n0a),
            "label_mode": "roi_jfa",
            "distance_connectivity": int(cfg.distance_connectivity),
            "seed_spec": str(seed_spec),
        }
        return geom, meta

    print("[flow-run] label backend: ROI-JFA optional backend imported")
else:
    print("[flow-run] label backend: exact_geodesic")

summary = pvfv_run_complete_half_gfps_seed_suite(
    profile=PVFV_FLOW_PROFILE,
    zip_outputs=PVFV_ZIP_OUTPUTS,
    include_strategy=PVFV_INCLUDE_STRATEGY,
    clean_output=PVFV_CLEAN_OUTPUT,
)

run_manifest = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook": "flow_solver.ipynb",
    "entry_cell": "final manuscript-aware run control cell",
    "kernel": "cu129_win",
    "profile": PVFV_FLOW_PROFILE,
    "out_dir": summary.get("out_dir"),
    "zip_path": summary.get("zip_path"),
    "supported_outputs": summary.get("figures_tables_fully_supported_by_this_notebook", []),
    "not_supported_outputs": summary.get("figures_tables_not_supported_by_this_notebook", []),
    "case_scope_note": (
        "This production suite currently covers canonical duct/skewed duct and ABC synthetic cases. "
        "Sandstone/fibrous proxy rows in the current tex require either a dedicated proxy-flow block "
        "or a manuscript wording change before they can be claimed as outputs of this suite."
    ),
}
Path(summary["out_dir"]).joinpath("pvfv_manuscript_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2), encoding="utf-8"
)
summary
